# PSMA pKi Classification Pipeline with Reaction-Based Enumeration (F9)

Predicts whether a compound is a potent PSMA (glutamate carboxypeptidase II / FOLH1)
inhibitor — **pKi ≥ 8.0, i.e. Ki ≤ 10 nM** — and ranks a **virtual library enumerated from
reaction SMARTS and building blocks**, mixed with a stratified sample of a commercial
catalogue as controls.

Two changes from F8, both consequences of what F8 measured.

## 1. Stripped to pKi alone

F8's leave-one-cluster-out results made the choice for us. On pKi, the models cleared the
nearest-neighbour floor by +0.11 ROC-AUC (LightGBM 0.796 against Tanimoto kNN's 0.688). On
pIC50, **nothing cleared it at all** — every model, including a neural network, matched pure
similarity lookup on whole-ranking discrimination. pKi is the endpoint where there is
something to model.

What that removes:

- **The neural network, and with it the ChEMBL pretraining set, the pretraining pass and the
  entire Optuna search.** Two ablations found pretraining did nothing (F4 regression: +0.062
  R² on random splits and nothing on grouped; F6 classification: −0.0066 AUC on random
  splits and nothing on grouped), and on pKi the network was the joint-worst model at 0.688.
  Removing it also removes a live web-API dependency: the descriptor curation and feature
  scaling are now fitted on the PSMA set, where 700+ compounds is ample.
- **The multi-task machinery.** Only the network shared a trunk; every classical model was
  already fitted per endpoint, so nothing is lost.

What it deliberately keeps: **compounds with a pIC50 but no pKi**. They carry a NaN label
and cannot train the model, but they are known PSMA ligands, so they still define the
applicability domain and still shape the Butina clustering that builds the folds. Dropping
them would shrink the domain from ~730 compounds to ~280 for no gain.

One bug this surfaced: F7 and F8 decided which endpoint's derived values to drop by testing
`ep != primary_endpoint`. That worked only because pIC50 was primary and pKi was derived. With
pKi as the modelled endpoint it would have **kept every Cheng–Prusoff value**. There is now an
explicit `derived_endpoint` setting.

## 2. A library built rather than bought

F8 settled why nothing was ever in range, and corrected an earlier error of mine in doing so.
The library health report showed the Enamine file **is** a normal drug-like collection —
median MW 337, 24 heavy atoms, 1% fragment-like — so my repeated claim that it was a reagent
catalogue was wrong, an artifact of reading the first five rows of a file ordered
small-to-large. The real reason only 36 of 200,560 compounds were within structural reach is
a **genuine chemistry mismatch**: PSMA ligands are Glu-urea peptidomimetics, highly polar
with multiple carboxylates, and a diversity deck is biased toward lead-like, less acidic
chemistry.

Part A.2 therefore builds the library instead. Reaction SMARTS are applied to building
blocks around the Glu-urea pharmacophore, and **every reaction ships a self-test** —
reactants plus expected product — which runs before enumeration and disables any template
that fails. A subtly wrong SMARTS does not raise; it silently yields nothing or the wrong
regiochemistry, so this check is not optional. Every product carries its core, reaction and
cap, making a hit a synthesis proposal rather than only a structure.

**And the catalogue is mixed back in**, stratified three ways, because an enumerated library
built from the training pharmacophore is in-domain by construction and ranking twenty
thousand near-identical Glu-ureas proves nothing:

- `library_near` — the catalogue compounds closest to the training set: the scaffold-hop
  candidates, and the only non-circular output this screen can produce.
- `library_matched` — **property-matched decoys** following the enumerated library's own MW
  distribution. Without this the model could separate the populations on size alone
  (enumerated ~MW 600–900 against a catalogue median of 337) and a good-looking ranking
  would mean nothing.
- `library_random` — unbiased background.

How many catalogue compounds *can* be property-matched is itself a measurement: it puts the
chemistry gap in compounds rather than in fingerprint distance.

## What to read, and in what order

Part E.4 is built around the circularity. **First**: do the enumerated products that match
known training compounds rank in the top decile? They are a positive control, and if they do
not, nothing else on the page is worth reading. **Second**: enumerated versus
property-matched decoys, which is the size-controlled comparison. **Third**: any catalogue
compound reaching the enumerated top decile — a real scaffold hop. **Fourth**: the spread of
scores across caps, which says whether the model is reading the variable part of the molecule
or only recognising the shared core.

## One F8 correction carried in

F8's floor comparison was measured on ROC-AUC only, and reported a tie against both floors
on pIC50 — but precision@50 there was 0.88 for the best model against 0.64 for kNN. AUC
averages over a whole ranking including the middle nobody acts on. Part E.2b now reports both
and says which is operative when they disagree.

## What has not changed

Nothing has been tested in a lab. 282 pKi-labelled compounds is a small dataset, one filing
supplies a sixth of the whole set, and the leave-one-cluster-out numbers should be read as
the prediction of a hit rate rather than as a hit rate.

## 0. Setup

Install dependencies. Safe to re-run — pip skips anything already satisfied.


In [ ]:
%pip install -q duckdb rdkit pandas numpy scikit-learn matplotlib seaborn requests tqdm scipy joblib lightgbm

In [ ]:
import os
import io
import re
import json
import zipfile
import shutil
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Sequence, List, Dict, Tuple

import duckdb
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem import rdChemReactions
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import DataStructs

from sklearn.model_selection import (train_test_split, KFold, StratifiedKFold,
                                     cross_val_predict, GridSearchCV)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from sklearn.cross_decomposition import PLSRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.svm import SVC
from sklearn.isotonic import IsotonicRegression
import joblib

from lightgbm import LGBMClassifier

import matplotlib.pyplot as plt
import seaborn as sns

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

np.random.seed(0)

# F9 is classical-only: LightGBM, kernel PLS-DA, RandomForest, a Tanimoto-kernel SVM,
# Tanimoto kNN, L2 logistic regression and Bernoulli naive Bayes. The neural network and
# its ChEMBL pretraining went with the multi-task formulation, which also removes torch,
# Optuna and scikit-optimize from the dependency list — and with them the GPU setup, which
# nothing here would have used. Everything below runs on CPU in minutes.

print("Environment ready.")

## 1. Configuration

Everything that changes between projects lives here: ChEMBL pretraining scope, the
BindingDB target/endpoints, screening library path, AD thresholds, model
architectures, and every hyperparameter-search setting. Nothing below this cell
needs to change for a routine re-run.


In [ ]:
@dataclass
class Config:
    # ---- ChEMBL pretraining set (domain-targeted by default) -------------------
    # A bioactivity set used only to *pretrain* the shared NN trunk (transfer
    # learning) before it ever sees PSMA data. Pulled live via
    # chembl_webresource_client; cached to `chembl_pretrain_cache` so re-runs are free.
    #
    # "domain_targeted" (default): restrict pretraining to targets related to PSMA —
    # PSMA (GCPII/FOLH1) is a zinc-dependent, co-catalytic-dinuclear-zinc metallopeptidase
    # (MEROPS clan MH, family M28). Other zinc metallopeptidases/metalloproteases are
    # the natural "related" pool: their inhibitors are built around the same handful of
    # zinc-binding chemotypes PSMA inhibitors use (phosphonates, thiols, hydroxamates,
    # ureas), so pretraining here teaches the trunk that specific structure-activity
    # vocabulary rather than generic, unrelated bioactivity patterns.
    # "broad": the original target-agnostic sample across arbitrary human targets.
    chembl_pretrain_mode: str = "domain_targeted"     # "domain_targeted" | "broad"
    chembl_pretrain_target_keywords: Sequence[str] = field(default_factory=lambda: [
        "glutamate carboxypeptidase",       # PSMA's own family (paralogs, e.g. NAALADase L)
        "aminopeptidase",                   # other M-clan zinc peptidases, same catalytic chemistry
        "leucine aminopeptidase",
        "carboxypeptidase",
        "angiotensin-converting enzyme",    # zinc metallopeptidase; classic phosphonate/thiol ZBGs
        "neprilysin",                       # zinc metallopeptidase (NEP), thiol/phosphonate inhibitors
        "endothelin-converting enzyme",     # related M13 zinc metallopeptidase
        "thermolysin",                      # archetypal zinc metallopeptidase used to design ZBGs
        "matrix metalloproteinase",         # zinc metalloproteinases, hydroxamate ZBGs
        "insulin-degrading enzyme",         # zinc metallopeptidase
        "meprin",                           # zinc metalloproteinase
        "carbonic anhydrase",               # different fold, but a classic zinc metalloenzyme drug
                                             # target with a rich, structurally-diverse zinc-binding
                                             # inhibitor literature (sulfonamides, etc.)
    ])
    chembl_pretrain_target_type: str = "SINGLE PROTEIN"
    chembl_pretrain_min_targets_or_fallback_to_broad: int = 5   # if fewer targets match, fall back to "broad"
    chembl_pretrain_cache: str = "./ad_pipeline_data/chembl_pretrain_domain.csv"
    chembl_pretrain_standard_types: Sequence[str] = field(
        default_factory=lambda: ["IC50", "Ki", "Kd", "EC50"]
    )
    chembl_pretrain_organism: str = "Homo sapiens"
    chembl_pretrain_max_targets: int = 150      # cap on distinct targets sampled ("broad" mode only)
    chembl_pretrain_per_target_cap: int = 300   # max activities pulled per target
    chembl_pretrain_max_compounds: int = 10000   # hard cap on the final pretraining set
    chembl_pretrain_endpoint_name: str = "pX_generic"   # generic potency label used only for pretraining
    force_rebuild_chembl_cache: bool = False

    # ---- BindingDB acquisition (PSMA fine-tuning set) --------------------------
    bindingdb_tsv_zip_name: str = "BindingDB_All_202609_tsv.zip"
    bindingdb_download_url: str = (
        "https://www.bindingdb.org/rwd/bind/downloads/BindingDB_All_202609_tsv.zip"
    )
    auto_resolve_bindingdb_url: bool = True
    data_dir: str = "./ad_pipeline_data"
    force_redownload: bool = False

    # ---- Target / multi-task endpoint selection (BindingDB SQL filter) --------
    target_uniprot_id: Optional[str] = "Q04609"          # PSMA / FOLH1 / GCPII
    target_name_like: Optional[str] = "Glutamate carboxypeptidase 2"
    organism_like: Optional[str] = "Homo sapiens"

    # Multi-task: model BOTH endpoints jointly instead of picking one.
    affinity_columns: Sequence[str] = field(default_factory=lambda: ["IC50 (nM)", "Ki (nM)"])
    endpoint_names: Sequence[str] = field(default_factory=lambda: ["pIC50", "pKi"])

    max_affinity_nm: float = 1e6
    min_affinity_nm: float = 1e-3

    # ---- Curated screening library ----------------------------------------------
    screening_library_path: str = "/Enamine_screen_library.sdf"
    screening_smiles_col: str = "SMILES"
    screening_id_col: Optional[str] = "compound_id"

    # ---- Featurization ------------------------------------------------------------
    morgan_radius: int = 2
    morgan_n_bits: int = 2048
    feature_set: str = "fp_plus_descriptors"   # "fp_plus_descriptors" | "descriptors_only" | "fp_only"

    # ---- Applicability domain thresholds (calibrated on the PSMA fine-tuning set) --
    ad_percentile_threshold: float = 95.0
    unfamiliarity_k: int = 5
    latent_pca_components: int = 20

    # ---- Neural-network transfer-learning model ------------------------------------
    nn_hidden_dims: Sequence[int] = field(default_factory=lambda: (256, 128))
    nn_dropout: float = 0.2
    nn_lr: float = 1e-3
    nn_finetune_lr: float = 5e-4
    nn_weight_decay: float = 1e-5
    nn_pretrain_epochs: int = 40
    nn_finetune_total_epochs: int = 80
    nn_freeze_epochs: int = 15          # epochs of head-only training before unfreezing the trunk
    nn_batch_size: int = 32
    nn_pretrain_batch_size: int = 128
    nn_mc_dropout_passes: int = 30      # forward passes for MC-dropout uncertainty

    # ---- LightGBM baseline (trained directly on BindingDB, one model per task) -----
    n_bootstrap: int = 30               # bootstrap ensemble size for GBM uncertainty

    # ---- Hyperparameter search ------------------------------------------------------
    run_gridsearch: bool = True
    run_bayes_search: bool = True       # scikit-optimize BayesSearchCV
    run_optuna: bool = True             # Optuna TPE search for the NN
    bayes_search_n_iter: int = 20
    optuna_n_trials: int = 25
    optuna_cv_folds: int = 3            # cheap inner CV used inside the Optuna objective

    test_size: float = 0.2
    cv_folds: int = 10
    random_state: int = 42
    n_eval_repeats: int = 10          # repeated random splits for robust held-out statistics
    n_calibration_bins: int = 5       # uncertainty-calibration reliability diagram bins

    # ---- Output ----------------------------------------------------------------------
    output_dir: str = "./ad_pipeline_outputs/f9"


    # ================================================================================
    # F6: single-task activity classification
    # ================================================================================
    # The data pull below still fetches BOTH endpoints, because Part A.2b's diagnostic
    # needs them to demonstrate *why* pKi is dropped. Everything from Part A.2c onward
    # models `primary_endpoint` alone.
    task_mode: str = "classification"        # "classification" | "regression"

    # ---- F7: multi-task over both endpoints -------------------------------------
    # F6 modelled pIC50 alone and so never saw the 229 compounds that carry only a Ki
    # measurement - a third of the data. Both endpoints are modelled jointly again, with
    # a masked loss so a compound needs only one of the two labels to contribute.
    # ---- F9: single-task pKi ------------------------------------------------------
    # Both endpoints are still PULLED, because Part A.1b's diagnostic needs pIC50 to
    # detect which pKi values were derived from it. Only pKi is MODELLED.
    #
    # Why pKi and not pIC50: F8's leave-one-cluster-out put pKi's best model at 0.796
    # pooled ROC-AUC against pIC50's 0.770, and more decisively, pKi was the endpoint
    # where the models actually cleared the nearest-neighbour floor (+0.11 AUC over
    # Tanimoto kNN) while on pIC50 nothing cleared it at all. pKi is where there is
    # signal to model.
    #
    # Note what this costs: pKi has 282 labelled compounds against pIC50's 483. The
    # classical models were already single-task per endpoint, so nothing is lost there;
    # only the neural network had been sharing a trunk, and it was the weakest model on
    # pKi (0.688, tied with lookup). Hence use_neural_network = False below.
    multitask: bool = False
    modelled_endpoints: Sequence[str] = field(default_factory=lambda: ["pKi"])
    primary_endpoint: str = "pKi"
    primary_affinity_column: str = "Ki (nM)"
    derived_endpoint: str = "pKi"            # the endpoint whose values may be computed

    # Compounds with a pIC50 but no pKi are still carried through, with a NaN label. They
    # cannot train the model, but they are known PSMA ligands, so they still define the
    # applicability domain and still shape the Butina clustering that builds the folds.
    # Dropping them would shrink the domain to 282 compounds for no reason.
    keep_unlabelled_for_domain: bool = True

    # Per-endpoint active cut, on the pAffinity scale. Both are 10 nM, i.e. the SAME
    # pharmacological potency - which is only coherent once the derived Ki values are
    # removed (see drop_derived_endpoint_values).
    active_thresholds: Dict[str, float] = field(
        default_factory=lambda: {"pIC50": 8.0, "pKi": 8.0}
    )

    # ---- Derived-value removal ---------------------------------------------------
    # Part A.2b found 139 of 169 dual-measured compounds sitting in fixed-offset blocks:
    # 111 at IC50/Ki = 140.0 (US20250213705) and 28 at 5.01 (US10736974). Those Ki values
    # were computed from the IC50 values by Cheng-Prusoff, not measured. They carry no
    # information AND they break any single pKi threshold, so the pKi label is dropped for
    # those compounds while their pIC50 is kept.
    drop_derived_endpoint_values: bool = True
    derived_min_block_size: int = 5          # compounds sharing an offset before it counts
    derived_offset_tol: float = 0.02         # log units

    # ---- Transfer learning: OFF ---------------------------------------------------
    # Measured twice. F4 regression: +0.062 R2 on random splits (10/10, p=0.002), nothing
    # on grouped. F6 classification: -0.0066 AUC on random splits (1/10, p=0.020), nothing
    # on grouped. Effects under 0.01 AUC with the sign flipping by objective. The code
    # path is kept so the claim stays reproducible, not deleted.
    use_pretraining: bool = False
    run_pretraining_ablation: bool = False   # True re-measures it (costs one pretrain pass)

    # ---- Probability recalibration ------------------------------------------------
    # F6 predicted ~0.50 where the observed rate was 0.265 (ECE 0.24-0.31) because the
    # balanced class weighting pushes outputs toward 0.5. Cross-fitted across the LOCO
    # folds: each fold is calibrated on the OTHER folds' out-of-fold predictions.
    # Platt (sigmoid) by default, not isotonic. The miscalibration being corrected is a
    # systematic ~2x inflation from the balanced class weighting, which is close to a pure
    # logit shift - exactly what a 2-parameter sigmoid models. Isotonic is more flexible
    # but fits a step function, and with only a handful of LOCO folds it overfits and
    # collapses distinct scores into ties, which destroys ranking resolution (a smoke test
    # on noise features moved AUC by 0.10 that way). Switch to "isotonic" if the
    # before/after table shows a miscalibration shape a sigmoid cannot follow.
    recalibration: str = "sigmoid"           # "sigmoid" (Platt) | "isotonic" | "none"

    # ---- Leave-one-cluster-out folds ----------------------------------------------
    # F6 reported 8 folds of which 4 could not be scored (0 or 1 actives, or a 71% base
    # rate on 14 compounds). A cluster now needs size AND actives to get its own row.
    loco_min_cluster_size: int = 25
    loco_min_fold_actives: int = 5
    loco_n_remainder_folds: int = 3          # size-balanced folds for everything smaller

    # ---- The model zoo -------------------------------------------------------------
    # F3-F7 ran a committee of three: the multi-task network, LightGBM, and kernel PLS-DA.
    # F8 adds four classical QSAR classifiers, all cheap at this scale (~500 x 2048) and
    # all with different inductive biases:
    #
    #   RandomForest     - the QSAR workhorse; bagged trees, different bias to boosting
    #   SVM (Tanimoto)   - support vectors on a precomputed Tanimoto kernel. The strongest
    #                      classical method for fingerprint classification at small n, and
    #                      the kernel is the chemists' own similarity measure rather than
    #                      a Euclidean stand-in for it
    #   kNN (Tanimoto)   - the similarity principle, unadorned. Kept mainly as a DIAGNOSTIC:
    #                      if the committee cannot beat "it looks like a known active", then
    #                      nothing is being learned beyond lookup, which given the
    #                      applicability-domain findings is exactly the question worth
    #                      asking
    #   LogReg (L2)      - the linear floor. If the ensembles cannot beat it, the
    #                      complexity is not paying for itself
    #   BernoulliNB      - Laplacian-corrected naive Bayes on fingerprint bits, the old
    #                      Pipeline Pilot standard; near-free and a useful oddity
    #
    # `enabled_models` controls what gets fitted and scored. `committee` controls which
    # members enter the blend, PER ENDPOINT - F7 found the network helped on pIC50 and
    # actively diluted the blend on pKi (0.763 with it, 0.796 without), so membership is
    # no longer one global choice.
    # kNN and LogReg stay enabled as FLOORS even though kNN is not in the committee:
    # Part E.2b measures every model against them.
    enabled_models: Sequence[str] = field(default_factory=lambda: [
        "LightGBM", "KPLS-DA", "RandomForest", "SVM (Tanimoto)",
        "kNN (Tanimoto)", "LogReg (L2)", "BernoulliNB",
    ])
    # F8 pooled ROC-AUC on pKi: LightGBM 0.796, LogReg 0.780, KPLS-DA 0.775,
    # SVM (Tanimoto) 0.772, RandomForest 0.769, BernoulliNB 0.764, kNN 0.688, NN 0.688.
    # Everything except kNN and the NN cleared the lookup floor; nothing cleared LogReg.
    # The committee is the five that cleared it, and the blend is reported against each
    # member individually since in F8 it did not beat its best member on either endpoint.
    committee: Dict[str, Sequence[str]] = field(default_factory=lambda: {
        "pKi": ["LightGBM", "LogReg (L2)", "KPLS-DA", "SVM (Tanimoto)", "RandomForest"],
    })
    # The network is gone. Two ablations found pretraining did nothing (F4 regression:
    # +0.062 R2 on random splits, nothing on grouped; F6 classification: -0.0066 AUC),
    # and on pKi the network itself was the joint-worst model. Removing it also removes
    # the ChEMBL acquisition, the pretraining pass and the Optuna search.
    use_neural_network: bool = False
    screen_chunk_size: int = 20000           # rows per block when scoring the library

    # ---- Reporting floors ----------------------------------------------------------
    # F7 reported a pKi fold with n = 5, whose metrics ranged 0.500-0.917 across models
    # and meant nothing. A per-fold row now needs this many labelled test compounds.
    loco_min_fold_n: int = 10
    # F7's pKi probabilities were already well calibrated (ECE 0.072-0.084) because that
    # endpoint's base rate is ~0.5 and balanced class weighting naturally targets 0.5.
    # Recalibrating them anyway made the blend worse (0.084 -> 0.167). Skip when the raw
    # calibration is already this good.
    recalibrate_min_ece: float = 0.10

    # ---- Deduplication -------------------------------------------------------------
    # On the InChIKey of the RDKit-STANDARDISED structure, so LigPrep protonation and
    # tautomer variants collapse. Applies to the library and the training set; the latter
    # closes the old issue of deduplicating on raw SMILES before standardisation.
    dedupe_on_inchikey: bool = True

    # ---- F9: where the screening library comes from --------------------------------
    # "enumeration+library" (default) enumerates a virtual library from reaction SMARTS
    # and building blocks, then mixes in a stratified sample of the catalogue file as
    # negative controls, property-matched decoys and scaffold-hop candidates.
    # "enumeration" is the virtual library alone; "file" is the old behaviour.
    screening_source: str = "enumeration+library"
    enum_max_products: Optional[int] = None      # cap on raw products before dedup
    enum_property_quantile: float = 0.05         # window = this..1-this of the ACTIVES
    enum_apply_property_filter: bool = True      # False flags instead of removing
    # Floor on the property filter: if the window would keep less than this fraction
    # of the enumerated products, it is reported and flagged rather than applied, so a
    # narrow window cannot silently empty the library.
    enum_property_min_keep_frac: float = 0.20

    mix_library_path: str = "/Enamine_screen_library.sdf"
    mix_pool_sample: int = 60000     # pre-sample the file before distance computation
    mix_n_near: int = 5000           # nearest to training — the scaffold-hop candidates
    mix_n_matched: int = 10000       # property-matched decoys (MW-matched to enumerated)
    mix_n_random: int = 5000         # unbiased background
    mix_mw_tolerance: float = 25.0   # MW slack when matching a bin

    # ---- Library prefilter (OFF by default) ----------------------------------------
    # The file in use is a reagent/building-block catalogue, not a screening set. A
    # drug-likeness cut is available, but silently discarding a third of a library is
    # worse than being told the library is wrong, so Part A.3 reports and does not act.
    library_prefilter: bool = False
    prefilter_min_heavy_atoms: int = 12
    prefilter_mw_range: Sequence[float] = field(default_factory=lambda: (150.0, 700.0))

    # Pre-registered class boundaries on the pAffinity scale. Anchored on pharmacology
    # (10 nM / 1 uM), NOT on quantiles of this particular dataset - a quantile cut makes
    # "active" mean "top third of this snapshot", which is not transferable and shifts
    # when data is added.
    n_activity_classes: int = 2              # 2 = active vs not-active; 3 = + moderate
    active_threshold: float = 8.0            # pIC50 >= 8.0  <=>  IC50 <= 10 nM
    inactive_threshold: float = 6.0          # pIC50 <  6.0  <=>  IC50 >  1 uM (3-class only)
    keep_censored_as_labels: bool = True     # recover '>' rows as inactives, '<' as actives
    class_weight_balanced: bool = True       # class-weighted loss / fitting

    # Exploratory: down-weight over-represented sources (one patent is 26% of the
    # pIC50 labels). Inverse-sqrt-frequency per source tag. Default OFF - this is a
    # modelling choice with no obviously right answer, exposed so its effect is
    # measurable rather than assumed.
    downweight_dominant_source: bool = False

    # ---- Applicability domain ---------------------------------------------------
    # F5's consensus rule let compounds pass while failing the Tanimoto test, which for a
    # congeneric target is the flag with teeth: all 15 F5 top hits were tanimoto_OOD and
    # all 15 passed. With the veto on, failing Tanimoto alone puts a compound out of
    # domain regardless of the other three metrics.
    ad_tanimoto_veto: bool = True
    ad_consensus_min_flags: int = 3

    # ---- Leave-one-cluster-out evaluation ---------------------------------------
    butina_cutoff: float = 0.4               # Tanimoto distance cutoff for clustering
    loco_max_clusters: int = 12              # largest N clusters get their own fold
    # NOTE: `loco_min_cluster_size` is defined once, in the F7 block above. F7 shipped it
    # twice (25 there, 12 here) and Python silently took the later value, so the config
    # misreported what ran. Only `loco_random_reference_repeats` belongs here now.
    loco_random_reference_repeats: int = 10  # random-split repeats, for comparison only

    # ---- Screening metrics -------------------------------------------------------
    enrichment_fractions: Sequence[float] = field(
        default_factory=lambda: [0.01, 0.05, 0.10]
    )
    precision_at_n: Sequence[int] = field(default_factory=lambda: [50, 100, 200])

    def __post_init__(self):
        Path(self.data_dir).mkdir(parents=True, exist_ok=True)
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)


CONFIG = Config()
CONFIG

## 2. Part A.1 — BindingDB PSMA set (both endpoints pulled, only pKi modelled)

Pulls both affinity columns and pivots to a wide, per-compound table with a `pIC50` column,
a `pKi` column and `NaN` where a compound lacks one. Both are fetched because F7 models
both: of 681 compounds, 283 carry only pIC50, 169 carry both, and **229 carry only pKi** —
so a single-task pIC50 model, as F6 was, cannot see a third of the data.

Part A.1b diagnoses the distributions and Part A.1c turns the pKi column into class
labels, dropping the pKi values that turn out to be arithmetic transforms of pIC50
rather than independent measurements. **pIC50 is still pulled**, for two reasons: the
derived-value detector needs it to find those transforms, and a compound measured only
by IC50 is still a known PSMA ligand, so it is carried through unlabelled to define
the applicability domain and shape the clustering.

In [ ]:
def resolve_bindingdb_tsv_url(cfg: Config) -> tuple:
    '''Scrape the BindingDB download page for the CURRENT 'BindingDB_All_<YYYYMM>_tsv.zip'
    filename (BindingDB republishes this monthly, so a hardcoded name goes stale).'''
    download_page = "https://www.bindingdb.org/rwd/bind/chemsearch/marvin/Download.jsp"
    try:
        resp = requests.get(download_page, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        resp.raise_for_status()
        m = re.search(r"download_file=(/rwd/bind/downloads/BindingDB_All_\d{6}_tsv\.zip)", resp.text)
        if m:
            file_param = m.group(1)
            zip_name = file_param.rsplit("/", 1)[-1]
            direct_url = f"https://www.bindingdb.org{file_param}"
            print(f"Resolved current BindingDB TSV: {zip_name}")
            return direct_url, zip_name
        print(f"Could not find the current filename on {download_page}; using CONFIG default.")
    except Exception as e:
        print(f"Could not reach {download_page} ({e}); using CONFIG default.")
    return cfg.bindingdb_download_url, cfg.bindingdb_tsv_zip_name


def _fetch_to_file(url: str, dest: Path) -> str:
    headers = {"User-Agent": "Mozilla/5.0 (compatible; AD-pipeline/1.0)"}
    with requests.get(url, stream=True, timeout=120, headers=headers) as r:
        r.raise_for_status()
        content_type = r.headers.get("content-type", "")
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as pbar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                pbar.update(len(chunk))
    return content_type


def _recover_real_download_link(html_path: Path) -> Optional[str]:
    try:
        text = html_path.read_text(errors="ignore")
    except Exception:
        return None
    m = re.search(r'(https://www\.bindingdb\.org/rwd/bind/downloads/[^\s"\x27<>)]+\.zip)', text)
    return m.group(1) if m else None


def download_bindingdb(cfg: Config) -> Optional[Path]:
    '''Download (if needed) and unzip the full BindingDB TSV dump. Returns None (instead of
    raising) if the download can't be completed, so the notebook can fall back to a
    synthetic fine-tuning set and still run end-to-end.'''
    data_dir = Path(cfg.data_dir)
    try:
        if cfg.auto_resolve_bindingdb_url:
            url, zip_name = resolve_bindingdb_tsv_url(cfg)
        else:
            url, zip_name = cfg.bindingdb_download_url, cfg.bindingdb_tsv_zip_name

        zip_path = data_dir / zip_name
        tsv_path = data_dir / zip_name.replace(".zip", "")

        if tsv_path.exists() and not cfg.force_redownload:
            print(f"Found existing TSV: {tsv_path}")
            return tsv_path

        if not zip_path.exists() or cfg.force_redownload:
            print(f"Downloading {url} ...")
            content_type = _fetch_to_file(url, zip_path)
            print(f"Saved to {zip_path} (content-type: {content_type})")

            if not zipfile.is_zipfile(zip_path):
                real_url = _recover_real_download_link(zip_path)
                if real_url and real_url != url:
                    print(f"Not a zip — retrying with the real link found on the landing page: {real_url}")
                    _fetch_to_file(real_url, zip_path)

        if not zipfile.is_zipfile(zip_path):
            with open(zip_path, "rb") as f:
                head = f.read(300)
            zip_path.unlink(missing_ok=True)
            raise RuntimeError(f"Downloaded file is not a valid zip. First bytes: {head!r}")

        print("Unzipping ...")
        with zipfile.ZipFile(zip_path) as zf:
            member = [m for m in zf.namelist() if m.lower().endswith(".tsv")][0]
            with zf.open(member) as src, open(tsv_path, "wb") as dst:
                shutil.copyfileobj(src, dst)
        print(f"Extracted: {tsv_path}")
        return tsv_path

    except Exception as e:
        print(f"Could not obtain the BindingDB TSV dump ({e}).")
        return None


def query_bindingdb_duckdb_multitask(tsv_path: Path, cfg: Config) -> pd.DataFrame:
    '''Pull one long dataframe with a row per (compound, endpoint) for every column in
    `cfg.affinity_columns`, using DuckDB SQL streaming over the TSV (never materialized
    fully in pandas).'''
    con = duckdb.connect(database=":memory:")
    con.execute("PRAGMA threads=4;")
    con.execute(f'''
        CREATE OR REPLACE VIEW bdb AS
        SELECT * FROM read_csv(
            '{tsv_path.as_posix()}', delim='\t', header=True, quote='"', escape='"',
            strict_mode=False, ignore_errors=True, all_varchar=True
        )
    ''')

    base_where = ['"Ligand SMILES" IS NOT NULL']
    if cfg.target_uniprot_id:
        uniprot_conditions = [
            f'"UniProt (SwissProt) Primary ID of Target Chain {i}" = \'{cfg.target_uniprot_id}\''
            for i in range(1, 6)
        ]
        base_where.append("(" + " OR ".join(uniprot_conditions) + ")")
    if cfg.target_name_like:
        safe = cfg.target_name_like.replace("'", "''")
        base_where.append(f'"Target Name" ILIKE \'%{safe}%\'')
    if cfg.organism_like:
        safe = cfg.organism_like.replace("'", "''")
        base_where.append(f'"Target Source Organism According to Curator or DataSource" ILIKE \'%{safe}%\'')

    frames = []
    for col, endpoint in zip(cfg.affinity_columns, cfg.endpoint_names):
        where_sql = " AND ".join(base_where + [f'"{col}" IS NOT NULL'])
        query = f'''
            SELECT
                "Ligand SMILES"                                    AS smiles,
                "BindingDB Ligand Name"                            AS ligand_name,
                "{col}"                                            AS affinity_raw,
                '{endpoint}'                                       AS endpoint_name
            FROM bdb
            WHERE {where_sql}
        '''
        part = con.execute(query).fetch_df()
        print(f"  {endpoint} ('{col}'): {len(part):,} raw rows.")
        frames.append(part)
    con.close()
    return pd.concat(frames, ignore_index=True)


def _synthetic_bindingdb_finetune_set(cfg: Config, n: int = 250) -> pd.DataFrame:
    print("!! Using SYNTHETIC BindingDB PSMA fine-tuning data (network/download unavailable) !!")
    demo_smiles = [
        "CCOc1ccc2nc(S(N)(=O)=O)sc2c1", "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
        "O=C(O)c1ccccc1O", "CC(=O)Oc1ccccc1C(=O)O",
        "Clc1ccc(C(c2ccccc2Cl)N2CCN(CCO)CC2)cc1",
        "COc1cc2c(cc1OC)C(=O)C(CC1CCN(C)CC1)C2",
        "CN1CCC[C@H]1c1cccnc1", "CC(C)NCC(O)COc1cccc2ccccc12",
        "NC(CCC(=O)O)C(=O)O", "OC(=O)CCC(NC(=O)NC(CCC(=O)O)C(=O)O)C(=O)O",
    ]
    rng = np.random.RandomState(cfg.random_state)
    rows = []
    for i in range(n):
        smi = demo_smiles[i % len(demo_smiles)]
        row = {"smiles": smi, "ligand_name": f"synthetic_{i}"}
        if rng.rand() < 0.75:
            row[cfg.endpoint_names[0]] = float(np.clip(rng.normal(6.8, 1.2), 3, 10))
        if rng.rand() < 0.6:
            row[cfg.endpoint_names[1]] = float(np.clip(rng.normal(6.5, 1.3), 3, 10))
        rows.append(row)
    return pd.DataFrame(rows)


def clean_bindingdb_multitask(df_long: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    '''Coerce to numeric, drop censored (>,<) measurements, compute pAffinity, take the
    per-compound median for each endpoint separately, then pivot wide so every
    compound is one row with (possibly-NaN) pIC50 / pKi columns.'''
    df = df_long.copy()
    df["is_censored"] = df["affinity_raw"].astype(str).str.contains("[<>]", regex=True)
    df = df[~df["is_censored"]]
    df["affinity_nm"] = pd.to_numeric(df["affinity_raw"], errors="coerce")
    df = df.dropna(subset=["affinity_nm", "smiles"])
    df = df[(df["affinity_nm"] >= cfg.min_affinity_nm) & (df["affinity_nm"] <= cfg.max_affinity_nm)]
    df["pAffinity"] = 9 - np.log10(df["affinity_nm"])

    per_endpoint = (
        df.groupby(["smiles", "endpoint_name"], as_index=False)
        .agg({"pAffinity": "median", "ligand_name": "first"})
    )
    wide = per_endpoint.pivot_table(index="smiles", columns="endpoint_name", values="pAffinity")
    names = per_endpoint.groupby("smiles")["ligand_name"].first()
    wide = wide.join(names).reset_index()
    for ep in cfg.endpoint_names:
        if ep not in wide.columns:
            wide[ep] = np.nan
    print(f"Cleaned multi-task fine-tuning set: {len(wide):,} unique compounds "
          + ", ".join(f"({ep}: {wide[ep].notna().sum():,})" for ep in cfg.endpoint_names))
    return wide.reset_index(drop=True)


# --- run it -------------------------------------------------------------------
tsv_path = download_bindingdb(CONFIG)
if tsv_path is not None:
    raw_bdb_df = query_bindingdb_duckdb_multitask(tsv_path, CONFIG)
    train_df = clean_bindingdb_multitask(raw_bdb_df, CONFIG)
else:
    train_df = _synthetic_bindingdb_finetune_set(CONFIG)

train_df.head()


## 2b. Part A.1b — Where should the activity-class cut go?

A read-only diagnostic, added to settle the threshold question before any switch from
numeric prediction to activity classes. It defines nothing the rest of the pipeline
consumes, so it can be run or skipped freely.

It needs `raw_bdb_df` and `train_df`, both created by the cell above — `raw_bdb_df`
is the **pre-cleaning** long frame, so it still contains the censored (`>` / `<`)
measurements that `clean_bindingdb_multitask` discards. That is what makes the
censored-row audit possible without re-downloading anything. If the kernel has been
restarted, re-run the cell above first (it reuses the cached TSV).

Four questions, in order:

1. **Shape.** Percentiles, skew, kurtosis, a Gaussian-mixture BIC comparison for
   1 / 2 / 3 components, and a kernel-density scan for the *valleys* — the emptiest
   places to put a boundary, where a cut splits the fewest near-identical compounds.
2. **Censored-row audit.** How many `>` and `<` rows were dropped per endpoint, where
   their bounds sit (a `>1 µM` is a far weaker claim than a `>100 µM`), and how many
   distinct compounds they would add back as confident inactives or actives. This is
   the single number that decides three-class versus binary.
3. **Class counts** under candidate schemes side by side: field-standard
   (100 nM / 10 µM), PSMA-tightened (10 nM / 1 µM), PSMA-strict (1 nM / 1 µM), plus
   observed tertiles and the density valleys for comparison — each shown before and
   after the censored data is folded in.
4. **Boundary noise.** For compounds measured both ways, how often pIC50 and pKi land
   in *different* classes. That disagreement is assay noise on the same molecule, so it
   is a hard ceiling on classifier accuracy. A scheme below roughly 80% agreement has
   boundaries cutting through noise rather than signal.

Note on interpretation: anchor the thresholds on pharmacology and use the distribution
to *check* them — that each class has enough members, and that the cut does not land on
a density peak. Choosing tertiles because they balance the classes makes "active" mean
"top third of this particular snapshot", which is not transferable and shifts the moment
ChEMBL data or decoys are added.


In [ ]:
# =====================================================================================
# Part A.2b — Label-distribution diagnostic: where should the activity-class cuts go?
#
# Drop this in as a new cell immediately AFTER cell 9 (the BindingDB pull), which is
# where `raw_bdb_df` and `train_df` come into existence. It is read-only: it adds no
# variables the rest of the pipeline depends on and changes nothing.
#
# It answers four questions:
#   1. What shape is the label distribution really? (percentiles, skew, bimodality)
#   2. How many censored (">" / "<") rows were dropped, and what would they add back
#      as confident inactives / actives?
#   3. How many compounds land in each class under candidate threshold schemes?
#   4. How noisy is the labelling at the boundaries, measured by pIC50-vs-pKi
#      disagreement on compounds that have both?
# =====================================================================================
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

EPS = list(CONFIG.endpoint_names)          # ['pIC50', 'pKi']
_COLS = dict(zip(CONFIG.endpoint_names, CONFIG.affinity_columns))

print("=" * 86)
print("1. SHAPE OF THE LABEL DISTRIBUTION")
print("=" * 86)

_shape_rows = []
for ep in EPS:
    v = train_df[ep].dropna().values
    if len(v) < 10:
        continue
    q = np.percentile(v, [1, 5, 10, 25, 50, 75, 90, 95, 99])
    _shape_rows.append({
        "endpoint": ep, "n": len(v),
        "mean": v.mean(), "sd": v.std(ddof=0),
        "p1": q[0], "p5": q[1], "p10": q[2], "p25": q[3], "p50": q[4],
        "p75": q[5], "p90": q[6], "p95": q[7], "p99": q[8],
        "skew": stats.skew(v), "excess_kurt": stats.kurtosis(v),
        "min": v.min(), "max": v.max(),
    })
shape_df = pd.DataFrame(_shape_rows).set_index("endpoint").round(3)
display(shape_df)

# --- is it one mode or several? -------------------------------------------------------
# A kernel-density scan: local minima of the density between the 5th and 95th percentile
# are the natural "valleys", i.e. the least-populated places to put a cut.
def density_valleys(v, n_grid=512, lo_pct=5, hi_pct=95):
    kde = stats.gaussian_kde(v)
    lo, hi = np.percentile(v, [lo_pct, hi_pct])
    grid = np.linspace(lo, hi, n_grid)
    dens = kde(grid)
    # interior local minima
    is_min = (dens[1:-1] < dens[:-2]) & (dens[1:-1] < dens[2:])
    valleys = grid[1:-1][is_min]
    depths = dens[1:-1][is_min]
    order = np.argsort(depths)              # shallowest density = emptiest = best cut
    return grid, dens, valleys[order], depths[order]

print("\nBimodality check and natural density valleys")
print("-" * 86)
valley_store = {}
for ep in EPS:
    v = train_df[ep].dropna().values
    if len(v) < 30:
        print(f"  {ep}: too few labelled rows for a density scan.")
        continue
    grid, dens, valleys, depths = density_valleys(v)
    valley_store[ep] = (grid, dens, valleys)
    # Crude unimodality signal: compare the best single Gaussian to a 2-component mix.
    try:
        from sklearn.mixture import GaussianMixture
        X = v.reshape(-1, 1)
        bic1 = GaussianMixture(1, random_state=0).fit(X).bic(X)
        bic2 = GaussianMixture(2, random_state=0).fit(X).bic(X)
        bic3 = GaussianMixture(3, random_state=0).fit(X).bic(X)
        best = {1: bic1, 2: bic2, 3: bic3}
        n_modes = min(best, key=best.get)
        verdict = (f"{n_modes}-component mixture fits best "
                   f"(BIC 1/2/3 = {bic1:.0f} / {bic2:.0f} / {bic3:.0f})")
    except Exception as e:
        verdict = f"mixture fit unavailable ({e})"
    vstr = ", ".join(f"{x:.2f}" for x in valleys[:4]) if len(valleys) else "none found"
    print(f"  {ep}: {verdict}")
    print(f"         emptiest cut points (shallowest density first): {vstr}")
    print(f"         -> a threshold placed in a valley splits fewer near-identical "
          f"compounds across the boundary")

print("\n" + "=" * 86)
print("2. CENSORED-ROW AUDIT  (the data classification would recover)")
print("=" * 86)

_cens_rows = []
_recovered = {}          # endpoint -> DataFrame of censored measurements
for ep in EPS:
    col = _COLS[ep]
    sub = raw_bdb_df[raw_bdb_df["endpoint_name"] == ep].copy()
    raw = sub["affinity_raw"].astype(str)
    sub["qualifier"] = raw.str.extract(r"^\s*([<>]=?)")[0]
    sub["value_nm"] = pd.to_numeric(raw.str.replace(r"^\s*[<>]=?\s*", "", regex=True),
                                    errors="coerce")
    cens = sub[sub["qualifier"].notna() & sub["value_nm"].notna()].copy()
    cens["pAffinity_bound"] = 9 - np.log10(cens["value_nm"])
    _recovered[ep] = cens

    greater = cens[cens["qualifier"].str.startswith(">")]
    lesser = cens[cens["qualifier"].str.startswith("<")]
    _cens_rows.append({
        "endpoint": ep,
        "raw_rows": len(sub),
        "censored_rows": len(cens),
        "gt_rows_INACTIVE": len(greater),
        "lt_rows_ACTIVE": len(lesser),
        "gt_distinct_cpds": greater["smiles"].nunique(),
        "lt_distinct_cpds": lesser["smiles"].nunique(),
        "numeric_cpds_now": int(train_df[ep].notna().sum()),
    })

cens_df = pd.DataFrame(_cens_rows).set_index("endpoint")
display(cens_df)

print("Where the '>' bounds sit (these become inactives; a '>1 uM' is a much weaker")
print("statement than a '>100 uM', so they are not interchangeable):")
for ep in EPS:
    g = _recovered[ep]
    g = g[g["qualifier"].str.startswith(">")]
    if len(g) == 0:
        print(f"  {ep}: no '>' rows.")
        continue
    vc = g["value_nm"].value_counts().sort_index()
    print(f"  {ep}:")
    for val, n in vc.items():
        print(f"      > {val:>12,.1f} nM  (pAffinity < {9 - np.log10(val):.2f})   n = {n}")

# How many of the '>' compounds are genuinely NEW, vs already present with a number?
print("\nOverlap with the compounds already in train_df:")
for ep in EPS:
    g = _recovered[ep]
    g = g[g["qualifier"].str.startswith(">")]
    if len(g) == 0:
        continue
    new = set(g["smiles"]) - set(train_df["smiles"])
    print(f"  {ep}: {g['smiles'].nunique()} distinct '>' compounds, "
          f"of which {len(new)} are not in the current training set at all")
print("  (Note: matching is on raw SMILES strings, as the pipeline does, so this is an")
print("   under-count of true overlap - standardisation would merge more. Dedup on")
print("   InChIKey after standardisation before trusting these numbers.)")

print("\n" + "=" * 86)
print("3. CLASS COUNTS UNDER CANDIDATE THRESHOLD SCHEMES")
print("=" * 86)

# (name, active_cut, inactive_cut) on the pAffinity scale; class is
#   ACTIVE    if  p >= active_cut
#   INACTIVE  if  p <  inactive_cut
#   MODERATE  otherwise
_p = lambda nm: 9 - np.log10(nm)

SCHEMES = {
    "field-standard (100 nM / 10 uM)": (_p(100), _p(10_000)),
    "PSMA-tightened (10 nM / 1 uM)":   (_p(10),  _p(1_000)),
    "PSMA-strict (1 nM / 1 uM)":       (_p(1),   _p(1_000)),
}
# add tertiles of the observed numeric distribution, per endpoint, for comparison only
for ep in EPS:
    v = train_df[ep].dropna().values
    if len(v) > 30:
        lo, hi = np.percentile(v, [33.3, 66.7])
        SCHEMES[f"tertiles of observed {ep} ({hi:.2f} / {lo:.2f})"] = (hi, lo)
# add the emptiest density valleys, if two were found
for ep in EPS:
    if ep in valley_store and len(valley_store[ep][2]) >= 2:
        two = sorted(valley_store[ep][2][:2])
        SCHEMES[f"density valleys of {ep} ({two[1]:.2f} / {two[0]:.2f})"] = (two[1], two[0])


def classify(p, active_cut, inactive_cut):
    if not np.isfinite(p):
        return np.nan
    if p >= active_cut:
        return "active"
    if p < inactive_cut:
        return "inactive"
    return "moderate"


rows = []
for name, (a_cut, i_cut) in SCHEMES.items():
    for ep in EPS:
        v = train_df[ep].dropna()
        cls = v.map(lambda p: classify(p, a_cut, i_cut))
        counts = cls.value_counts()
        # censored '>' rows become inactives, '<' rows become actives
        g = _recovered[ep]
        add_inact = g[g["qualifier"].str.startswith(">")]
        # only count a '>' row as inactive if its bound actually implies inactivity
        add_inact = add_inact[add_inact["pAffinity_bound"] <= i_cut]
        add_act = g[g["qualifier"].str.startswith("<")]
        add_act = add_act[add_act["pAffinity_bound"] >= a_cut]
        rows.append({
            "scheme": name, "endpoint": ep,
            "active": int(counts.get("active", 0)),
            "moderate": int(counts.get("moderate", 0)),
            "inactive": int(counts.get("inactive", 0)),
            "add_inact": add_inact["smiles"].nunique(),
            "add_act": add_act["smiles"].nunique(),
            "inact_total": int(counts.get("inactive", 0)) + add_inact["smiles"].nunique(),
        })

class_counts = pd.DataFrame(rows).set_index(["scheme", "endpoint"])
display(class_counts)

print("Reading this table:")
print("  * 'inact_total' is the figure that decides 3-class vs binary. Under ~80")
print("    compounds and three classes will not train reliably - go binary")
print("    (active vs not-active) instead.")
print("  * A scheme that puts >70% of compounds in one class will need class weights")
print("    or stratified sampling whichever way you go.")
print("  * Prefer a pharmacologically meaningful cut that happens to sit near a density")
print("    valley. Do NOT pick tertiles because they balance the classes - that makes")
print("    'active' mean 'top third of this snapshot', which moves when data is added.")

print("\n" + "=" * 86)
print("4. BOUNDARY NOISE: do pIC50 and pKi agree on the class?")
print("=" * 86)
print("Compounds measured both ways should land in the same class. Where they do not,")
print("the disagreement is assay noise, and it bounds how good any classifier can be.\n")

both = train_df[train_df[EPS[0]].notna() & train_df[EPS[1]].notna()]
print(f"Compounds with both {EPS[0]} and {EPS[1]}: {len(both)}")
if len(both) >= 10:
    d = (both[EPS[0]] - both[EPS[1]]).abs()
    print(f"  |{EPS[0]} - {EPS[1]}|:  median {d.median():.2f}, mean {d.mean():.2f}, "
          f"90th pct {d.quantile(0.9):.2f}, max {d.max():.2f} log units")
    print(f"  fraction differing by >0.5 log: {100 * (d > 0.5).mean():.1f}%")
    print(f"  fraction differing by >1.0 log: {100 * (d > 1.0).mean():.1f}%")
    print(f"  Pearson r between the two endpoints: {both[EPS[0]].corr(both[EPS[1]]):.3f}\n")
    for name, (a_cut, i_cut) in SCHEMES.items():
        c0 = both[EPS[0]].map(lambda p: classify(p, a_cut, i_cut))
        c1 = both[EPS[1]].map(lambda p: classify(p, a_cut, i_cut))
        agree = (c0 == c1).mean()
        try:
            from sklearn.metrics import cohen_kappa_score
            kappa = cohen_kappa_score(c0, c1)
            kstr = f", kappa = {kappa:.3f}"
        except Exception:
            kstr = ""
        print(f"  {name:<46s} agreement = {100 * agree:5.1f}%{kstr}")
    print("\n  Agreement here is a CEILING on classifier accuracy: the same molecule,")
    print("  measured twice, disagrees this often. If a scheme falls below ~80%, its")
    print("  boundaries are cutting through the noise rather than through the signal.")
else:
    print("  Too few dual-measured compounds to assess boundary noise.")

print("\n" + "=" * 86)
print("5. PLOTS")
print("=" * 86)

fig, axes = plt.subplots(1, len(EPS), figsize=(7.5 * len(EPS), 4.8))
axes = np.atleast_1d(axes)
_cut_colors = plt.get_cmap("tab10")
for ax, ep in zip(axes, EPS):
    v = train_df[ep].dropna().values
    ax.hist(v, bins=40, color="#9aa0a6", edgecolor="white", linewidth=0.5,
            alpha=0.85, label=f"{ep} (n={len(v)})")
    if ep in valley_store:
        grid, dens, valleys = valley_store[ep]
        ax2 = ax.twinx()
        ax2.plot(grid, dens, color="#1a1a1a", linewidth=1.6, label="density")
        ax2.set_yticks([])
        for x in valleys[:3]:
            ax2.axvline(x, color="#1a1a1a", linestyle=":", linewidth=1.2, alpha=0.7)
    for ci, (name, (a_cut, i_cut)) in enumerate(list(SCHEMES.items())[:3]):
        col = _cut_colors(ci)
        ax.axvline(a_cut, color=col, linestyle="--", linewidth=1.8,
                   label=f"{name.split(' (')[0]}: active")
        ax.axvline(i_cut, color=col, linestyle="-.", linewidth=1.4, alpha=0.8)
    g = _recovered[ep]
    g = g[g["qualifier"].str.startswith(">")]
    if len(g):
        ax.scatter(g["pAffinity_bound"], np.full(len(g), ax.get_ylim()[1] * 0.04),
                   marker="|", s=180, color="#d1495b", alpha=0.6,
                   label=f"censored '>' bounds (n={len(g)})")
    ax.set_xlabel(ep)
    ax.set_ylabel("compounds")
    ax.set_title(f"{ep}: distribution, density valleys (dotted) and candidate cuts")
    ax.legend(fontsize=7, loc="upper left")
plt.tight_layout()
plt.show()

print("Dotted vertical lines are density valleys - the emptiest places to cut.")
print("Red ticks are the censored '>' bounds that classification would recover as")
print("inactives; where they pile up tells you where the real inactive mass sits.")

## 2c. Part A.1c — pKi classification labels

Four steps, in order. The assembly machinery is still written per endpoint, because
the derived-value detection genuinely needs both columns; `modelled_endpoints` is
what reduces the output to a single pKi label.

**Find the derived values.** `detect_derived_endpoint` looks for blocks of compounds whose
two endpoints differ by a *constant*. A fixed offset shared across many compounds from one
source is the signature of one endpoint being computed from the other — Cheng–Prusoff
Ki = IC50 / (1 + [S]/Km) with a single substrate concentration produces exactly this — and
not of two independent assays. F6's version reported overlapping rounding windows as
separate blocks (three rows for one block of 111); this one merges blocks whose offsets sit
within `derived_offset_tol` of each other, so each real block is reported once.

**Drop them.** For a compound in a derived block, the pKi label goes and the pIC50 label
stays. This is not a judgement call about data quality — a value equal to pIC50 ÷ 140 adds
no information by construction, and keeping it would make a single pKi threshold wrong for
most of the column.

**Recover the censored measurements, on both endpoints.** A `>` bound at or below the
active cut proves the compound is not active and becomes a negative; a `<` bound at or
above it becomes a positive. Bounds that straddle the cut are uninformative and are
dropped with a count. This is the only step in the whole pipeline that adds genuinely new
information rather than rearranging what was there.

**Build the masked label matrix.** One row per compound, one column per endpoint, `NaN`
where that compound has no usable label for that endpoint. Downstream, the network trains
on both columns at once with a masked loss so partially-labelled compounds still
contribute, and the tree and kernel baselines are fitted once per endpoint on that
endpoint's labelled rows.

Each compound also carries a `source_tag` parsed from its BindingDB ligand name, which is
what makes single-filing over-representation visible and lets the leave-one-cluster-out
report in Part E.2 name the family it is holding out.

In [ ]:
# --- Multi-task label assembly ---------------------------------------------------------
import re

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

EPS = list(CONFIG.modelled_endpoints)
EP_COLS = dict(zip(CONFIG.endpoint_names, CONFIG.affinity_columns))
A_CUTS = dict(CONFIG.active_thresholds)
CLASS_NAMES = ["not-active", "active"]
POSITIVE_CLASS = 1
N_CLASSES = 2
PRIMARY = CONFIG.primary_endpoint        # the one modelled endpoint, used throughout

GENERIC_SOURCE_TAGS = {"CHEMBL", "other", "unknown"}
_PATENT_RE = re.compile(r"\b((?:US|WO|EP|CN|JP|KR)\s?\d{6,})", re.I)
_CHEMBL_RE = re.compile(r"\b(CHEMBL\d+)", re.I)


def source_tag(name) -> str:
    """Coarse provenance from the BindingDB ligand name. A patent number identifies one
    filing; CHEMBL and 'other' are catch-all buckets and must not be read as one source."""
    if not isinstance(name, str) or not name.strip():
        return "unknown"
    m = _PATENT_RE.search(name)
    if m:
        return m.group(1).replace(" ", "").upper()
    if _CHEMBL_RE.search(name):
        return "CHEMBL"
    return "other"


print(f"Modelling {len(EPS)} endpoints jointly: {EPS}")
for ep in EPS:
    print(f"  {ep}: active >= {A_CUTS[ep]:.2f}  "
          f"(<= {10 ** (9 - A_CUTS[ep]):.3g} nM)")


# =====================================================================================
# 1. Derived-value detection, with overlapping blocks merged
# =====================================================================================
def detect_derived_endpoint(df_wide, ep_a, ep_b, min_block=5, tol=0.02):
    """Blocks of compounds whose two endpoints differ by a FIXED amount.

    Returns (report, mask). Unlike F6's version, candidate offsets whose means fall within
    `tol` of an already-accepted block are merged into it rather than reported separately —
    F6 printed one block of 111 compounds as three rows of 106/103/102 because its +/-tol
    windows around the 2.14 / 2.15 / 2.16 rounding buckets overlapped.
    """
    empty = (pd.DataFrame(), pd.Series(False, index=df_wide.index))
    if ep_a not in df_wide.columns or ep_b not in df_wide.columns:
        return empty
    both = df_wide[df_wide[ep_a].notna() & df_wide[ep_b].notna()].copy()
    if len(both) < min_block:
        return empty
    both["_diff"] = both[ep_b] - both[ep_a]

    blocks, claimed = [], set()
    # work through candidate offsets most-populated first, skipping compounds already
    # claimed by a wider block, so each compound belongs to exactly one block
    for val, _n in both["_diff"].round(2).value_counts().items():
        sel = both[((both["_diff"] - val).abs() <= tol) & (~both.index.isin(claimed))]
        if len(sel) < min_block:
            continue
        if any(abs(b["offset"] - float(sel["_diff"].mean())) <= tol * 1.5 for b in blocks):
            continue                      # same block, different rounding bucket
        srcs = (sel["source_tag"].value_counts() if "source_tag" in sel
                else pd.Series(dtype=int))
        blocks.append({
            "offset": round(float(sel["_diff"].mean()), 4),
            "implied_ratio": round(float(10 ** sel["_diff"].mean()), 2),
            "n_compounds": len(sel),
            "offset_sd": round(float(sel["_diff"].std(ddof=0)), 5),
            "dominant_source": srcs.index[0] if len(srcs) else "?",
            "source_share": round(float(srcs.iloc[0] / len(sel)), 3) if len(srcs) else np.nan,
            "_idx": list(sel.index),
        })
        claimed.update(sel.index)

    mask = pd.Series(False, index=df_wide.index)
    mask.loc[sorted(claimed)] = True
    report = (pd.DataFrame([{k: v for k, v in b.items() if k != "_idx"} for b in blocks])
              .sort_values("n_compounds", ascending=False) if blocks else pd.DataFrame())
    return report, mask


train_df_mt = train_df.copy()
train_df_mt["source_tag"] = train_df_mt["ligand_name"].map(source_tag)
derived_report, derived_mask = detect_derived_endpoint(
    train_df_mt, CONFIG.endpoint_names[0], CONFIG.endpoint_names[1],
    min_block=CONFIG.derived_min_block_size, tol=CONFIG.derived_offset_tol,
)
derived_smiles = set(train_df_mt.loc[derived_mask, "smiles"])

print("\n" + "=" * 90)
print("DERIVED-VALUE DETECTOR — fixed-offset blocks between the two endpoints")
print("=" * 90)
if len(derived_report):
    display(derived_report)
    _n_pairs = int((train_df_mt[CONFIG.endpoint_names[0]].notna()
                    & train_df_mt[CONFIG.endpoint_names[1]].notna()).sum())
    print(f"{int(derived_mask.sum())} of {_n_pairs} dual-measured compounds "
          f"({100 * derived_mask.sum() / max(_n_pairs, 1):.0f}%) sit in a fixed-offset block.")
    print("A constant offset across many compounds from one source means the second endpoint")
    print("was computed from the first (Cheng-Prusoff), not measured. Those pKi labels carry")
    print("no information and would make any single pKi threshold wrong for most of the")
    if CONFIG.drop_derived_endpoint_values:
        print("column, so they are DROPPED below. Each affected compound keeps its pIC50.")
    else:
        print("column. drop_derived_endpoint_values is False, so they are KEPT — expect the")
        print("endpoint-agreement problem from Part A.2b to persist.")
else:
    print("No fixed-offset blocks found. The endpoints look independently measured.")


# =====================================================================================
# 2. Per-endpoint re-derivation from the raw pull, keeping censored rows
# =====================================================================================
def build_endpoint_labels(ep: str, a_cut: float):
    """Returns {smiles: record} for one endpoint, with numeric and censored labels."""
    col = EP_COLS[ep]
    sub = raw_bdb_df[raw_bdb_df["endpoint_name"] == ep].copy()
    raw = sub["affinity_raw"].astype(str)
    sub["qualifier"] = raw.str.extract(r"^\s*([<>]=?)")[0]
    sub["value_nm"] = pd.to_numeric(raw.str.replace(r"^\s*[<>]=?\s*", "", regex=True),
                                    errors="coerce")
    sub = sub.dropna(subset=["value_nm", "smiles"])
    sub = sub[(sub["value_nm"] >= CONFIG.min_affinity_nm)
              & (sub["value_nm"] <= CONFIG.max_affinity_nm)]
    sub["pAff"] = 9 - np.log10(sub["value_nm"])

    numeric = sub[sub["qualifier"].isna()]
    gt = sub[sub["qualifier"].notna() & sub["qualifier"].str.startswith(">")]
    lt = sub[sub["qualifier"].notna() & sub["qualifier"].str.startswith("<")]

    num_agg = (numeric.groupby("smiles", as_index=False)
               .agg(pAff=("pAff", "median"), ligand_name=("ligand_name", "first"),
                    n_meas=("pAff", "size")))
    # several '>' bounds must all hold -> the most informative is the smallest
    gt_agg = (gt.groupby("smiles", as_index=False)
              .agg(p_upper=("pAff", "min"), ligand_name=("ligand_name", "first")))
    lt_agg = (lt.groupby("smiles", as_index=False)
              .agg(p_lower=("pAff", "max"), ligand_name=("ligand_name", "first")))

    rec, stats = {}, {"numeric_rows": len(numeric), "numeric_cpds": len(num_agg),
                      "gt_rows": len(gt), "lt_rows": len(lt),
                      "recovered_inactive": 0, "recovered_active": 0,
                      "dropped_ambiguous": 0, "dropped_derived": 0}

    for r in num_agg.itertuples(index=False):
        # F9: the test is whether THIS endpoint is the derived one, not whether it is the
        # primary one. In F7/F8 those coincided (pIC50 was primary, pKi was derived); with
        # pKi as the modelled endpoint, keying off `primary_endpoint` would have kept every
        # Cheng-Prusoff value instead of dropping it.
        if (CONFIG.drop_derived_endpoint_values and ep == CONFIG.derived_endpoint
                and r.smiles in derived_smiles):
            stats["dropped_derived"] += 1
            continue
        rec[r.smiles] = {"ligand_name": r.ligand_name, "pAff": float(r.pAff),
                         "y": int(r.pAff >= a_cut), "label_source": "numeric",
                         "n_meas": int(r.n_meas)}

    if CONFIG.keep_censored_as_labels:
        for r in gt_agg.itertuples(index=False):
            if r.smiles in rec:
                continue
            if (CONFIG.drop_derived_endpoint_values and ep == CONFIG.derived_endpoint
                    and r.smiles in derived_smiles):
                stats["dropped_derived"] += 1
                continue
            if r.p_upper <= a_cut:                      # provably not active
                rec[r.smiles] = {"ligand_name": r.ligand_name, "pAff": np.nan, "y": 0,
                                 "label_source": "censored", "n_meas": 0}
                stats["recovered_inactive"] += 1
            else:
                stats["dropped_ambiguous"] += 1
        for r in lt_agg.itertuples(index=False):
            if r.smiles in rec:
                continue
            if r.p_lower >= a_cut:                      # provably active
                rec[r.smiles] = {"ligand_name": r.ligand_name, "pAff": np.nan, "y": 1,
                                 "label_source": "censored", "n_meas": 0}
                stats["recovered_active"] += 1
            else:
                stats["dropped_ambiguous"] += 1
    return rec, stats


endpoint_records, endpoint_stats = {}, {}
for ep in EPS:
    endpoint_records[ep], endpoint_stats[ep] = build_endpoint_labels(ep, A_CUTS[ep])

print("\n" + "=" * 90)
print("PER-ENDPOINT LABEL ASSEMBLY")
print("=" * 90)
display(pd.DataFrame(endpoint_stats).T)


# =====================================================================================
# 3. Masked multi-task table
# =====================================================================================
# Membership: every compound with ANY PSMA measurement, not only the modelled endpoint.
# A pIC50-only compound carries a NaN pKi label and so cannot train the model, but it is
# still a known PSMA ligand: it defines the applicability domain and shapes the Butina
# clustering that builds the folds. Excluding those would shrink the domain from ~730
# compounds to ~280 for no gain.
if CONFIG.keep_unlabelled_for_domain:
    all_smiles = sorted(set(train_df_mt["smiles"]).union(
        *[set(r) for r in endpoint_records.values()]))
else:
    all_smiles = sorted(set().union(*[set(r) for r in endpoint_records.values()]))

_names = dict(zip(train_df_mt["smiles"], train_df_mt["ligand_name"]))
rows = []
for smi in all_smiles:
    row = {"smiles": smi}
    name = _names.get(smi)
    for ep in EPS:
        r = endpoint_records[ep].get(smi)
        if r is None:
            row[ep] = np.nan
            row[f"y_{ep}"] = np.nan
            row[f"src_{ep}"] = None
        else:
            row[ep] = r["pAff"]
            row[f"y_{ep}"] = float(r["y"])
            row[f"src_{ep}"] = r["label_source"]
            name = name or r["ligand_name"]
    row["ligand_name"] = name
    rows.append(row)

mt_df = pd.DataFrame(rows)
mt_df["source_tag"] = mt_df["ligand_name"].map(source_tag)
mt_df["n_labels"] = mt_df[[f"y_{ep}" for ep in EPS]].notna().sum(axis=1)

print("\n" + "=" * 90)
print("MULTI-TASK LABEL COVERAGE")
print("=" * 90)
_cov = []
for ep in EPS:
    col = f"y_{ep}"
    _cov.append({"endpoint": ep, "labelled": int(mt_df[col].notna().sum()),
                 "active": int((mt_df[col] == 1).sum()),
                 "not_active": int((mt_df[col] == 0).sum()),
                 "base_rate": round(float(mt_df.loc[mt_df[col].notna(), col].mean()), 3)})
display(pd.DataFrame(_cov).set_index("endpoint"))

_n_lab = int(mt_df[f"y_{PRIMARY}"].notna().sum())
print(f"Compounds carrying a {PRIMARY} label (these train the model) : {_n_lab:,}")
print(f"Compounds with no {PRIMARY} label (domain + clustering only) : "
      f"{len(mt_df) - _n_lab:,}")
print(f"TOTAL compounds carried through                              : {len(mt_df):,}")
print("\nThe unlabelled compounds are known PSMA ligands measured on the other endpoint.")
print("They cannot train the model, but they define the applicability domain and shape the")
print("Butina clusters that build the leave-one-cluster-out folds, so the domain stays as")
print("broad as the knowledge actually is.")

print("\nSource composition:")
_src = mt_df["source_tag"].value_counts()
for tag, n in _src.head(8).items():
    print(f"  {tag:>16s}: {n:4d} ({100 * n / len(mt_df):5.1f}%)")
_specific = _src[~_src.index.isin(GENERIC_SOURCE_TAGS)]
if len(_specific) and _specific.iloc[0] / len(mt_df) > 0.15:
    print(f"\n!! '{_specific.index[0]}' — a single filing — supplies {_specific.iloc[0]} of "
          f"{len(mt_df)} compounds ({100 * _specific.iloc[0] / len(mt_df):.0f}%).")
    print("   Part E.2 holds it out as its own fold so its contribution is measurable.")

train_df = mt_df.copy()
print(f"\n`train_df` is now the multi-task classification set: {train_df.shape}")
train_df.head()

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully inside Colab.")
except ModuleNotFoundError:
    print("Running on a local environment; skipping Google Drive mount as it is not needed.")

## 3. Part A.2 — The screening library

F3–F8 screened a catalogue file. F9 **builds** its library instead, by running reaction
SMARTS over a set of PSMA cores and a set of caps, and then mixes a stratified sample of
the old catalogue back in as controls.

The loader below is kept because the mixing step in Part A.3c still needs it, but nothing
is read until then. Part A.3b enumerates; Part A.3c mixes; Part A.3d reports on whatever
came out and applies the optional drug-likeness prefilter.

In [ ]:
# --- Load the screening library, and report on it -----------------------------------------
import gzip, io

_SMILES_NAMES = ["smiles", "smi", "canonical_smiles", "structure", "mol", "smiles_string"]
_ID_NAMES = ["compound_id", "id", "idnumber", "catalog_id", "name", "title", "zinc_id",
             "chembl_id", "molecule_name", "corp_id"]


def _sniff_column(df: pd.DataFrame, candidates, fallback_parse=False):
    low = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        if cand in low:
            return low[cand]
    if fallback_parse:
        # no recognisable name — pick the column whose values actually parse as SMILES
        for c in df.columns:
            sample = df[c].dropna().astype(str).head(20)
            if len(sample) == 0:
                continue
            ok = sum(Chem.MolFromSmiles(s) is not None for s in sample)
            if ok >= max(3, int(0.7 * len(sample))):
                print(f"  SMILES column auto-detected by parsing: '{c}'")
                return c
    return None


def _open_maybe_gz(path: Path):
    if str(path).lower().endswith(".gz"):
        return io.TextIOWrapper(gzip.open(path, "rb"))
    return open(path, "r", errors="ignore")


def load_screening_library(cfg: Config, path_override: Optional[str] = None) -> pd.DataFrame:
    raw_path = path_override or cfg.screening_library_path
    path = Path(raw_path)
    # A leading "/" meant "the Drive root" in the Colab versions of this notebook; on a
    # real filesystem it means an absolute path. Only fall back to the relative reading
    # when the absolute path does not exist and the relative one does.
    if not path.exists() and raw_path.startswith(("/", "\\")):
        _alt = Path("." + raw_path)
        if _alt.exists():
            print(f"  {raw_path} not found; using {_alt} instead.")
            path = _alt

    if not path.exists():
        print("!" * 90)
        print(f"!! SCREENING LIBRARY NOT FOUND: {raw_path}")
        print("!! Writing an 8-compound DEMO file so the notebook still runs end to end.")
        print("!! Every screening result below is meaningless until this path is fixed.")
        print("!" * 90)
        path.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame({
            "compound_id": [f"demo_{i}" for i in range(8)],
            "SMILES": ["CCOc1ccc2nc(S(N)(=O)=O)sc2c1", "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
                       "O=C(O)c1ccccc1O", "CC(=O)Oc1ccccc1C(=O)O",
                       "Clc1ccc(C(c2ccccc2Cl)N2CCN(CCO)CC2)cc1",
                       "COc1cc2c(cc1OC)C(=O)C(CC1CCN(C)CC1)C2",
                       "CN1CCC[C@H]1c1cccnc1", "CC(C)NCC(O)COc1cccc2ccccc12"],
        }).to_csv(path, index=False)

    name = str(path).lower()
    print(f"Loading: {path}")

    if name.endswith((".sdf", ".sd", ".sdf.gz", ".sd.gz")):
        fmt = "SDF"
        if name.endswith(".gz"):
            suppl = Chem.ForwardSDMolSupplier(gzip.open(path, "rb"))
        else:
            suppl = Chem.SDMolSupplier(str(path))
        rows, n_fail = [], 0
        for i, mol in enumerate(tqdm(suppl, desc="  reading SDF", unit="mol")):
            if mol is None:
                n_fail += 1
                continue
            cid = mol.GetProp("_Name") if mol.HasProp("_Name") else f"{path.name}:{i + 1}"
            rows.append({"smiles": Chem.MolToSmiles(mol), "compound_id": cid or f"mol_{i}"})
        df = pd.DataFrame(rows)

    elif name.endswith((".smi", ".smi.gz", ".ism", ".ism.gz")):
        fmt = "SMILES list"
        rows, n_fail = [], 0
        with _open_maybe_gz(path) as fh:
            for i, line in enumerate(fh):
                parts = line.split()
                if not parts:
                    continue
                rows.append({"smiles": parts[0],
                             "compound_id": parts[1] if len(parts) > 1 else f"row_{i}"})
        df = pd.DataFrame(rows)

    else:
        fmt = "delimited text"
        sep = "\t" if name.endswith((".tsv", ".tsv.gz", ".tab")) else None
        df_raw = pd.read_csv(path, sep=sep, engine="python" if sep is None else "c")
        smi_col = (cfg.screening_smiles_col
                   if cfg.screening_smiles_col in df_raw.columns else None) \
            or _sniff_column(df_raw, _SMILES_NAMES, fallback_parse=True)
        if smi_col is None:
            raise ValueError(f"No SMILES column found in {path}. Columns: "
                             f"{list(df_raw.columns)[:12]}")
        id_col = (cfg.screening_id_col
                  if cfg.screening_id_col in df_raw.columns else None) \
            or _sniff_column(df_raw, _ID_NAMES)
        df = pd.DataFrame({"smiles": df_raw[smi_col].astype(str)})
        df["compound_id"] = (df_raw[id_col].astype(str) if id_col
                             else [f"row_{i}" for i in range(len(df_raw))])
        n_fail = 0
        print(f"  SMILES column: '{smi_col}'   id column: "
              f"'{id_col if id_col else '(generated)'}'")

    df = df.dropna(subset=["smiles"])
    df = df[df["smiles"].astype(str).str.strip() != ""].reset_index(drop=True)
    print(f"  format: {fmt} | rows loaded: {len(df):,}"
          + (f" | unparseable records skipped: {n_fail:,}" if n_fail else ""))
    return df


# F9: nothing is loaded here. Part A.3c calls this with an explicit path when it
# mixes the catalogue into the enumerated library.


# =====================================================================================
# Library health report — what IS this file?
# =====================================================================================
def library_health_report(df: pd.DataFrame, n_sample: int = 20000):
    from rdkit.Chem import Descriptors as _D
    rng = np.random.RandomState(CONFIG.random_state)
    take = df if len(df) <= n_sample else df.iloc[rng.choice(len(df), n_sample,
                                                             replace=False)]
    heavy, mw, bad = [], [], 0
    for smi in tqdm(take["smiles"].astype(str).tolist(), desc="  profiling", unit="mol",
                     leave=False):
        m = Chem.MolFromSmiles(smi)
        if m is None:
            bad += 1
            continue
        heavy.append(m.GetNumHeavyAtoms())
        mw.append(_D.MolWt(m))
    heavy, mw = np.asarray(heavy), np.asarray(mw)
    if len(heavy) == 0:
        print("  !! nothing in the sample parsed; the file is unusable as loaded.")
        return

    print("\n" + "=" * 86)
    print(f"LIBRARY HEALTH REPORT  (sample of {len(heavy):,} of {len(df):,})")
    print("=" * 86)
    print(f"  unparseable in sample : {bad:,}")
    print(f"  heavy atoms           : median {np.median(heavy):.0f}, "
          f"10th pct {np.percentile(heavy, 10):.0f}, 90th pct {np.percentile(heavy, 90):.0f}")
    print(f"  molecular weight      : median {np.median(mw):.0f}, "
          f"10th pct {np.percentile(mw, 10):.0f}, 90th pct {np.percentile(mw, 90):.0f}")
    frag = float((heavy < 12).mean())
    tiny = float((mw < 150).mean())
    big = float((mw > 700).mean())
    print(f"  fragment-like (<12 heavy atoms) : {100 * frag:5.1f}%")
    print(f"  very small (MW < 150)           : {100 * tiny:5.1f}%")
    print(f"  very large (MW > 700)           : {100 * big:5.1f}%")
    if frag > 0.10 or tiny > 0.05:
        print("\n  !! This does not look like a drug-like screening collection. A large")
        print("     fragment / small-molecule fraction is the signature of a building-block")
        print("     or reagent catalogue. Scoring it will produce a ranking, but the")
        print("     applicability domain in Part B.2 is what decides whether that ranking")
        print("     means anything — and in F6/F7 it put 36 of ~200,000 compounds in range.")
        print("     Replacing this file is worth more than any modelling change here.")
    else:
        print("\n  Size profile looks drug-like.")
    if CONFIG.library_prefilter:
        print(f"\n  library_prefilter is ON: dropping heavy atoms < "
              f"{CONFIG.prefilter_min_heavy_atoms} or MW outside "
              f"[{CONFIG.prefilter_mw_range[0]}, {CONFIG.prefilter_mw_range[1]}].")
    else:
        print("\n  library_prefilter is OFF — nothing has been removed. Set it True to")
        print("  apply the drug-likeness cut above.")


## 3b. Part A.2b — Reaction-based enumeration

F8 settled the question of why nothing was ever in range. The library health report showed
that the Enamine file **is** a normal drug-like collection — median MW 337, 24 heavy atoms,
1% fragment-like — so the earlier claim that it was a reagent catalogue was wrong; that was
an artifact of reading the first five rows of a file ordered small-to-large. The real reason
only 36 of 200,560 compounds were within structural reach is a **genuine chemistry
mismatch**: PSMA ligands are Glu-urea peptidomimetics, highly polar with multiple
carboxylates, and a diversity-oriented commercial deck is biased toward lead-like, less
acidic chemistry. The two populations barely overlap in fingerprint space.

No modelling change fixes that. The fix is to screen chemistry that exists in the same
space, which is what this cell builds: a **virtual library enumerated by applying reaction
SMARTS to building blocks**, around the pharmacophore the training data actually covers.

### How it works

Three ingredients, all in `CONFIG`-adjacent dictionaries you can edit:

1. **`REACTION_LIBRARY`** — named reaction SMARTS, each shipping with a **self-test**: a
   pair of reactant SMILES and the expected product. The cell runs every test before
   enumerating and **disables any reaction whose test fails**. This matters because a
   subtly wrong SMARTS does not raise — it silently yields nothing, or yields the wrong
   regiochemistry. The triazole (CuAAC) template is the one most likely to fail that check;
   if it does, it is dropped with a message rather than quietly poisoning the library.
2. **`CORES`** — scaffolds carrying the pharmacophore and one reactive handle. The defaults
   are the Glu-urea class: Lys-urea-Glu (the KuE core behind PSMA-617, PSMA-11 and DCFPyL),
   plus Orn-, Dap- and Glu-urea-Glu variants, each with a free side-chain amine as the
   attachment point.
3. **`CAPS`** — the combinatorial axis: carboxylic acids, sulfonyl chlorides, isocyanates,
   aldehydes and aryl halides that acylate, sulfonylate or alkylate that amine.

Every product carries its **provenance** — which core, which reaction, which cap — so a hit
is immediately a synthesis proposal rather than just a structure. That is the whole
advantage of enumeration over a catalogue.

### Two safeguards

**The property filter is derived from the training actives, not from Lipinski.** PSMA-617
has MW ≈ 1042 and eight carboxylic acids; a rule-of-five filter would delete every real
active in the class. The bounds come from the 5th–95th percentile of the known actives'
own MW, TPSA, HBD and rotatable-bond distributions.

**Known compounds are flagged, not removed.** Any product whose standardised InChIKey
matches a training compound is marked `is_known`. These are a **positive control**: if the
known actives do not land near the top of the final ranking, nothing further down it is
worth reading. Check that first in Part E.4.

### The circularity to keep in mind

Enumerating around the same pharmacophore the model was trained on makes the library
in-domain *by construction*. The applicability domain will pass most of it, and P(active)
will be high because Glu-urea compounds are what the actives look like. The informative
part is therefore **not** that the library scores well — it is which **caps** rank above
others, and that is a much weaker signal than the absolute numbers suggest. Part E.4 reports
the score spread across caps for exactly this reason: if every cap scores the same, the
model is recognising the core and telling you nothing about the variable part.

Set `CONFIG.screening_source = "file"` to screen a supplied file instead.

In [ ]:
# --- Reaction-based enumeration ------------------------------------------------------------
from rdkit.Chem import AllChem, Descriptors as _D
from rdkit.Chem import rdChemReactions

# =====================================================================================
# 1. Reaction library, each with a self-test
# =====================================================================================
# A wrong reaction SMARTS does not raise - it silently matches nothing, or builds the
# wrong regiochemistry. Every entry therefore ships a (reactants -> expected product)
# test, run below, and any reaction that fails is DISABLED rather than trusted.
REACTION_LIBRARY: Dict[str, dict] = {
    "amide_coupling": {
        "smarts": "[C:1](=[O:2])[OX2H1].[NX3;H2,H1;!$(N-[C,S]=[O,S,N]):3]"
                  ">>[C:1](=[O:2])[N:3]",
        "roles": ("acid", "amine"),
        "note": "carboxylic acid + amine -> amide (HATU/EDC in practice)",
        "test": (("CC(=O)O", "CN"), "CNC(C)=O"),
    },
    "sulfonamide": {
        "smarts": "[S:1](=[O:2])(=[O:3])[Cl].[NX3;H2,H1;!$(N-[C,S]=[O,S,N]):4]"
                  ">>[S:1](=[O:2])(=[O:3])[N:4]",
        "roles": ("sulfonyl_chloride", "amine"),
        "note": "sulfonyl chloride + amine -> sulfonamide",
        "test": (("CS(=O)(=O)Cl", "CN"), "CNS(C)(=O)=O"),
    },
    "urea_from_isocyanate": {
        "smarts": "[N:1]=[C:2]=[O:3].[NX3;H2,H1;!$(N-[C,S]=[O,S,N]):4]"
                  ">>[N:1][C:2](=[O:3])[N:4]",
        "roles": ("isocyanate", "amine"),
        "note": "isocyanate + amine -> urea",
        "test": (("CN=C=O", "CN"), "CNC(=O)NC"),
    },
    "thiourea_from_isothiocyanate": {
        "smarts": "[N:1]=[C:2]=[S:3].[NX3;H2,H1;!$(N-[C,S]=[O,S,N]):4]"
                  ">>[N:1][C:2](=[S:3])[N:4]",
        "roles": ("isothiocyanate", "amine"),
        "note": "isothiocyanate + amine -> thiourea",
        "test": (("CN=C=S", "CN"), "CNC(=S)NC"),
    },
    "reductive_amination": {
        "smarts": "[CX3H1:1]=[OX1].[NX3;H2,H1;!$(N-[C,S]=[O,S,N]):2]>>[CH2:1][N:2]",
        "roles": ("aldehyde", "amine"),
        "note": "aldehyde + amine -> secondary amine (NaBH(OAc)3)",
        "test": (("CC=O", "CN"), "CCNC"),
    },
    "snar_amination": {
        "smarts": "[c:1][F,Cl,Br,I].[NX3;H2,H1;!$(N-[C,S]=[O,S,N]):2]>>[c:1][N:2]",
        "roles": ("aryl_halide", "amine"),
        "note": "aryl halide + amine -> arylamine (SNAr or Buchwald)",
        "test": (("Clc1ccccc1", "CN"), "CNc1ccccc1"),
    },
    "click_triazole": {
        # CuAAC, 1,4-disubstituted. The product is written in Kekule form and left to
        # sanitisation to aromatise; this is the entry most likely to fail its test.
        "smarts": "[CH1:1]#[C:2].[N:3]=[N+:4]=[N-:5]>>[C:1]1=[C:2][N:5]=[N:4][N:3]1",
        "roles": ("alkyne", "azide"),
        "note": "terminal alkyne + azide -> 1,4-triazole (CuAAC)",
        "test": (("OCC#C", "C(c1ccccc1)N=[N+]=[N-]"), "OCc1cn(Cc2ccccc2)nn1"),
    },
}


def _canon(smi: str) -> Optional[str]:
    m = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(m) if m is not None else None


def _run_reaction(rxn, reactants: Sequence) -> List[str]:
    """All distinct products, as canonical SMILES. Returns [] on any failure."""
    out = []
    try:
        for prods in rxn.RunReactants(tuple(reactants)):
            for p in prods:
                try:
                    Chem.SanitizeMol(p)
                except Exception:
                    continue
                s = Chem.MolToSmiles(p)
                if s:
                    out.append(s)
    except Exception:
        return []
    return out


print("=" * 90)
print("REACTION SELF-TESTS")
print("=" * 90)
ACTIVE_REACTIONS: Dict[str, dict] = {}
for name, spec in REACTION_LIBRARY.items():
    status, detail = "FAIL", ""
    try:
        rxn = rdChemReactions.ReactionFromSmarts(spec["smarts"])
        rxn.Initialize()
        (ra, rb), expect = spec["test"]
        mols = [Chem.MolFromSmiles(ra), Chem.MolFromSmiles(rb)]
        if any(m is None for m in mols):
            detail = "test reactants did not parse"
        else:
            got = _run_reaction(rxn, mols)
            want = _canon(expect)
            if want is None:
                detail = "expected product did not parse"
            elif want in got:
                status, detail = "PASS", f"-> {want}"
                spec["_rxn"] = rxn
                ACTIVE_REACTIONS[name] = spec
            elif got:
                detail = f"expected {want}, got {sorted(set(got))[:3]}"
            else:
                detail = "produced nothing"
    except Exception as e:
        detail = f"{type(e).__name__}: {e}"
    print(f"  {status}  {name:<30s} {detail}")

print(f"\n{len(ACTIVE_REACTIONS)} of {len(REACTION_LIBRARY)} reactions validated and "
      f"available for enumeration.")
if len(ACTIVE_REACTIONS) < len(REACTION_LIBRARY):
    print("  Failed reactions are DISABLED — their SMARTS needs fixing before they can be")
    print("  trusted. A silently wrong template is worse than a missing one.")


# =====================================================================================
# 2. Cores — the pharmacophore, with one reactive handle
# =====================================================================================
# The Glu-urea class: a urea linking L-glutamate to a second amino acid, whose side-chain
# amine is the attachment point. This is the scaffold behind the clinically used PSMA
# ligands, and it is what the BindingDB training set is mostly made of.
CORES: List[dict] = [
    # An azido core, so the validated click reaction has something to work on. The lysine
    # side-chain amine of KuE is replaced by an azide; CuAAC onto it is the standard route
    # for attaching a chelator or a label in the PSMA tracer literature.
    {"id": "KuE_N3",  "role": "azide",
     "smiles": "[N-]=[N+]=NCCCC[C@@H](NC(=O)N[C@@H](CCC(=O)O)C(=O)O)C(=O)O",
     "note": "azido-Lys-urea-Glu; the CuAAC handle"},
    {"id": "KuE",     "role": "amine",
     "smiles": "NCCCC[C@@H](NC(=O)N[C@@H](CCC(=O)O)C(=O)O)C(=O)O",
     "note": "Lys-urea-Glu; the core of PSMA-617 / PSMA-11 / DCFPyL"},
    {"id": "OuE",     "role": "amine",
     "smiles": "NCCC[C@@H](NC(=O)N[C@@H](CCC(=O)O)C(=O)O)C(=O)O",
     "note": "Orn-urea-Glu; one methylene shorter"},
    {"id": "DapuE",   "role": "amine",
     "smiles": "NC[C@@H](NC(=O)N[C@@H](CCC(=O)O)C(=O)O)C(=O)O",
     "note": "Dap-urea-Glu; shortest side chain"},
    {"id": "KuE_Asp", "role": "amine",
     "smiles": "NCCCC[C@@H](NC(=O)N[C@@H](CC(=O)O)C(=O)O)C(=O)O",
     "note": "Lys-urea-Asp; shorter zinc-binding arm"},
    {"id": "Ku2Aad",  "role": "amine",
     "smiles": "NCCCC[C@@H](NC(=O)N[C@@H](CCCC(=O)O)C(=O)O)C(=O)O",
     "note": "Lys-urea-2-aminoadipate; longer zinc-binding arm"},
]

# =====================================================================================
# 3. Caps — the combinatorial axis
# =====================================================================================
CAPS: List[dict] = []


def _add_caps(role: str, items: Sequence[tuple]):
    for cid, smi in items:
        CAPS.append({"id": cid, "role": role, "smiles": smi})


_add_caps("acid", [
    ("benzoyl", "OC(=O)c1ccccc1"),
    ("4F-benzoyl", "OC(=O)c1ccc(F)cc1"),
    ("4I-benzoyl", "OC(=O)c1ccc(I)cc1"),
    ("4Br-benzoyl", "OC(=O)c1ccc(Br)cc1"),
    ("4NO2-benzoyl", "OC(=O)c1ccc([N+](=O)[O-])cc1"),
    ("4OMe-benzoyl", "OC(=O)c1ccc(OC)cc1"),
    ("4CF3-benzoyl", "OC(=O)c1ccc(C(F)(F)F)cc1"),
    ("3,5-diF-benzoyl", "OC(=O)c1cc(F)cc(F)c1"),
    ("2-naphthoyl", "OC(=O)c1ccc2ccccc2c1"),
    ("1-naphthoyl", "OC(=O)c1cccc2ccccc12"),
    ("nicotinoyl", "OC(=O)c1cccnc1"),
    ("isonicotinoyl", "OC(=O)c1ccncc1"),
    ("picolinoyl", "OC(=O)c1ccccn1"),
    ("quinoline-2-carbonyl", "OC(=O)c1ccc2ccccc2n1"),
    ("quinoline-6-carbonyl", "OC(=O)c1ccc2ncccc2c1"),
    ("indole-2-carbonyl", "OC(=O)c1cc2ccccc2[nH]1"),
    ("indole-5-carbonyl", "OC(=O)c1ccc2[nH]ccc2c1"),
    ("benzothiazole-2-carbonyl", "OC(=O)c1nc2ccccc2s1"),
    ("benzofuran-2-carbonyl", "OC(=O)c1cc2ccccc2o1"),
    ("thiophene-2-carbonyl", "OC(=O)c1cccs1"),
    ("furan-2-carbonyl", "OC(=O)c1ccco1"),
    ("pyrazole-4-carbonyl", "OC(=O)c1cn[nH]c1"),
    ("imidazole-4-carbonyl", "OC(=O)c1cnc[nH]1"),
    ("biphenyl-4-carbonyl", "OC(=O)c1ccc(-c2ccccc2)cc1"),
    ("phenylacetyl", "OC(=O)Cc1ccccc1"),
    ("3-phenylpropanoyl", "OC(=O)CCc1ccccc1"),
    ("trans-cinnamoyl", "OC(=O)/C=C/c1ccccc1"),
    ("4-aminobenzoyl", "OC(=O)c1ccc(N)cc1"),
    ("4-carboxybenzoyl", "OC(=O)c1ccc(C(=O)O)cc1"),
    ("glutaryl", "OC(=O)CCCC(=O)O"),
    ("suberoyl", "OC(=O)CCCCCCC(=O)O"),
    ("2-(naphthalen-2-yl)acetyl", "OC(=O)Cc1ccc2ccccc2c1"),
    ("Ac", "CC(=O)O"),
    ("tranexamoyl", "OC(=O)C1CCC(CN)CC1"),
])

_add_caps("sulfonyl_chloride", [
    ("tosyl", "Cc1ccc(S(=O)(=O)Cl)cc1"),
    ("dansyl-like", "CN(C)c1cccc2c(S(=O)(=O)Cl)cccc12"),
    ("benzenesulfonyl", "ClS(=O)(=O)c1ccccc1"),
    ("4F-benzenesulfonyl", "ClS(=O)(=O)c1ccc(F)cc1"),
    ("2-naphthalenesulfonyl", "ClS(=O)(=O)c1ccc2ccccc2c1"),
    ("quinoline-8-sulfonyl", "ClS(=O)(=O)c1cccc2cccnc12"),
])

_add_caps("isocyanate", [
    ("phenyl-NCO", "O=C=Nc1ccccc1"),
    ("4F-phenyl-NCO", "O=C=Nc1ccc(F)cc1"),
    ("naphthalen-1-yl-NCO", "O=C=Nc1cccc2ccccc12"),
    ("benzyl-NCO", "O=C=NCc1ccccc1"),
])

_add_caps("isothiocyanate", [
    ("phenyl-NCS", "S=C=Nc1ccccc1"),
    ("4F-phenyl-NCS", "S=C=Nc1ccc(F)cc1"),
    ("fluorescein-like-NCS", "S=C=Nc1ccc2ccccc2c1"),
])

_add_caps("aldehyde", [
    ("benzaldehyde", "O=Cc1ccccc1"),
    ("2-naphthaldehyde", "O=Cc1ccc2ccccc2c1"),
    ("pyridine-3-carbaldehyde", "O=Cc1cccnc1"),
    ("4F-benzaldehyde", "O=Cc1ccc(F)cc1"),
    ("quinoline-2-carbaldehyde", "O=Cc1ccc2ccccc2n1"),
])

_add_caps("alkyne", [
    # Terminal alkynes for the click reaction onto KuE_N3.
    ("propargyl-benzamide", "C#CCNC(=O)c1ccccc1"),
    ("propargyl-naphthamide", "C#CCNC(=O)c1ccc2ccccc2c1"),
    ("propargyl-4I-benzamide", "C#CCNC(=O)c1ccc(I)cc1"),
    ("propargyl-glycine", "C#CCNC(=O)CN"),
    ("propargyl-DOTA-like", "C#CCNC(=O)CN(CC(=O)O)CC(=O)O"),
    ("hexynoic-acid", "C#CCCCC(=O)O"),
])

_add_caps("aryl_halide", [
    ("2-fluoropyridine", "Fc1ccccn1"),
    ("4-fluoronitrobenzene", "Fc1ccc([N+](=O)[O-])cc1"),
    ("2-chloroquinoline", "Clc1ccc2ccccc2n1"),
    ("4-fluorobenzonitrile", "Fc1ccc(C#N)cc1"),
])

print("\n" + "=" * 90)
print("BUILDING BLOCKS")
print("=" * 90)
_cap_roles = pd.Series([c["role"] for c in CAPS]).value_counts()
print(f"  cores: {len(CORES)}  ({', '.join(c['id'] for c in CORES)})")
for role, n in _cap_roles.items():
    usable = any(spec["roles"][0] == role for spec in ACTIVE_REACTIONS.values())
    print(f"  caps/{role:<22s} {n:3d}" + ("" if usable else "   [no validated reaction]"))


# =====================================================================================
# 4. Enumerate
# =====================================================================================
def enumerate_library(cores, caps, reactions, max_products=None, seed=0) -> pd.DataFrame:
    """Cross product of (core x reaction x cap), with provenance.

    Reactant order follows each reaction's `roles`, so the cap matches template 0 and the
    core matches template 1 (or vice versa). RunReactants may fire at several sites on one
    core — those extra regiochemistries are kept and deduplicated later, since they are
    genuine alternative products rather than errors.
    """
    core_mols = {}
    for c in cores:
        m = Chem.MolFromSmiles(c["smiles"])
        if m is None:
            print(f"  !! core '{c['id']}' did not parse; skipped")
            continue
        core_mols[c["id"]] = (c, m)
    cap_mols = {}
    for c in caps:
        m = Chem.MolFromSmiles(c["smiles"])
        if m is None:
            print(f"  !! cap '{c['id']}' did not parse; skipped")
            continue
        cap_mols[c["id"]] = (c, m)

    rows, n_attempt, n_empty = [], 0, 0
    combos = [(rn, rs, ci, cj) for rn, rs in reactions.items()
              for ci in core_mols for cj in cap_mols
              if cap_mols[cj][0]["role"] == rs["roles"][0]
              and core_mols[ci][0]["role"] == rs["roles"][1]]
    for rname, rspec, cid, capid in tqdm(combos, desc="Enumerating", unit="combo"):
        core, cmol = core_mols[cid]
        cap, capmol = cap_mols[capid]
        n_attempt += 1
        prods = _run_reaction(rspec["_rxn"], (capmol, cmol))
        if not prods:
            n_empty += 1
            continue
        for k, smi in enumerate(dict.fromkeys(prods)):   # preserve order, drop dupes
            rows.append({"smiles": smi, "core_id": cid, "reaction": rname,
                         "cap_id": capid, "cap_role": cap["role"], "variant": k})

    df = pd.DataFrame(rows)
    print(f"\n  {n_attempt} core x reaction x cap combinations attempted")
    print(f"  {n_empty} produced nothing ({100 * n_empty / max(n_attempt, 1):.0f}%)")
    # A validated reaction with no eligible building block contributes nothing and would
    # otherwise vanish silently: the count above only covers combinations that were tried.
    _used = {rn for rn, _, _, _ in combos}
    _unused = {rn: reactions[rn]["roles"] for rn in reactions if rn not in _used}
    if _unused:
        print(f"  {len(_unused)} validated reaction(s) contributed NOTHING, because no "
              f"building block carries the role they need:")
        _have = sorted({c["role"] for c in caps}) + sorted({c["role"] for c in cores})
        for rn, roles in _unused.items():
            print(f"    {rn:<30s} needs cap role '{roles[0]}' + core role '{roles[1]}'")
        print(f"    roles actually present: {', '.join(sorted(set(_have)))}")
        print("    Add caps (or cores) with those roles, or drop the reaction.")
    print(f"  {len(df):,} raw products before deduplication")
    if max_products and len(df) > max_products:
        df = df.sample(max_products, random_state=seed).reset_index(drop=True)
        print(f"  sampled down to {max_products:,} (CONFIG.enum_max_products)")
    return df


if "enumeration" in CONFIG.screening_source:
    if not ACTIVE_REACTIONS:
        raise RuntimeError("No reaction passed its self-test; cannot enumerate. Fix the "
                           "SMARTS in REACTION_LIBRARY or set screening_source='file'.")
    print("\n" + "=" * 90)
    print("ENUMERATION")
    print("=" * 90)
    enum_df = enumerate_library(CORES, CAPS, ACTIVE_REACTIONS,
                                max_products=CONFIG.enum_max_products,
                                seed=CONFIG.random_state)

    # --- deduplicate on standardised InChIKey, and flag anything already known ---------
    from rdkit.Chem.MolStandardize import rdMolStandardize as _std
    _norm, _lfc, _unch = (_std.Normalizer(), _std.LargestFragmentChooser(),
                          _std.Uncharger())

    def _std_key(smi):
        m = Chem.MolFromSmiles(smi)
        if m is None:
            return None, None
        try:
            m = _unch.uncharge(_lfc.choose(_norm.normalize(m)))
            Chem.SanitizeMol(m)
        except Exception:
            return None, None
        try:
            from rdkit.Chem.inchi import MolToInchiKey
            k = MolToInchiKey(m)
        except Exception:
            k = Chem.MolToSmiles(m)
        return k, m

    keys, mols, props = [], [], []
    for smi in tqdm(enum_df["smiles"].tolist(), desc="Standardising products", unit="mol"):
        k, m = _std_key(smi)
        keys.append(k)
        mols.append(m)
        props.append((np.nan, np.nan, np.nan, np.nan) if m is None else
                     (_D.MolWt(m), _D.TPSA(m), _D.NumHDonors(m),
                      _D.NumRotatableBonds(m)))
    enum_df["_key"] = keys
    enum_df[["MW", "TPSA", "HBD", "RotB"]] = pd.DataFrame(props, index=enum_df.index)
    enum_df = enum_df[enum_df["_key"].notna()].reset_index(drop=True)
    n0 = len(enum_df)
    enum_df = enum_df.drop_duplicates("_key").reset_index(drop=True)
    print(f"  deduplicated on standardised InChIKey: {n0:,} -> {len(enum_df):,}")

    # --- property window from the KNOWN ACTIVES, not from Lipinski --------------------
    # PSMA-617 is MW ~1042 with eight carboxylic acids. A rule-of-five filter would
    # delete the entire active class, so the window comes from the actives themselves.
    _act_mask = (train_df[f"y_{PRIMARY}"] == 1).values
    _act_props = []
    for smi in train_df.loc[_act_mask, "smiles"].tolist():
        m = Chem.MolFromSmiles(smi)
        if m is not None:
            _act_props.append((_D.MolWt(m), _D.TPSA(m), _D.NumHDonors(m),
                               _D.NumRotatableBonds(m)))
    if len(_act_props) >= 10:
        A = pd.DataFrame(_act_props, columns=["MW", "TPSA", "HBD", "RotB"])
        lo = A.quantile(CONFIG.enum_property_quantile)
        hi = A.quantile(1 - CONFIG.enum_property_quantile)
        print(f"\n  Property window from {len(A)} known actives "
              f"({100 * CONFIG.enum_property_quantile:.0f}th-"
              f"{100 * (1 - CONFIG.enum_property_quantile):.0f}th percentile):")
        for c in A.columns:
            print(f"    {c:<5s} {lo[c]:8.1f} .. {hi[c]:8.1f}   "
                  f"(actives median {A[c].median():.1f})")
        keep = np.ones(len(enum_df), dtype=bool)
        for c in A.columns:
            keep &= enum_df[c].between(lo[c], hi[c]).values
        _frac = float(keep.mean()) if len(keep) else 0.0
        print(f"  property filter: {len(enum_df):,} -> {int(keep.sum()):,} products "
              f"({100 * _frac:.0f}% kept)")
        # Safety valve. An enumerated product is a core PLUS a cap, so it is systematically
        # heavier and more polar than the actives it was built from, and a window drawn from
        # those actives can exclude nearly all of it. Removing them anyway would leave an
        # empty or tiny library and make every number downstream vacuous, so below the floor
        # the window is reported and FLAGGED rather than applied.
        if not CONFIG.enum_apply_property_filter:
            enum_df["in_property_window"] = keep
            print("  (enum_apply_property_filter is False — flagged, not removed)")
        elif _frac < CONFIG.enum_property_min_keep_frac:
            enum_df["in_property_window"] = keep
            print(f"\n  !! The window would keep only {100 * _frac:.0f}% of products, below")
            print(f"     the {100 * CONFIG.enum_property_min_keep_frac:.0f}% floor, so it has"
                  f" NOT been applied — the products are")
            print("     flagged in `in_property_window` instead. This is the expected")
            print("     direction: a product is a core plus a cap, so it is heavier and more")
            print("     polar than the actives the window came from. Either widen the window")
            print("     (raise `enum_property_quantile`), or treat the flag as a soft")
            print("     preference when reading the shortlist. An empty library would have")
            print("     made every result below meaningless.")
        else:
            enum_df = enum_df[keep].reset_index(drop=True)
    else:
        print("\n  Too few known actives to derive a property window; no filter applied.")

    # --- positive control: which products are already known compounds? ----------------
    _train_keys = set()
    for smi in train_df["smiles"].tolist():
        k, _ = _std_key(smi)
        if k:
            _train_keys.add(k)
    enum_df["is_known"] = enum_df["_key"].isin(_train_keys)
    n_known = int(enum_df["is_known"].sum())
    print(f"\n  POSITIVE CONTROL: {n_known} enumerated products match a training compound")
    if n_known == 0:
        print("    !! None. The enumeration does not reproduce any compound the model was")
        print("       trained on, so there is no internal check that the ranking puts")
        print("       known actives at the top. Consider adding cores/caps that recreate")
        print("       a few known ligands — without that control the shortlist is")
        print("       unfalsifiable.")
    else:
        print("    Part E.4 checks where these rank. If known compounds do not appear near")
        print("    the top, nothing further down the ranking is worth reading.")

    enum_df["compound_id"] = [f"{r.core_id}|{r.reaction}|{r.cap_id}"
                              + (f"|v{r.variant}" if r.variant else "")
                              for r in enum_df.itertuples(index=False)]
    screen_df = enum_df[["smiles", "compound_id", "core_id", "reaction", "cap_id",
                         "cap_role", "is_known", "MW", "TPSA", "HBD", "RotB"]].copy()

    print(f"\n  Final enumerated library: {len(screen_df):,} compounds")
    print(f"    by reaction : "
          + ", ".join(f"{k} {v}" for k, v in screen_df['reaction'].value_counts().items()))
    print(f"    by core     : "
          + ", ".join(f"{k} {v}" for k, v in screen_df['core_id'].value_counts().items()))
    print("\n  Every row carries its core, reaction and cap, so a hit is a synthesis")
    print("  proposal rather than only a structure.")
    display(screen_df.head(10))

else:
    print(f"\nCONFIG.screening_source = '{CONFIG.screening_source}' — loading a file "
          f"instead of enumerating.")
    screen_df = load_screening_library(CONFIG)
    library_health_report(screen_df)
    for c in ("core_id", "reaction", "cap_id", "cap_role", "is_known"):
        if c not in screen_df.columns:
            screen_df[c] = None

## 3c. Part A.2c — Mixing in the out-of-domain catalogue

An enumerated library built from the training pharmacophore is in-domain *by construction*,
and ranking twenty thousand near-identical Glu-ureas against each other says very little.
Mixing in structures from the Enamine file fixes three things at once:

- **A negative control.** Catalogue compounds are, by construction, not PSMA-like. The
  score distribution by source is then a real test rather than a foregone conclusion.
- **A scaffold-hop channel.** A catalogue compound that scores highly is far more
  interesting than another Glu-urea that does — it is a novel chemotype the model likes.
- **A non-trivial ranking task.** This is the part that needs care: the enumerated products
  sit around MW 600–900 and the catalogue median is 337, so a plain random sample would let
  the model separate the two populations on *size alone* and a good-looking ranking would
  prove nothing.

The sample is therefore stratified three ways, each labelled in a `source` column so every
downstream diagnostic can break results down by provenance:

| stratum | what it is | what it tests |
|---|---|---|
| `library_near` | the `mix_n_near` catalogue compounds with the **smallest Tanimoto distance to the training set** | the genuine scaffold-hop candidates — the near-misses |
| `library_matched` | **property-matched decoys**, drawn to follow the enumerated library's own MW distribution | whether the ranking survives when size and polarity are controlled for |
| `library_random` | a plain random draw | unbiased background |

The matched stratum doubles as a measurement: how many catalogue compounds *can* be
property-matched to the enumerated library quantifies the chemistry gap F8 found, in
compounds rather than in fingerprint distance. If that number is small, the gap is real and
structural rather than an artifact of the similarity metric.

In [ ]:
# --- Assemble the final screening library ---------------------------------------------------
if CONFIG.screening_source in ("enumeration+library", "library+enumeration"):
    print("=" * 90)
    print("MIXING IN THE OUT-OF-DOMAIN CATALOGUE")
    print("=" * 90)
    screen_df["source"] = "enumerated"

    pool = load_screening_library(CONFIG, path_override=CONFIG.mix_library_path)
    rng = np.random.RandomState(CONFIG.random_state)

    # Pre-sample before the distance computation: the full file against 700+ training
    # compounds is tens of millions of similarity evaluations and we only need a pool.
    if CONFIG.mix_pool_sample and len(pool) > CONFIG.mix_pool_sample:
        pool = pool.iloc[rng.choice(len(pool), CONFIG.mix_pool_sample, replace=False)] \
            .reset_index(drop=True)
        print(f"  pre-sampled the catalogue to {len(pool):,} compounds for distance "
              f"computation")

    # Fingerprints computed inline (the featurizer class is defined later in the notebook).
    def _fp(smi):
        m = Chem.MolFromSmiles(smi)
        return None if m is None else AllChem.GetMorganFingerprintAsBitVect(
            m, CONFIG.morgan_radius, nBits=CONFIG.morgan_n_bits)

    tr_fps = [f for f in (_fp(s) for s in train_df["smiles"]) if f is not None]
    pool_fps, pool_ok = [], []
    for smi in tqdm(pool["smiles"].tolist(), desc="  catalogue fingerprints", unit="mol"):
        f = _fp(smi)
        pool_ok.append(f is not None)
        if f is not None:
            pool_fps.append(f)
    pool = pool[np.asarray(pool_ok)].reset_index(drop=True)

    dis = np.empty(len(pool_fps))
    for i, f in enumerate(tqdm(pool_fps, desc="  distance to training set", unit="mol")):
        dis[i] = 1.0 - max(DataStructs.BulkTanimotoSimilarity(f, tr_fps))
    pool["tanimoto_to_train"] = dis

    # properties, for the matched stratum
    _pp = []
    for smi in tqdm(pool["smiles"].tolist(), desc="  catalogue properties", unit="mol"):
        m = Chem.MolFromSmiles(smi)
        _pp.append((np.nan, np.nan, np.nan, np.nan) if m is None else
                   (_D.MolWt(m), _D.TPSA(m), _D.NumHDonors(m), _D.NumRotatableBonds(m)))
    pool[["MW", "TPSA", "HBD", "RotB"]] = pd.DataFrame(_pp, index=pool.index)

    print(f"\n  catalogue pool: {len(pool):,} compounds")
    print(f"    Tanimoto distance to training set: min {dis.min():.3f}, "
          f"median {np.median(dis):.3f}, max {dis.max():.3f}")
    print(f"    MW: median {pool['MW'].median():.0f}   "
          f"(enumerated library median {screen_df['MW'].median():.0f})")

    taken, strata = set(), []

    # --- stratum 1: nearest to the training set -- the scaffold-hop candidates ---------
    near = pool.sort_values("tanimoto_to_train").head(CONFIG.mix_n_near)
    taken.update(near.index)
    strata.append(("library_near", near))
    print(f"\n  library_near    : {len(near):,} compounds, distance "
          f"{near['tanimoto_to_train'].min():.3f}-{near['tanimoto_to_train'].max():.3f}")

    # --- stratum 2: property-matched decoys -------------------------------------------
    # Follow the enumerated library's own MW distribution, so the two populations cannot
    # be separated on size. Shortfalls per bin are reported: they ARE the chemistry gap.
    rest = pool.drop(index=list(taken))
    target = screen_df["MW"].dropna()
    matched_idx, shortfall, want_total = [], 0, 0
    if len(target) and CONFIG.mix_n_matched > 0:
        edges = np.unique(np.percentile(target, np.linspace(0, 100, 11)))
        if len(edges) < 2:
            # A near-degenerate MW distribution (very similar caps) collapses the
            # percentile edges to one value. Without this the bin loop never runs and the
            # stratum silently reports that nothing was wanted.
            _c = float(target.median())
            edges = np.array([_c - CONFIG.mix_mw_tolerance, _c + CONFIG.mix_mw_tolerance])
            print(f"    (enumerated MW is near-constant at {_c:.0f}; matching in a single "
                  f"bin of +/-{CONFIG.mix_mw_tolerance:.0f})")
        for lo, hi in zip(edges[:-1], edges[1:]):
            frac = float(((target >= lo) & (target < hi)).mean())
            want = int(round(frac * CONFIG.mix_n_matched))
            want_total += want
            cand = rest[(rest["MW"] >= lo - CONFIG.mix_mw_tolerance)
                        & (rest["MW"] < hi + CONFIG.mix_mw_tolerance)]
            cand = cand.drop(index=[i for i in cand.index if i in taken], errors="ignore")
            if len(cand) == 0:
                shortfall += want
                continue
            take = min(want, len(cand))
            pick = cand.iloc[rng.choice(len(cand), take, replace=False)]
            matched_idx.extend(pick.index.tolist())
            taken.update(pick.index)
            shortfall += want - take
    matched = pool.loc[matched_idx]
    strata.append(("library_matched", matched))
    print(f"  library_matched : {len(matched):,} compounds "
          f"(wanted {want_total:,}, short by {shortfall:,})")
    if want_total and shortfall / max(want_total, 1) > 0.25:
        print(f"    !! {100 * shortfall / want_total:.0f}% of the requested matched decoys")
        print("       do not exist in the catalogue. That IS the chemistry gap F8 found,")
        print("       measured in compounds: the catalogue simply does not contain")
        print("       molecules of the size and polarity the PSMA actives occupy.")

    # --- stratum 3: unbiased random background ----------------------------------------
    rest = pool.drop(index=list(taken))
    n_rand = min(CONFIG.mix_n_random, len(rest))
    rand = rest.iloc[rng.choice(len(rest), n_rand, replace=False)] if n_rand else rest.head(0)
    strata.append(("library_random", rand))
    print(f"  library_random  : {len(rand):,} compounds")

    add = []
    for label, part in strata:
        if len(part) == 0:
            continue
        p = part[["smiles", "compound_id", "MW", "TPSA", "HBD", "RotB"]].copy()
        p["source"] = label
        for c in ("core_id", "reaction", "cap_id", "cap_role"):
            p[c] = None
        p["is_known"] = False
        add.append(p)

    screen_df = pd.concat([screen_df] + add, ignore_index=True)
    # a catalogue compound could coincide with an enumerated product; keep the first
    n0 = len(screen_df)
    screen_df = screen_df.drop_duplicates("smiles").reset_index(drop=True)
    if len(screen_df) != n0:
        print(f"\n  dropped {n0 - len(screen_df)} duplicate SMILES across strata")

    print(f"\n  FINAL MIXED LIBRARY: {len(screen_df):,} compounds")
    display(screen_df["source"].value_counts().rename("compounds").to_frame())
    print("  Part E.4 breaks every result down by `source`. The two things to read there:")
    print("    1. do the known compounds (is_known) rank at the top? If not, stop.")
    print("    2. does any `library_*` compound outrank the enumerated products? That")
    print("       would be a genuine scaffold hop, and the most interesting output here.")

## 3d. Part A.2d — What came out


In [ ]:
# --- Report on the assembled library, and prefilter if asked --------------------------------
print("=" * 90)
print("FINAL SCREENING LIBRARY")
print("=" * 90)
library_health_report(screen_df)

if CONFIG.library_prefilter:
    keep = []
    lo, hi = CONFIG.prefilter_mw_range
    for smi in tqdm(screen_df["smiles"].astype(str).tolist(), desc="Prefiltering",
                    unit="mol"):
        m = Chem.MolFromSmiles(smi)
        keep.append(bool(m is not None
                         and m.GetNumHeavyAtoms() >= CONFIG.prefilter_min_heavy_atoms
                         and lo <= _D.MolWt(m) <= hi))
    n0 = len(screen_df)
    screen_df = screen_df[np.asarray(keep)].reset_index(drop=True)
    print(f"Prefilter: {n0:,} -> {len(screen_df):,} compounds "
          f"({n0 - len(screen_df):,} removed)")
    if "source" in screen_df.columns:
        print("  surviving compounds by source:")
        for k, v in screen_df["source"].value_counts().items():
            print(f"    {k:18s} {v:,}")

# Columns the rest of the notebook expects, whichever branch produced the library.
for _c in ("source", "core_id", "reaction", "cap_id", "cap_role"):
    if _c not in screen_df.columns:
        screen_df[_c] = None
if "is_known" not in screen_df.columns:
    screen_df["is_known"] = False

screen_df.head()

## 4. Part B.1 — Featurization

Same RDKit standardization + Morgan-fingerprint + descriptor wrapper as the original
pipeline, now applied to **three** sets: the broad ChEMBL pretraining set, the PSMA
BindingDB fine-tuning set, and the screening library. Using the identical featurizer
everywhere is what makes the pretrained trunk's weights meaningful once loaded into
the fine-tuning model — the input feature space has to match.


In [ ]:
class RDKitFeaturizer:
    '''Reusable RDKit wrapper: standardization + fingerprints + descriptors.

    Uses RDKit's full 2D descriptor catalog (~200 physicochemical/topological/graph
    descriptors from `Descriptors._descList`, in the spirit of AutoQSAR's ~500-descriptor
    pool[1]) rather than a small hand-picked subset. The redundancy this introduces is
    dealt with explicitly in Part B.1b below, following AutoQSAR's own near-constant +
    correlation-clustering filter, rather than left for the model to sort out.

    [1] Dixon et al., "AutoQSAR: an automated machine learning tool for best-practice
        QSAR modeling," Future Med. Chem. 2016.
    '''

    DESCRIPTOR_NAMES = [name for name, _ in Descriptors._descList]

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self._normalizer = rdMolStandardize.Normalizer()
        self._lfc = rdMolStandardize.LargestFragmentChooser()
        self._uncharger = rdMolStandardize.Uncharger()
        self._desc_fns = dict(Descriptors._descList)

    def standardize(self, smiles: str) -> Optional[Chem.Mol]:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        try:
            mol = self._normalizer.normalize(mol)
            mol = self._lfc.choose(mol)
            mol = self._uncharger.uncharge(mol)
            Chem.SanitizeMol(mol)
        except Exception:
            return None
        return mol

    def morgan_fp(self, mol: Chem.Mol):
        return AllChem.GetMorganFingerprintAsBitVect(mol, self.cfg.morgan_radius, nBits=self.cfg.morgan_n_bits)

    def morgan_fp_with_bit_info(self, mol: Chem.Mol):
        '''Same fingerprint as `morgan_fp`, but also returns RDKit's bitInfo dict
        (bit -> [(center_atom_idx, radius), ...]), needed to map bits back to atoms
        for the interpretability analysis in Part E.6.'''
        bit_info: Dict[int, list] = {}
        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol, self.cfg.morgan_radius, nBits=self.cfg.morgan_n_bits, bitInfo=bit_info
        )
        return fp, bit_info

    def descriptors(self, mol: Chem.Mol) -> np.ndarray:
        vals = []
        for name in self.DESCRIPTOR_NAMES:
            fn = self._desc_fns.get(name)
            try:
                vals.append(fn(mol) if fn else np.nan)
            except Exception:
                vals.append(np.nan)
        return np.array(vals, dtype=float)

    def featurize_dataframe(self, df: pd.DataFrame, smiles_col: str = "smiles") -> dict:
        mols, fps, desc_rows, keep_idx = [], [], [], []
        for i, smi in enumerate(tqdm(df[smiles_col].tolist(), desc="Featurizing",
                                     unit="mol", leave=True)):
            mol = self.standardize(smi)
            if mol is None:
                continue
            mols.append(mol)
            fps.append(self.morgan_fp(mol))
            desc_rows.append(self.descriptors(mol))
            keep_idx.append(i)
        desc_arr = np.vstack(desc_rows) if desc_rows else np.empty((0, len(self.DESCRIPTOR_NAMES)))
        clean_df = df.iloc[keep_idx].reset_index(drop=True)
        n_dropped = len(df) - len(clean_df)
        if n_dropped:
            print(f"  dropped {n_dropped} molecules that failed standardization/parsing.")
        return {"df": clean_df, "mols": mols, "fps": fps, "descriptors": desc_arr,
                "descriptor_names": self.DESCRIPTOR_NAMES}


featurizer = RDKitFeaturizer(CONFIG)
# F9: no ChEMBL pretraining set. The network is gone, so there is nothing to pretrain, and
# the descriptor curation and feature scaling below are fitted on the PSMA set instead —
# 700+ compounds is ample for a near-constant and correlation filter, and it removes a
# live web-API dependency from the pipeline.
train_feat = featurizer.featurize_dataframe(train_df, smiles_col="smiles")
screen_feat = featurizer.featurize_dataframe(screen_df, smiles_col="smiles")


# --- F7: deduplicate on the InChIKey of the STANDARDISED structure --------------------
# Two long-standing problems, one fix. (1) F6's shortlist of 25 rows held 5 compounds
# twice or three times - LigPrep protonation and tautomer variants of one molecule, which
# wastes shortlist slots and inflates the apparent number of hits. (2) Training compounds
# were deduplicated on the RAW SMILES string before RDKit standardisation, so salt and
# tautomer variants survived as separate rows and could straddle a fold boundary.
#
# The featurizer already standardises (Normalizer -> LargestFragmentChooser -> Uncharger),
# so variants of one compound collapse to the same standardised mol and therefore the same
# InChIKey. Deduplicating there fixes both.
try:
    from rdkit.Chem.inchi import MolToInchiKey
    _HAVE_INCHI = True
except Exception:
    _HAVE_INCHI = False
    print("!! InChI support not built into this RDKit; falling back to canonical SMILES "
          "of the standardised structure (weaker, but still post-standardisation).")


def _structure_key(mol) -> str:
    if _HAVE_INCHI:
        try:
            k = MolToInchiKey(mol)
            if k:
                return k
        except Exception:
            pass
    return Chem.MolToSmiles(mol)


def dedupe_featurized(feat: dict, label: str, keep: str = "first") -> dict:
    """Collapse duplicate structures, keeping one representative per InChIKey."""
    if not CONFIG.dedupe_on_inchikey or len(feat["mols"]) == 0:
        return feat
    keys = [_structure_key(m) for m in tqdm(feat["mols"], desc=f"{label}: InChIKeys",
                                             unit="mol", leave=False)]
    df = feat["df"].reset_index(drop=True).copy()
    df["_struct_key"] = keys
    keep_idx = (df.reset_index().groupby("_struct_key")["index"]
                .first().sort_values().values)
    n_before, n_after = len(df), len(keep_idx)
    if n_after == n_before:
        print(f"  {label}: no duplicate structures ({n_before:,} compounds)")
    else:
        print(f"  {label}: {n_before:,} -> {n_after:,} compounds "
              f"({n_before - n_after:,} duplicate structures merged)")
    return {
        "df": df.iloc[keep_idx].drop(columns=["_struct_key"]).reset_index(drop=True),
        "mols": [feat["mols"][i] for i in keep_idx],
        "fps": [feat["fps"][i] for i in keep_idx],
        "descriptors": feat["descriptors"][keep_idx],
        "descriptor_names": feat["descriptor_names"],
        "struct_keys": [keys[i] for i in keep_idx],
    }


print("\nDeduplicating on standardised-structure InChIKey:")
train_feat = dedupe_featurized(train_feat, "PSMA training set")
screen_feat = dedupe_featurized(screen_feat, "screening library")

train_df = train_feat["df"]
screen_df = screen_feat["df"]

train_fps, train_desc_raw = train_feat["fps"], train_feat["descriptors"]
screen_fps, screen_desc_raw = screen_feat["fps"], screen_feat["descriptors"]
descriptor_names_raw = train_feat["descriptor_names"]

print(f"BindingDB PSMA set after featurization: {len(train_df):,}")
print(f"Screening set after featurization: {len(screen_df):,}")
print(f"Raw descriptor pool: {len(descriptor_names_raw)} descriptors (before redundancy filtering below).")


## 4b. Part B.1b — Descriptor curation (near-constant + correlation-cluster filtering)

Directly follows the AutoQSAR paper's two-stage descriptor curation, applied to the
broad ChEMBL pretraining set (the largest, most chemically diverse sample, so its
correlation structure generalizes best):

1. **Drop near-constant / near-zero-variance descriptors** — a descriptor where ≥90%
   of compounds share one value carries almost no information (e.g., "number of
   tetrazoles" is 0 for nearly everything).
2. **Cluster the survivors by |Pearson correlation|** (hierarchical, complete-linkage,
   cut at r = 0.8) and keep only the descriptor closest to each cluster's centroid.
   This directly addresses the collinearity in a ~200-descriptor pool without
   depending on any one model family to "figure it out" from redundant inputs.

The same descriptor columns (fit once, on the pretraining set) are then applied to
the fine-tuning and screening descriptor matrices, so everything downstream — the
shared `feature_scaler`, the NN trunk, the GBM/KPLS baselines — sees a consistent,
de-redundified feature space.


In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform


class AutoQSARDescriptorSelector:
    '''Near-constant filter + correlation-cluster redundancy reduction, as described in
    the AutoQSAR paper (Dixon et al., Future Med. Chem. 2016).'''

    def __init__(self, near_constant_frac: float = 0.90, corr_threshold: float = 0.8):
        self.near_constant_frac = near_constant_frac
        self.corr_threshold = corr_threshold
        self.selected_idx: Optional[np.ndarray] = None
        self.selected_names: Optional[List[str]] = None

    def fit(self, X: np.ndarray, names: Sequence[str]) -> "AutoQSARDescriptorSelector":
        X = np.nan_to_num(X)
        n = len(X)

        # Stage 1: near-constant filter.
        keep1 = []
        for j in range(X.shape[1]):
            col = X[:, j]
            _, counts = np.unique(col, return_counts=True)
            most_common_frac = counts.max() / n
            if most_common_frac < self.near_constant_frac and np.var(col) > 1e-8:
                keep1.append(j)
        if not keep1:
            keep1 = list(range(X.shape[1]))
        X1 = X[:, keep1]
        names1 = [names[j] for j in keep1]
        print(f"  Stage 1 (near-constant filter): {X.shape[1]} -> {X1.shape[1]} descriptors.")

        # Stage 2: correlation-cluster redundancy reduction.
        if X1.shape[1] > 1:
            corr = np.corrcoef(X1, rowvar=False)
            corr = np.nan_to_num(corr, nan=0.0)
            np.fill_diagonal(corr, 1.0)
            dist = 1.0 - np.abs(corr)
            dist = (dist + dist.T) / 2.0        # enforce exact symmetry (float noise)
            np.fill_diagonal(dist, 0.0)
            condensed = squareform(dist, checks=False)
            Z = linkage(condensed, method="complete")
            cluster_labels = fcluster(Z, t=1.0 - self.corr_threshold, criterion="distance")

            keep2 = []
            for cl in np.unique(cluster_labels):
                members = np.where(cluster_labels == cl)[0]
                if len(members) == 1:
                    keep2.append(members[0])
                    continue
                sub_corr = np.abs(corr[np.ix_(members, members)])
                centroid_scores = sub_corr.mean(axis=1)   # closest-to-centroid = highest mean similarity
                keep2.append(members[np.argmax(centroid_scores)])
            keep2 = sorted(keep2)
        else:
            keep2 = list(range(X1.shape[1]))

        self.selected_idx = np.array([keep1[j] for j in keep2])
        self.selected_names = [names[j] for j in self.selected_idx]
        print(f"  Stage 2 (correlation clustering, r<={self.corr_threshold}): "
              f"{X1.shape[1]} -> {len(self.selected_idx)} descriptors.")
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        return np.nan_to_num(X)[:, self.selected_idx]


print("Fitting AutoQSAR-style descriptor selector on the PSMA set...")
descriptor_selector = AutoQSARDescriptorSelector(corr_threshold=0.8).fit(train_desc_raw, descriptor_names_raw)

train_desc = descriptor_selector.transform(train_desc_raw)
screen_desc = descriptor_selector.transform(screen_desc_raw)
descriptor_names = descriptor_selector.selected_names

print(f"Final curated descriptor set ({len(descriptor_names)}): {descriptor_names[:12]}"
      f"{' ...' if len(descriptor_names) > 12 else ''}")


## 5. Part B.2 — Applicability domain

Unchanged logic from the original pipeline: four complementary distance-from-training
metrics (Tanimoto dissimilarity on fingerprints, Manhattan distance in descriptor
space, Manhattan distance in a PCA "latent" proxy, and k-NN unfamiliarity), each
calibrated against a percentile of the **training set's own internal nearest-neighbor
distances**. Calibrated on the **PSMA-specific BindingDB fine-tuning set** — that's
the domain the final predictions need to be trustworthy within, not the broad ChEMBL
pretraining set.


In [ ]:
def fps_to_array(fps) -> np.ndarray:
    arr = np.zeros((len(fps), fps[0].GetNumBits()), dtype=np.uint8)
    for i, fp in enumerate(fps):
        DataStructs.ConvertToNumpyArray(fp, arr[i])
    return arr


def bulk_min_tanimoto_dissimilarity(query_fps, ref_fps,
                                     desc: str = "Tanimoto to nearest reference") -> np.ndarray:
    out = np.empty(len(query_fps))
    for i, qfp in enumerate(tqdm(query_fps, desc=desc, unit="mol", leave=False)):
        sims = DataStructs.BulkTanimotoSimilarity(qfp, ref_fps)
        out[i] = 1.0 - max(sims)
    return out


def train_internal_tanimoto_dissimilarity(train_fps) -> np.ndarray:
    out = np.empty(len(train_fps))
    for i, qfp in enumerate(tqdm(train_fps, desc="AD calibration: internal NN Tanimoto",
                                  unit="mol", leave=False)):
        sims = DataStructs.BulkTanimotoSimilarity(qfp, train_fps)
        sims[i] = -1
        out[i] = 1.0 - max(sims)
    return out


class ApplicabilityDomain:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.scaler: Optional[StandardScaler] = None
        self.pca: Optional[PCA] = None
        self.thresholds: dict = {}

    def fit(self, train_fps, train_desc: np.ndarray):
        self.scaler = StandardScaler().fit(np.nan_to_num(train_desc))
        train_desc_scaled = self.scaler.transform(np.nan_to_num(train_desc))

        n_comp = min(self.cfg.latent_pca_components, train_desc_scaled.shape[1], train_desc_scaled.shape[0] - 1)
        self.pca = PCA(n_components=max(n_comp, 1), random_state=self.cfg.random_state).fit(train_desc_scaled)
        train_latent = self.pca.transform(train_desc_scaled)

        tanimoto_internal = train_internal_tanimoto_dissimilarity(train_fps)
        self.thresholds["tanimoto"] = np.percentile(tanimoto_internal, self.cfg.ad_percentile_threshold)

        nn_desc_l1 = NearestNeighbors(n_neighbors=2, p=1).fit(train_desc_scaled)
        d_l1, _ = nn_desc_l1.kneighbors(train_desc_scaled)
        self.thresholds["descriptor_manhattan"] = np.percentile(d_l1[:, 1], self.cfg.ad_percentile_threshold)

        nn_latent_l1 = NearestNeighbors(n_neighbors=2, p=1).fit(train_latent)
        d_lat, _ = nn_latent_l1.kneighbors(train_latent)
        self.thresholds["latent_manhattan"] = np.percentile(d_lat[:, 1], self.cfg.ad_percentile_threshold)

        k = min(self.cfg.unfamiliarity_k + 1, len(train_desc_scaled))
        nn_unfam = NearestNeighbors(n_neighbors=k).fit(train_desc_scaled)
        d_unfam, _ = nn_unfam.kneighbors(train_desc_scaled)
        unfam_internal = d_unfam[:, 1:].mean(axis=1)
        self.thresholds["unfamiliarity"] = np.percentile(unfam_internal, self.cfg.ad_percentile_threshold)

        self._train_desc_scaled = train_desc_scaled
        self._train_latent = train_latent
        self._nn_desc_l1 = NearestNeighbors(n_neighbors=1, p=1).fit(train_desc_scaled)
        self._nn_latent_l1 = NearestNeighbors(n_neighbors=1, p=1).fit(train_latent)
        self._nn_unfam = NearestNeighbors(n_neighbors=min(self.cfg.unfamiliarity_k, len(train_desc_scaled))).fit(
            train_desc_scaled
        )
        return self

    def score(self, screen_fps, screen_desc: np.ndarray, train_fps) -> pd.DataFrame:
        screen_desc_scaled = self.scaler.transform(np.nan_to_num(screen_desc))
        screen_latent = self.pca.transform(screen_desc_scaled)

        tanimoto_diss = bulk_min_tanimoto_dissimilarity(
            screen_fps, train_fps,
            desc="AD: Tanimoto to nearest training compound")
        with tqdm(total=3, desc="AD: descriptor / latent / k-NN distances",
                  leave=False) as _ad_bar:
            desc_manhattan, _ = self._nn_desc_l1.kneighbors(screen_desc_scaled)
            _ad_bar.update(1)
            latent_manhattan, _ = self._nn_latent_l1.kneighbors(screen_latent)
            _ad_bar.update(1)
            unfam_d, _ = self._nn_unfam.kneighbors(screen_desc_scaled)
            _ad_bar.update(1)
        unfamiliarity = unfam_d.mean(axis=1)

        out = pd.DataFrame({
            "tanimoto_dissimilarity": tanimoto_diss,
            "descriptor_manhattan": desc_manhattan[:, 0],
            "latent_manhattan": latent_manhattan[:, 0],
            "unfamiliarity": unfamiliarity,
        })
        out["tanimoto_OOD"] = out["tanimoto_dissimilarity"] > self.thresholds["tanimoto"]
        out["descriptor_OOD"] = out["descriptor_manhattan"] > self.thresholds["descriptor_manhattan"]
        out["latent_OOD"] = out["latent_manhattan"] > self.thresholds["latent_manhattan"]
        out["unfamiliarity_OOD"] = out["unfamiliarity"] > self.thresholds["unfamiliarity"]
        out["n_flags_OOD"] = out[["tanimoto_OOD", "descriptor_OOD", "latent_OOD", "unfamiliarity_OOD"]].sum(axis=1)

        # F6: Tanimoto is a VETO, not one vote in four.
        #
        # F5 used `n_flags_OOD >= 3`, which let a compound pass while failing the
        # structural-similarity test - and for a congeneric target that is the flag with
        # teeth. The other three metrics compare bulk physicochemical properties, so a
        # molecule of roughly the right size and lipophilicity passes them while being
        # structurally unrelated to anything in training. Every one of F5's 15 top hits
        # was flagged tanimoto_OOD (dissimilarity 0.73-0.83 against a 0.457 threshold)
        # and every one of them was reported as in-domain. Meanwhile the Butina-grouped
        # evaluation showed that this is exactly the regime where the model fails.
        out["consensus_OOD"] = out["n_flags_OOD"] >= self.cfg.ad_consensus_min_flags
        if getattr(self.cfg, "ad_tanimoto_veto", False):
            out["consensus_OOD"] = out["consensus_OOD"] | out["tanimoto_OOD"]
        return out


# Calibrated on EVERY compound carried through, including those with no pKi label. The
# applicability domain is a statement about chemical space coverage, not about which label
# happens to be present: a compound with a pIC50 measurement is still evidence that its
# region of chemistry is familiar PSMA ground.
_n_pki = int(train_df[f"y_{PRIMARY}"].notna().sum())
print(f"Calibrating the applicability domain on all {len(train_fps):,} compounds "
      f"(of which {_n_pki:,} carry a {PRIMARY} label).")
ad = ApplicabilityDomain(CONFIG).fit(train_fps, train_desc)
ad_scores = ad.score(screen_fps, screen_desc, train_fps)
print("AD thresholds (95th percentile of the PSMA training set's own internal NN distances):")
for k, v in ad.thresholds.items():
    print(f"  {k}: {v:.4f}")

_n = len(ad_scores)
print(f"\nScreening library: {_n:,} compounds")
for flag in ("tanimoto_OOD", "descriptor_OOD", "latent_OOD", "unfamiliarity_OOD"):
    print(f"  {flag:>18s} fires on {ad_scores[flag].sum():>7,} "
          f"({100 * ad_scores[flag].mean():5.1f}%)")
_veto = getattr(CONFIG, "ad_tanimoto_veto", False)
_in_dom = int((~ad_scores["consensus_OOD"]).sum())
print(f"\n  Tanimoto veto: {'ON' if _veto else 'OFF'}"
      f"   (consensus_min_flags = {CONFIG.ad_consensus_min_flags})")
print(f"  -> in-domain: {_in_dom:,} / {_n:,} ({100 * _in_dom / _n:.2f}%)")
_would_be = int((ad_scores["n_flags_OOD"] < CONFIG.ad_consensus_min_flags).sum())
if _veto:
    print(f"     without the veto it would have been {_would_be:,} "
          f"({100 * _would_be / _n:.2f}%) - the veto removes "
          f"{_would_be - _in_dom:,} compounds that passed on bulk-property grounds")
    print("     while being structurally unlike anything in the training set.")
ad_scores.describe()


## 5b. Part B.2b — Which feature set actually works best here?

Both papers found that the best choice of input representation (fingerprints only,
descriptors only, or both together) is target-dependent rather than universal — the
KPLS paper found fingerprints alone beat descriptors alone on 9/10 targets, but the
*combination* won outright for one of them. Rather than assume `CONFIG.feature_set`
is right, a fast default-hyperparameter LightGBM is cross-validated on the PSMA
training data under all three options (scored on `pIC50`, the primary endpoint), and
whichever wins becomes the feature set used for every model built from here on.


In [ ]:
# --- Which representation works best? --------------------------------------------------
# Scored on the primary endpoint's labelled rows by ROC-AUC under stratified CV.
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier


def build_design_matrix(fps, desc: np.ndarray, scaler: StandardScaler, feature_set: str) -> np.ndarray:
    desc_scaled = scaler.transform(np.nan_to_num(desc))
    if feature_set == "descriptors_only":
        return desc_scaled
    fp_arr = fps_to_array(fps)
    if feature_set == "fp_only":
        return fp_arr
    if feature_set == "fp_plus_descriptors":
        return np.hstack([fp_arr, desc_scaled])
    raise ValueError(f"Unknown feature_set '{feature_set}'.")


_feature_set_options = ["fp_only", "descriptors_only", "fp_plus_descriptors"]
_quick_scaler = StandardScaler().fit(np.nan_to_num(train_desc))
_probe_col = f"y_{CONFIG.primary_endpoint}"
_probe_mask = train_df[_probe_col].notna().values
_y_probe = train_df.loc[_probe_mask, _probe_col].values.astype(int)
_skf = StratifiedKFold(n_splits=min(CONFIG.cv_folds, int(np.bincount(_y_probe).min())),
                       shuffle=True, random_state=CONFIG.random_state)

_fs_scores = {}
for fs in tqdm(_feature_set_options, desc="Feature-set bake-off", unit="option"):
    X_probe = build_design_matrix(train_fps, train_desc, _quick_scaler, fs)[_probe_mask]
    aucs = []
    for tr_i, te_i in tqdm(_skf.split(X_probe, _y_probe), total=_skf.get_n_splits(),
                            desc=f"  {fs}: CV folds", leave=False):
        m = LGBMClassifier(random_state=CONFIG.random_state, verbose=-1, n_estimators=150,
                            n_jobs=-1,
                            class_weight="balanced" if CONFIG.class_weight_balanced else None)
        m.fit(X_probe[tr_i], _y_probe[tr_i])
        aucs.append(roc_auc_score(_y_probe[te_i], m.predict_proba(X_probe[te_i])[:, 1]))
    _fs_scores[fs] = float(np.mean(aucs))
    print(f"  {fs:>22s}: mean CV ROC-AUC = {_fs_scores[fs]:.3f}")

_best_feature_set = max(_fs_scores, key=_fs_scores.get)
print(f"\nBest feature set on {CONFIG.primary_endpoint}: '{_best_feature_set}' "
      f"(CONFIG default was '{CONFIG.feature_set}')")
_fp, _best = _fs_scores["fp_only"], max(_fs_scores.values())
if _best - _fp < 0.005:
    _how = "is the best" if _best == _fp else f"is within {_best - _fp:.3f} of the best"
    print(f"  (fp_only {_how} — the 91 curated descriptors are adding essentially nothing "
          f"for this task, as in F6 and F7.)")
CONFIG.feature_set = _best_feature_set

## 6. Part B.3 — Design matrices

One shared feature scaler (`feature_scaler`, fit on the **broad ChEMBL pretraining
set**, since that's the larger and more chemically diverse sample) is reused to build
the model-input matrix for all three sets. Using the same scaler everywhere is what
lets the pretrained trunk's weights be meaningfully reused during fine-tuning — the
fine-tuning and screening inputs live in the exact same normalized feature space the
trunk was pretrained on. This is separate from `ad.scaler` above, which is
intentionally calibrated only on the PSMA set for domain-membership scoring.
`CONFIG.feature_set` here is whatever Part B.2b just selected.


In [ ]:
# --- Design matrices, masked label matrix, reference split -----------------------------
feature_scaler = StandardScaler().fit(np.nan_to_num(train_desc))

X_train_full = build_design_matrix(train_fps, train_desc, feature_scaler, CONFIG.feature_set)
X_screen = build_design_matrix(screen_fps, screen_desc, feature_scaler, CONFIG.feature_set)

# (n_compounds, n_endpoints), NaN where that compound has no label for that endpoint
Y_train_full = train_df[[f"y_{ep}" for ep in EPS]].values.astype(np.float32)
LABEL_MASK = np.isfinite(Y_train_full)


def source_weight_vector(df: pd.DataFrame, enabled: bool) -> np.ndarray:
    """Inverse-sqrt-frequency weight per source tag, mean-normalised. Exploratory; OFF by
    default, since discounting an over-represented filing is a judgement rather than a
    fact. Compare the pooled numbers both ways before adopting it."""
    if not enabled:
        return np.ones(len(df), dtype=np.float32)
    counts = df["source_tag"].map(df["source_tag"].value_counts()).values.astype(np.float64)
    w = 1.0 / np.sqrt(counts)
    return (w / w.mean()).astype(np.float32)


w_train_full = source_weight_vector(train_df, CONFIG.downweight_dominant_source)

# Reference split: stratified on the primary endpoint where it exists, random elsewhere,
# so the comparison with F6's random-split row stays meaningful.
_strat = np.where(np.isfinite(Y_train_full[:, 0]), Y_train_full[:, 0], 2).astype(int)
idx_tr, idx_te = train_test_split(
    np.arange(len(train_df)), test_size=CONFIG.test_size,
    random_state=CONFIG.random_state, stratify=_strat,
)
X_tr, X_te = X_train_full[idx_tr], X_train_full[idx_te]
Y_tr, Y_te = Y_train_full[idx_tr], Y_train_full[idx_te]
w_tr, w_te = w_train_full[idx_tr], w_train_full[idx_te]

print(f"Feature set: '{CONFIG.feature_set}' -> {X_train_full.shape[1]} input dimensions")
print(f"PSMA single-task ({PRIMARY}) — train: {X_tr.shape}, "
      f"held-out reference test: {X_te.shape}\n")
for t, ep in enumerate(EPS):
    print(f"  {ep}: train {int(np.isfinite(Y_tr[:, t]).sum()):4d} labelled "
          f"({int(np.nansum(Y_tr[:, t])):3d} active) | "
          f"test {int(np.isfinite(Y_te[:, t]).sum()):4d} labelled "
          f"({int(np.nansum(Y_te[:, t])):3d} active)")
print(f"\nTotal label instances: {int(LABEL_MASK.sum()):,} across "
      f"{len(train_df):,} compounds")
if CONFIG.downweight_dominant_source:
    print(f"Source down-weighting ON: weights {w_train_full.min():.3f}-{w_train_full.max():.3f}")

## 7. Part C.1 — The model zoo

Seven classical classifiers behind one `fit` / `predict_proba` interface, with a registry
(`MODEL_ZOO`) carrying each model's search grid and whether it needs fingerprints rather
than the design matrix. Adding another model means adding one registry entry; the tuning,
fold, screening and saving cells all iterate over it.

| model | why it is here |
|---|---|
| **LightGBM** | gradient-boosted trees; F7's strongest single model on pKi (0.796) |
| **KPLS-DA** | Nystroem kernel map → PLS → logistic, after An, Sherman & Dixon (2013) |
| **RandomForest** | the QSAR workhorse; bagged trees, a different bias to boosting |
| **SVM (Tanimoto)** | support vectors on a precomputed Tanimoto kernel — the chemists' own similarity measure, and the strongest classical method for fingerprints at small *n* |
| **kNN (Tanimoto)** | the similarity principle unadorned. A **floor**, not a contender: anything the committee gains over it is what the models add beyond lookup |
| **LogReg (L2)** | the **linear floor**. If the ensembles cannot beat it, the complexity is not paying |
| **BernoulliNB** | Laplacian-corrected naive Bayes on fingerprint bits, the old Pipeline Pilot standard; near-free |

Two practical notes. The Tanimoto models hold the training fingerprints and compute a Gram
at prediction time, so `predict_proba_any` chunks them when scoring the library. And every
model returns a full `(n, 2)` probability matrix even on a fold that never saw one of the
classes, so the equal-weight blend is always well defined — leave-one-cluster-out produces
badly unbalanced folds and used to break members that assumed otherwise.

In [ ]:
# --- The model zoo: seven classifiers behind one interface -------------------------------
from sklearn.cross_decomposition import PLSRegression
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB


def _labelled(Y, t):
    m = np.isfinite(Y[:, t])
    return m, Y[m, t].astype(int)


def _cw():
    return "balanced" if CONFIG.class_weight_balanced else None


def _align_proba(p: np.ndarray, classes) -> np.ndarray:
    """Pad to the full class set, so a fold that never saw a class still returns (n, 2)."""
    if p.shape[1] == N_CLASSES and list(classes) == list(range(N_CLASSES)):
        return p
    out = np.zeros((p.shape[0], N_CLASSES))
    for j, c in enumerate(classes):
        out[:, int(c)] = p[:, j]
    return out


# =====================================================================================
# Tanimoto kernel — the chemists' similarity measure, used directly
# =====================================================================================
def tanimoto_gram(query_fps, ref_fps, desc=None, leave=False) -> np.ndarray:
    """(n_query, n_ref) Tanimoto similarities, via RDKit's C++ bulk routine.

    float32 to halve the memory: the library Gram would be 200k x 731 otherwise. Callers
    scoring the whole library go through `predict_proba_any`, which chunks."""
    K = np.empty((len(query_fps), len(ref_fps)), dtype=np.float32)
    it = query_fps if desc is None else tqdm(query_fps, desc=desc, unit="mol", leave=leave)
    for i, q in enumerate(it):
        K[i] = DataStructs.BulkTanimotoSimilarity(q, ref_fps)
    return K


# =====================================================================================
# fit / predict pairs
# =====================================================================================
def fit_lgbm(X, y, params, fps=None, sample_weight=None, seed=42):
    m = LGBMClassifier(random_state=seed, verbose=-1, n_jobs=-1, class_weight=_cw(),
                       **params)
    m.fit(X, y, sample_weight=sample_weight)
    return m


def proba_lgbm(m, X, fps=None):
    return _align_proba(m.predict_proba(X), m.classes_)


def fit_kpls(X, y, params, fps=None, sample_weight=None, seed=42):
    """Nystroem kernel map -> PLS latent space -> logistic layer (PLS-DA).
    Landmarks clamped to the sample count: LOCO produces small training halves."""
    n_land = int(min(200, max(2, len(X) - 1)))
    nys = Nystroem(kernel="rbf", gamma=params["gamma"], n_components=n_land,
                   random_state=seed)
    Z = nys.fit_transform(X)
    n_comp = max(1, min(params["n_components"], min(50, Z.shape[1] - 1)))
    Y1h = np.zeros((len(y), N_CLASSES)); Y1h[np.arange(len(y)), y] = 1.0
    pls = PLSRegression(n_components=n_comp).fit(Z, Y1h)
    lg = LogisticRegression(max_iter=2000, class_weight=_cw())
    lg.fit(pls.transform(Z), y, sample_weight=sample_weight)
    return {"nystroem": nys, "pls": pls, "logit": lg, "classes_": lg.classes_}


def proba_kpls(m, X, fps=None):
    T = m["pls"].transform(m["nystroem"].transform(X))
    return _align_proba(m["logit"].predict_proba(T), m["classes_"])


def fit_rf(X, y, params, fps=None, sample_weight=None, seed=42):
    m = RandomForestClassifier(
        n_estimators=params.get("n_estimators", 300),
        max_features=params.get("max_features", "sqrt"),
        min_samples_leaf=params.get("min_samples_leaf", 1),
        class_weight="balanced_subsample" if CONFIG.class_weight_balanced else None,
        n_jobs=-1, random_state=seed)
    m.fit(X, y, sample_weight=sample_weight)
    return m


def proba_rf(m, X, fps=None):
    return _align_proba(m.predict_proba(X), m.classes_)


def fit_svm_tanimoto(X, y, params, fps=None, sample_weight=None, seed=42):
    """SVC on a PRECOMPUTED Tanimoto kernel. `probability=True` runs sklearn's internal
    Platt cross-validation, so this returns probabilities rather than margins."""
    K = tanimoto_gram(fps, fps)
    m = SVC(C=params.get("C", 1.0), kernel="precomputed", probability=True,
            class_weight=_cw(), random_state=seed)
    m.fit(K, y, sample_weight=sample_weight)
    return {"svc": m, "train_fps": list(fps)}


def proba_svm_tanimoto(m, X, fps=None):
    K = tanimoto_gram(fps, m["train_fps"])
    return _align_proba(m["svc"].predict_proba(K), m["svc"].classes_)


def fit_knn_tanimoto(X, y, params, fps=None, sample_weight=None, seed=42):
    """Lazy: stores the training fingerprints and labels."""
    return {"fps": list(fps), "y": np.asarray(y, dtype=int), "k": int(params.get("k", 5))}


def proba_knn_tanimoto(m, X, fps=None):
    """Similarity-weighted vote over the k nearest training compounds by Tanimoto."""
    S = tanimoto_gram(fps, m["fps"])
    k = int(min(m["k"], S.shape[1]))
    idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
    rows = np.arange(S.shape[0])[:, None]
    w = S[rows, idx].astype(np.float64) + 1e-9
    hit = (m["y"][idx] == POSITIVE_CLASS).astype(np.float64)
    p1 = (w * hit).sum(axis=1) / w.sum(axis=1)
    return np.column_stack([1.0 - p1, p1])


def fit_logreg(X, y, params, fps=None, sample_weight=None, seed=42):
    m = LogisticRegression(C=params.get("C", 1.0), penalty="l2", max_iter=3000,
                           class_weight=_cw(), random_state=seed)
    m.fit(X, y, sample_weight=sample_weight)
    return m


def proba_logreg(m, X, fps=None):
    return _align_proba(m.predict_proba(X), m.classes_)


def fit_nb(X, y, params, fps=None, sample_weight=None, seed=42):
    """Bernoulli naive Bayes on binarised features. With feature_set='fp_only' those are
    exactly the fingerprint bits, which is the classical formulation; with descriptors in
    the matrix the binarisation of those columns is crude but harmless."""
    m = BernoulliNB(alpha=params.get("alpha", 1.0), binarize=0.5)
    m.fit(X, y, sample_weight=sample_weight)
    return m


def proba_nb(m, X, fps=None):
    return _align_proba(m.predict_proba(X), m.classes_)


# =====================================================================================
# Registry
# =====================================================================================
MODEL_ZOO: Dict[str, dict] = {
    "LightGBM": {
        "fit": fit_lgbm, "proba": proba_lgbm, "needs_fps": False,
        "grid": [{"n_estimators": n, "learning_rate": lr, "num_leaves": nl, "max_depth": md}
                 for n in (100, 200, 400) for lr in (0.01, 0.05, 0.1)
                 for nl in (15, 31) for md in (-1, 6)],
    },
    "KPLS-DA": {
        "fit": fit_kpls, "proba": proba_kpls, "needs_fps": False,
        "grid": [{"n_components": nc, "gamma": g}
                 for nc in (2, 4, 6, 8, 12) for g in (None, 0.01, 0.05, 0.1)],
    },
    "RandomForest": {
        "fit": fit_rf, "proba": proba_rf, "needs_fps": False,
        "grid": [{"n_estimators": 300, "max_features": mf, "min_samples_leaf": ml}
                 for mf in ("sqrt", 0.05) for ml in (1, 3)],
    },
    "SVM (Tanimoto)": {
        "fit": fit_svm_tanimoto, "proba": proba_svm_tanimoto, "needs_fps": True,
        "grid": [{"C": c} for c in (0.1, 1.0, 10.0)],
    },
    "kNN (Tanimoto)": {
        "fit": fit_knn_tanimoto, "proba": proba_knn_tanimoto, "needs_fps": True,
        "grid": [{"k": k} for k in (1, 3, 5, 9)],
    },
    "LogReg (L2)": {
        "fit": fit_logreg, "proba": proba_logreg, "needs_fps": False,
        "grid": [{"C": c} for c in (0.01, 0.1, 1.0, 10.0)],
    },
    "BernoulliNB": {
        "fit": fit_nb, "proba": proba_nb, "needs_fps": False,
        "grid": [{"alpha": a} for a in (0.1, 1.0, 5.0)],
    },
}
# `gamma: None` in the KPLS grid means 1/n_features, resolved once X exists
for _g in MODEL_ZOO["KPLS-DA"]["grid"]:
    if _g["gamma"] is None:
        _g["gamma"] = 1.0 / X_tr.shape[1]

ZOO_MODELS = [m for m in CONFIG.enabled_models if m in MODEL_ZOO]
NN_ENABLED = ("NN" in CONFIG.enabled_models) and CONFIG.use_neural_network
if NN_ENABLED:
    raise RuntimeError(
        "F9 removed the neural network along with the ChEMBL pretraining set, so the\n"
        "classes it needs (MultiTaskClassifier, pretrain_trunk, finetune_multitask) are\n"
        "no longer defined in this notebook. Set use_neural_network = False and drop\n"
        "'NN' from enabled_models, or run F8, which still has it.")


def fit_any(name, X, y, params, fps=None, sample_weight=None, seed=42):
    spec = MODEL_ZOO[name]
    return spec["fit"](X, y, params, fps=fps if spec["needs_fps"] else None,
                       sample_weight=sample_weight, seed=seed)


def predict_proba_any(name, model, X, fps=None, chunk: Optional[int] = None) -> np.ndarray:
    """Probabilities, chunked when the model needs a kernel against the training set.

    Without chunking, scoring the library with SVM or kNN would materialise a
    200,560 x ~730 Gram. `chunk` keeps peak memory to chunk x n_train."""
    spec = MODEL_ZOO[name]
    if not spec["needs_fps"]:
        return spec["proba"](model, X, None)
    if chunk is None or len(X) <= chunk:
        return spec["proba"](model, X, fps)
    out = np.empty((len(X), N_CLASSES))
    for s in tqdm(range(0, len(X), chunk), desc=f"  {name}: scoring blocks",
                   unit="block", leave=False):
        e = min(s + chunk, len(X))
        out[s:e] = spec["proba"](model, X[s:e], fps[s:e])
    return out


def _stratified_bootstrap_idx(y, rng):
    by = {c: np.flatnonzero(y == c) for c in np.unique(y)}
    return np.concatenate([rng.choice(v, size=len(v), replace=True) for v in by.values()])


def bootstrap_any(name, X, y, params, n_bootstrap, fps=None, sample_weight=None, seed=42):
    """Stratified bootstrap ensemble, so a resample cannot lose a class."""
    rng = np.random.RandomState(seed)
    out = []
    for b in tqdm(range(n_bootstrap), desc=f"Bootstrap {name}", unit="model", leave=False):
        i = _stratified_bootstrap_idx(y, rng)
        f = None if fps is None else [fps[j] for j in i]
        sw = None if sample_weight is None else sample_weight[i]
        try:
            out.append(fit_any(name, X[i], y[i], params, fps=f, sample_weight=sw,
                               seed=seed + b))
        except Exception:
            continue
    return out


print(f"Model zoo: {len(MODEL_ZOO)} classical models registered, "
      f"{len(ZOO_MODELS)} enabled")
for nm in ZOO_MODELS:
    print(f"  {nm:<18s} {len(MODEL_ZOO[nm]['grid']):>3d} grid points"
          f"{'   [needs fingerprints]' if MODEL_ZOO[nm]['needs_fps'] else ''}")
print(f"  {'NN':<18s} {'via Optuna' if NN_ENABLED else 'DISABLED'}")
print("\nCommittee membership (which models enter the blend), per endpoint:")
for ep in EPS:
    mem = [m for m in CONFIG.committee.get(ep, ZOO_MODELS)
           if m in ZOO_MODELS or (m == "NN" and NN_ENABLED)]
    print(f"  {ep:<6s}: {', '.join(mem)}")
print("\nkNN (Tanimoto) is the diagnostic to watch: it is pure nearest-neighbour lookup,")
print("so anything the committee gains over it is what the models add beyond 'this looks")
print("like a known active'.")

## 7b. Part C.1b — Metrics

Fixed before any model is fitted, so the evaluation protocol is not chosen after seeing
results.

`classification_metrics` returns ROC-AUC, PR-AUC, balanced accuracy, MCC, precision,
recall, F1, Brier and ECE for one (fold, model, endpoint) row, returning `NaN` rather than
raising on a single-class fold.

The two that matter most for a screen: **`enrichment_factor`** is the hit rate in the top
fraction of a ranking divided by the overall hit rate (EF = 1 is worthless; EF = 10 at 1%
means the top percentile is ten times richer in actives), and **`precision_at_n`** is the
blunter form — of the top n ranked, what fraction are genuinely active. Note the small-fold
trap: at n = 30, `EF@1%` is computed over a single compound, so only the pooled row's
enrichment figures carry information.

`blend_equal` is the committee combiner. F5 fitted simplex weights and lost to a plain
average in all six scheme × endpoint comparisons, by up to 0.08 R² under grouped splits —
and fitted them on the very test set it then scored against.

In [ ]:
# --- Metrics, fixed before any model is fitted -----------------------------------------
from sklearn.metrics import (roc_auc_score, average_precision_score, balanced_accuracy_score,
                              matthews_corrcoef, brier_score_loss, precision_score,
                              recall_score, f1_score)


def _auc_safe(y_true, p_pos) -> float:
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, p_pos))


def enrichment_factor(y_true, scores, frac: float) -> float:
    """Hit rate in the top `frac` divided by the overall hit rate. EF = 1 is a worthless
    ranking. Beware small folds: at n = 14, the top 1% is one compound."""
    y_true = np.asarray(y_true)
    base = (y_true == POSITIVE_CLASS).mean()
    if base == 0 or len(y_true) == 0:
        return float("nan")
    k = max(1, int(round(frac * len(y_true))))
    top = np.argsort(-np.asarray(scores))[:k]
    return float((y_true[top] == POSITIVE_CLASS).mean() / base)


def precision_at_n(y_true, scores, n: int) -> float:
    y_true = np.asarray(y_true)
    n = min(n, len(y_true))
    if n == 0:
        return float("nan")
    top = np.argsort(-np.asarray(scores))[:n]
    return float((y_true[top] == POSITIVE_CLASS).mean())


def expected_calibration_error(y_bin, p, n_bins=10) -> float:
    edges = np.linspace(0, 1, n_bins + 1)
    ece, n = 0.0, len(p)
    if n == 0:
        return float("nan")
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p >= lo) & (p < hi) if hi < 1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        ece += (m.sum() / n) * abs(y_bin[m].mean() - p[m].mean())
    return float(ece)


def classification_metrics(y_true, p_pos, decision_threshold: float = 0.5) -> Dict[str, float]:
    """One (fold, model, endpoint) row. Returns NaN rather than raising on a single-class
    fold — leave-one-cluster-out will produce them."""
    y_true = np.asarray(y_true).astype(int)
    p_pos = np.asarray(p_pos, dtype=float)
    y_pred = (p_pos >= decision_threshold).astype(int)
    multi = len(np.unique(y_true)) > 1
    out = {
        "n": int(len(y_true)),
        "n_active": int((y_true == POSITIVE_CLASS).sum()),
        "base_rate": float((y_true == POSITIVE_CLASS).mean()) if len(y_true) else float("nan"),
        "roc_auc": _auc_safe(y_true, p_pos),
        "pr_auc": float(average_precision_score(y_true, p_pos)) if multi else float("nan"),
        "balanced_acc": float(balanced_accuracy_score(y_true, y_pred)) if multi else float("nan"),
        "mcc": float(matthews_corrcoef(y_true, y_pred)) if multi else float("nan"),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "brier": float(brier_score_loss(y_true, np.clip(p_pos, 0, 1))) if len(y_true) else float("nan"),
        "ECE": expected_calibration_error(y_true, np.clip(p_pos, 0, 1)) if len(y_true) else float("nan"),
    }
    for frac in CONFIG.enrichment_fractions:
        out[f"EF@{int(frac * 100)}%"] = enrichment_factor(y_true, p_pos, frac)
    return out


def blend_equal(p_list: List[np.ndarray]) -> np.ndarray:
    """Equal-weight average of P(active) vectors. F5's fitted simplex weights lost to a
    plain average in all six scheme x endpoint comparisons, by up to 0.08 R2 under grouped
    splits, and were fitted on the very test set they were scored against."""
    return np.mean(np.stack([np.asarray(p, dtype=float) for p in p_list], axis=0), axis=0)


print("Metrics defined: classification_metrics, enrichment_factor, precision_at_n,")
print("expected_calibration_error, blend_equal.")
print(f"EF at {', '.join(f'{int(f * 100)}%' for f in CONFIG.enrichment_fractions)}; "
      f"precision@{list(CONFIG.precision_at_n)}.")

## 8. Part D — Tuning the zoo

One generic loop over the registry: for each endpoint and each enabled model, every grid
point is cross-validated on that endpoint's labelled training rows with stratified folds
and ROC-AUC, then ranked by the AutoQSAR gap-aware score.

The gap penalty systematically prefers under-fit models — on F4's pKi data it chose a
50-tree, 7-leaf LightGBM over a configuration with a better held-out score. The cell now
prints, per model, how much test AUC that choice cost whenever the difference exceeds
0.005, so the trade is visible rather than buried.

Grids are deliberately small: the point of the extra models is a diversity of inductive
biases, not an exhaustive search of each. The CV numbers printed here come from the
training split and carry the same analogue leakage as any random split — Part E.2 is the
number that counts.

In [ ]:
# --- Tune every enabled model, per endpoint -----------------------------------------------
import numpy as np
from sklearn.model_selection import StratifiedKFold


def autoqsar_score(acc_train: float, acc_test: float) -> float:
    """AutoQSAR's gap-aware score (Dixon et al. 2016) on ROC-AUC with a 0.5 floor.

    Caveat carried from the F4 review: the gap penalty systematically prefers under-fit
    models. The cell says so explicitly whenever the winner is not the configuration with
    the better held-out AUC."""
    if acc_test < 0.5:
        return acc_test - 0.5
    return (acc_test - 0.5) * (1.0 - abs(acc_train - acc_test)) + 0.5


def cv_evaluate(name, params, X, y, fps, skf, seed):
    """(mean train AUC, mean test AUC) over the folds; NaN if it cannot be fitted."""
    tr_a, te_a = [], []
    for a_i, b_i in skf.split(X, y):
        try:
            f_a = None if fps is None else [fps[j] for j in a_i]
            f_b = None if fps is None else [fps[j] for j in b_i]
            m = fit_any(name, X[a_i], y[a_i], params, fps=f_a, seed=seed)
            tr_a.append(_auc_safe(y[a_i], predict_proba_any(name, m, X[a_i],
                                                            f_a)[:, POSITIVE_CLASS]))
            te_a.append(_auc_safe(y[b_i], predict_proba_any(name, m, X[b_i],
                                                            f_b)[:, POSITIVE_CLASS]))
        except Exception:
            continue
    if not te_a:
        return float("nan"), float("nan")
    return float(np.nanmean(tr_a)), float(np.nanmean(te_a))


best_params: Dict[str, Dict[str, dict]] = {ep: {} for ep in EPS}
tuning_rows = []

for t, ep in enumerate(tqdm(EPS, desc="Tuning", unit="endpoint")):
    m_lab, y_ep = _labelled(Y_tr, t)
    Xs = X_tr[m_lab]
    # global fingerprint indices for this endpoint's labelled training rows
    fps_s = [train_fps[i] for i in idx_tr[m_lab]]
    n_splits = min(CONFIG.cv_folds, int(np.bincount(y_ep).min()))
    if n_splits < 2:
        tqdm.write(f"\n=== {ep}: too few of one class to tune; zoo defaults used ===")
        for nm in ZOO_MODELS:
            best_params[ep][nm] = dict(MODEL_ZOO[nm]["grid"][0])
        continue
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=CONFIG.random_state)
    tqdm.write(f"\n=== {ep}: {len(y_ep)} labelled rows ({int(y_ep.sum())} active), "
               f"{n_splits}-fold stratified CV ===")

    for nm in tqdm(ZOO_MODELS, desc=f"  {ep}", unit="model", leave=False):
        grid = MODEL_ZOO[nm]["grid"]
        needs = MODEL_ZOO[nm]["needs_fps"]
        rows = []
        for params in tqdm(grid, desc=f"    {nm}", unit="cfg", leave=False):
            a, b = cv_evaluate(nm, params, Xs, y_ep, fps_s if needs else None, skf,
                               CONFIG.random_state)
            if not np.isfinite(b):
                continue
            rows.append({"params": params, "train_auc": a, "test_auc": b,
                         "autoqsar": autoqsar_score(a, b)})
        if not rows:
            best_params[ep][nm] = dict(grid[0])
            tqdm.write(f"  {nm:<18s} every configuration failed; default used")
            continue
        res = pd.DataFrame(rows).sort_values("autoqsar", ascending=False)
        best = res.iloc[0]
        best_params[ep][nm] = dict(best["params"])
        by_auc = res.sort_values("test_auc", ascending=False).iloc[0]
        flag = ""
        if by_auc["test_auc"] - best["test_auc"] > 0.005:
            flag = (f"   [gap penalty cost {by_auc['test_auc'] - best['test_auc']:+.3f} "
                    f"test AUC]")
        tqdm.write(f"  {nm:<18s} AutoQSAR={best['autoqsar']:.3f} "
                   f"(train {best['train_auc']:.3f}, test {best['test_auc']:.3f})  "
                   f"{best_params[ep][nm]}{flag}")
        tuning_rows.append({"endpoint": ep, "model": nm, "autoqsar": best["autoqsar"],
                            "train_auc": best["train_auc"], "test_auc": best["test_auc"],
                            "params": str(best_params[ep][nm]),
                            "best_possible_test_auc": by_auc["test_auc"]})

tuning_df = pd.DataFrame(tuning_rows)
print("\nBest CV test AUC per endpoint (tuning-set estimate, NOT the honest number):")
if len(tuning_df):
    display(tuning_df.pivot_table(index="model", columns="endpoint",
                                  values="test_auc").round(3))
print("These come from stratified CV on the training split, so they carry the same")
print("analogue leakage as any random split. Part E.2 is the number that counts.")

## 9. Part E.1 — Train the final models

The network is trained twice — once on the stratified random training split for the
reference numbers at the end of this section, once on the full set for the models that score
the library — and each baseline family is fitted per endpoint on both, plus stratified
bootstrap ensembles.

The reference scores are **not** the headline. A stratified random split of a set made of a
handful of congeneric families tests mostly on close analogues of the training compounds;
that is the flaw F4 quantified and Part E.2 replaces. In F6 the gap was ROC-AUC 0.89 random
against 0.69 pooled leave-one-cluster-out.

In [ ]:
# --- Train the final models ---------------------------------------------------------------
nn_hidden_dims_final = (nn_best_params["h1"], nn_best_params["h2"]) if NN_ENABLED else None

_PRETRAINED_STATE, pretrained_final = None, None
if NN_ENABLED and CONFIG.use_pretraining:
    pretrained_final = pretrain_trunk(
        X_pretrain, y_pretrain, hidden_dims=nn_hidden_dims_final,
        dropout=nn_best_params["dropout"], lr=CONFIG.nn_lr,
        weight_decay=nn_best_params["weight_decay"], epochs=CONFIG.nn_pretrain_epochs,
        batch_size=CONFIG.nn_pretrain_batch_size, seed=CONFIG.random_state,
        desc="Final trunk pretraining (full ChEMBL set)")
    _PRETRAINED_STATE = pretrained_final.trunk.state_dict()


def _train_nn(X, Y, w, trunk_state, seed, desc, leave=False):
    return finetune_multitask(
        trunk_state, X, Y, hidden_dims=nn_hidden_dims_final,
        dropout=nn_best_params["dropout"], lr=nn_best_params["lr"],
        weight_decay=nn_best_params["weight_decay"],
        freeze_epochs=nn_best_params.get("freeze_epochs", 0),
        total_epochs=CONFIG.nn_finetune_total_epochs,
        batch_size=nn_best_params["batch_size"], sample_weight=w,
        seed=seed, desc=desc, leave=leave)


nn_model_eval = nn_model_deploy = None
if NN_ENABLED:
    nn_model_eval = _train_nn(X_tr, Y_tr, w_tr, _PRETRAINED_STATE, CONFIG.random_state,
                              "Reference model (random split)", leave=True)
    nn_model_deploy = _train_nn(X_train_full, Y_train_full, w_train_full,
                                _PRETRAINED_STATE, CONFIG.random_state,
                                "Deploy model (full PSMA set)", leave=True)
else:
    print("Neural network disabled — classical models only.")

# --- every zoo model, per endpoint, on the reference split and on everything -------------
models_eval: Dict[str, Dict[str, object]] = {ep: {} for ep in EPS}
models_deploy: Dict[str, Dict[str, object]] = {ep: {} for ep in EPS}
models_boot: Dict[str, Dict[str, list]] = {ep: {} for ep in EPS}

for t, ep in enumerate(tqdm(EPS, desc="Fitting the zoo", unit="endpoint")):
    m_tr, y_tr_ep = _labelled(Y_tr, t)
    m_all, y_all_ep = _labelled(Y_train_full, t)
    fps_tr = [train_fps[i] for i in idx_tr[m_tr]]
    fps_all = [train_fps[i] for i in np.flatnonzero(m_all)]
    for nm in tqdm(ZOO_MODELS, desc=f"  {ep}", unit="model", leave=False):
        prm = best_params[ep][nm]
        try:
            models_eval[ep][nm] = fit_any(nm, X_tr[m_tr], y_tr_ep, prm, fps=fps_tr,
                                          sample_weight=w_tr[m_tr], seed=CONFIG.random_state)
            models_deploy[ep][nm] = fit_any(nm, X_train_full[m_all], y_all_ep, prm,
                                            fps=fps_all,
                                            sample_weight=w_train_full[m_all],
                                            seed=CONFIG.random_state)
            models_boot[ep][nm] = bootstrap_any(nm, X_train_full[m_all], y_all_ep, prm,
                                                CONFIG.n_bootstrap, fps=fps_all,
                                                sample_weight=w_train_full[m_all],
                                                seed=CONFIG.random_state)
        except Exception as e:
            tqdm.write(f"  {ep}/{nm}: failed ({type(e).__name__}: {e}); excluded")

print("\nFitted per endpoint:")
for ep in EPS:
    print(f"  {ep}: " + ", ".join(f"{nm} ({len(models_boot[ep].get(nm, []))} boot)"
                                  for nm in models_deploy[ep]))


def committee_for(ep: str) -> List[str]:
    """Blend membership for one endpoint, filtered to what actually fitted."""
    wanted = list(CONFIG.committee.get(ep, ["NN"] + ZOO_MODELS))
    out = []
    for nm in wanted:
        if nm == "NN":
            if NN_ENABLED:
                out.append(nm)
        elif nm in models_deploy[ep]:
            out.append(nm)
    return out


# --- reference-split scores, comparable with F7's random-split row ----------------------
_P_nn = predict_proba_mt(nn_model_eval, X_te) if NN_ENABLED else None
ref_rows = []
for t, ep in enumerate(EPS):
    m = np.isfinite(Y_te[:, t])
    if m.sum() < 5 or len(np.unique(Y_te[m, t])) < 2:
        continue
    y_true = Y_te[m, t].astype(int)
    fps_te = [train_fps[i] for i in idx_te[m]]
    preds = {}
    if NN_ENABLED:
        preds["NN"] = _P_nn[m, t, POSITIVE_CLASS]
    for nm in models_eval[ep]:
        try:
            preds[nm] = predict_proba_any(nm, models_eval[ep][nm], X_te[m],
                                          fps_te)[:, POSITIVE_CLASS]
        except Exception:
            continue
    mem = [nm for nm in committee_for(ep) if nm in preds]
    if len(mem) >= 2:
        preds[f"Blend ({len(mem)})"] = blend_equal([preds[nm] for nm in mem])
    for nm, p in preds.items():
        ref_rows.append({"endpoint": ep, "model": nm, **classification_metrics(y_true, p)})

ref_eval_df = pd.DataFrame(ref_rows).set_index(["endpoint", "model"]).round(3)
print("\nSingle stratified random split — REFERENCE ONLY. Part E.2's leave-one-cluster-out")
print("is the evaluation that reflects prospective use:\n")
display(ref_eval_df)

## 10. Part E.2 — Leave-one-cluster-out evaluation

Whole chemical families are held out one at a time, so the result reads as "generalises to
these families, fails on that one" rather than as an average that describes neither.

**What changed from F6.** F6 reported eight folds of which four could not be scored:
cluster 6 had zero actives (every metric `NaN`), cluster 4 had one active in 18 compounds
(ROC-AUC 0.059 — a single compound landing low in a ranking, not a measurement), cluster 1
had 14 compounds at a 71% base rate (AUC 1.000, nearly free at that balance). Averaging
those produced a macro row that meant very little, and it also made the
distance-versus-difficulty correlation (r = −0.056) look like a finding when it was really
an underpowered test.

A cluster now earns its own row only if it clears **both** `loco_min_cluster_size` and
`loco_min_fold_actives` on at least one endpoint. Everything smaller is distributed across
`loco_n_remainder_folds` size-balanced "small families" folds. Every compound still
receives an out-of-fold prediction, so pooled coverage stays complete — only the per-fold
table changes, and it becomes readable.

**Three aggregates.** **Pooled** concatenates all out-of-fold predictions and scores them
once; it weights every compound equally and is the number to quote. **Macro** averages the
per-fold scores and is shown only because the spread across families is itself
informative. **Random split** runs alongside purely so the leakage gap to F5/F6
methodology stays visible — in F6 that gap was 0.89 → 0.69 ROC-AUC.

**Five committee members.** The network, LightGBM, KPLS-DA, the three-way equal blend, and
a two-way LightGBM + KPLS-DA blend. The last is new: in F6 KPLS-DA alone beat the three-way
average on the honest evaluation (AUC 0.706 vs 0.690, precision@50 0.58 vs 0.52), which
raises the question of whether the network is diluting the other two. Everything is scored
**per endpoint**, since a compound may have a label for one and not the other.

In [ ]:
# --- Build the folds -------------------------------------------------------------------
from rdkit.ML.Cluster import Butina
from sklearn.model_selection import StratifiedShuffleSplit


def butina_cluster_groups(fps, cutoff: float = 0.4) -> np.ndarray:
    n = len(fps)
    dists: List[float] = []
    for i in tqdm(range(1, n), desc=f"Butina distances (cutoff={cutoff})", leave=False):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dists.extend(1.0 - s for s in sims)
    clusters = Butina.ClusterData(dists, n, cutoff, isDistData=True)
    groups = np.empty(n, dtype=int)
    for ci, members in enumerate(clusters):
        for idx in members:
            groups[idx] = ci
    return groups


butina_groups = butina_cluster_groups(train_fps, cutoff=CONFIG.butina_cutoff)
_sizes = pd.Series(butina_groups).value_counts()
print(f"Butina clustering of {len(train_df):,} compounds (cutoff {CONFIG.butina_cutoff}):")
print(f"  {len(_sizes)} clusters | largest {_sizes.iloc[0]} "
      f"({100 * _sizes.iloc[0] / len(butina_groups):.1f}%) | "
      f"median {int(_sizes.median())} | singletons {int((_sizes == 1).sum())}")


def _fold_is_scorable(mask) -> bool:
    """At least `loco_min_fold_actives` of BOTH classes on at least one endpoint."""
    for t in range(Y_train_full.shape[1]):
        col = Y_train_full[mask, t]
        lab = col[np.isfinite(col)]
        if (lab == 1).sum() >= CONFIG.loco_min_fold_actives and \
           (lab == 0).sum() >= CONFIG.loco_min_fold_actives:
            return True
    return False


own_fold, pooled_clusters, rejected = [], [], []
for c in _sizes.index:
    c = int(c)
    mask = butina_groups == c
    if _sizes[c] < CONFIG.loco_min_cluster_size:
        pooled_clusters.append(c)
    elif not _fold_is_scorable(mask):
        pooled_clusters.append(c)
        rejected.append((c, int(_sizes[c])))
    elif len(own_fold) < CONFIG.loco_max_clusters:
        own_fold.append(c)
    else:
        pooled_clusters.append(c)

print(f"\n  {len(own_fold)} clusters get their own fold "
      f"(>= {CONFIG.loco_min_cluster_size} compounds and "
      f">= {CONFIG.loco_min_fold_actives} of each class)")
if rejected:
    print(f"  {len(rejected)} large-enough clusters pooled for being too unbalanced "
          f"to score: " + ", ".join(f"cluster {c} (n={n})" for c, n in rejected[:6]))
print(f"  {len(pooled_clusters)} clusters pooled into "
      f"{CONFIG.loco_n_remainder_folds} size-balanced 'small families' folds")

# greedy size-balanced assignment of the pooled clusters
_bins = [[] for _ in range(max(1, CONFIG.loco_n_remainder_folds))]
_bin_sizes = [0] * len(_bins)
for c in sorted(pooled_clusters, key=lambda c: -_sizes[c]):
    j = int(np.argmin(_bin_sizes))
    _bins[j].append(c)
    _bin_sizes[j] += int(_sizes[c])

LOCO_FOLDS = []
for c in own_fold:
    LOCO_FOLDS.append({"fold": f"cluster {c}", "mask": butina_groups == c})
for j, cs in enumerate(_bins):
    if not cs:
        continue
    LOCO_FOLDS.append({"fold": f"small families #{j + 1}",
                       "mask": np.isin(butina_groups, cs)})

# metadata
for f in tqdm(LOCO_FOLDS, desc="Fold metadata", unit="fold"):
    m = f["mask"]
    sub = train_df[m]
    f["n"] = int(m.sum())
    for t, ep in enumerate(EPS):
        col = Y_train_full[m, t]
        lab = col[np.isfinite(col)]
        f[f"{ep}_n"] = int(len(lab))
        f[f"{ep}_active"] = int((lab == 1).sum())
    srcs = sub["source_tag"].value_counts()
    f["dominant_source"] = srcs.index[0] if len(srcs) else "?"
    f["source_share"] = round(float(srcs.iloc[0] / len(sub)), 2) if len(srcs) else np.nan
    d = bulk_min_tanimoto_dissimilarity(
        [train_fps[i] for i in np.flatnonzero(m)],
        [train_fps[i] for i in np.flatnonzero(~m)],
        desc=f"  {f['fold']}: distance to train")
    f["mean_tanimoto_to_train"] = round(float(np.mean(d)), 3)

loco_folds_df = pd.DataFrame([{k: v for k, v in f.items() if k != "mask"}
                              for f in LOCO_FOLDS]).set_index("fold")
fold_of = np.full(len(train_df), -1, dtype=int)
for i, f in enumerate(LOCO_FOLDS):
    fold_of[f["mask"]] = i

print(f"\n{len(LOCO_FOLDS)} folds covering "
      f"{int((fold_of >= 0).sum()):,} of {len(train_df):,} compounds:")
display(loco_folds_df)

_single = loco_folds_df[(loco_folds_df["source_share"] >= 0.6)
                        & (~loco_folds_df["dominant_source"].isin(GENERIC_SOURCE_TAGS))]
if len(_single):
    print("Folds dominated by a single filing — read these carefully, because holding one")
    print("out removes a large, chemically narrow block of training data at the same time")
    print("as it poses an unfamiliar test set:")
    for name, r in _single.iterrows():
        print(f"  {name}: {r['n']} compounds, {100 * r['source_share']:.0f}% from "
              f"{r['dominant_source']}, mean Tanimoto distance {r['mean_tanimoto_to_train']}")

In [ ]:
# --- Run the folds ------------------------------------------------------------------------
BLEND_NAMES: Dict[str, str] = {}
for ep in EPS:
    BLEND_NAMES[ep] = f"Blend ({len(committee_for(ep))})"

MODEL_ORDER = (["NN"] if NN_ENABLED else []) + list(ZOO_MODELS) \
    + sorted(set(BLEND_NAMES.values()))
if CONFIG.run_pretraining_ablation and NN_ENABLED:
    MODEL_ORDER = ["NN", "NN (transfer)"] + MODEL_ORDER[1:]

_ABLATION_STATE = None
if CONFIG.run_pretraining_ablation and NN_ENABLED:
    if _PRETRAINED_STATE is not None:
        _ABLATION_STATE = _PRETRAINED_STATE
    else:
        print("run_pretraining_ablation = True but use_pretraining = False: pretraining a "
              "trunk purely for the comparison.")
        _abl = pretrain_trunk(X_pretrain, y_pretrain, hidden_dims=nn_hidden_dims_final,
                              dropout=nn_best_params["dropout"], lr=CONFIG.nn_lr,
                              weight_decay=nn_best_params["weight_decay"],
                              epochs=CONFIG.nn_pretrain_epochs,
                              batch_size=CONFIG.nn_pretrain_batch_size,
                              seed=CONFIG.random_state,
                              desc="Ablation: pretraining a trunk for comparison")
        _ABLATION_STATE = _abl.trunk.state_dict()


def _fit_committee(tr_i, te_i, seed, label):
    """{model: {endpoint: P(active) on te_i}}. A member that cannot be fitted for an
    endpoint is dropped for that endpoint only, never for the whole fold."""
    Xa, Ya, wa = X_train_full[tr_i], Y_train_full[tr_i], w_train_full[tr_i]
    Xb = X_train_full[te_i]
    fps_b_all = [train_fps[i] for i in te_i]
    out: Dict[str, Dict[str, np.ndarray]] = {m: {} for m in MODEL_ORDER}

    if NN_ENABLED:
        arms = [("NN", _PRETRAINED_STATE if CONFIG.use_pretraining else None)]
        if CONFIG.run_pretraining_ablation:
            arms.append(("NN (transfer)", _ABLATION_STATE))
        for nm, trunk in arms:
            mdl = _train_nn(Xa, Ya, wa, trunk, seed, f"  {label}: {nm}", leave=False)
            P = predict_proba_mt(mdl, Xb)
            for t, ep in enumerate(EPS):
                out[nm][ep] = P[:, t, POSITIVE_CLASS]

    for t, ep in enumerate(EPS):
        m_a = np.isfinite(Ya[:, t])
        y_a = Ya[m_a, t].astype(int)
        if m_a.sum() < 10 or len(np.unique(y_a)) < 2:
            continue
        fps_a = [train_fps[i] for i in tr_i[m_a]]
        for nm in ZOO_MODELS:
            try:
                mdl = fit_any(nm, Xa[m_a], y_a, best_params[ep][nm], fps=fps_a,
                              sample_weight=wa[m_a], seed=seed)
                out[nm][ep] = predict_proba_any(nm, mdl, Xb, fps_b_all,
                                                chunk=CONFIG.screen_chunk_size)[:, POSITIVE_CLASS]
            except Exception as e:
                tqdm.write(f"  {label}/{ep}/{nm}: failed ({type(e).__name__})")
        mem = [nm for nm in committee_for(ep) if ep in out.get(nm, {})]
        if len(mem) >= 2:
            out[BLEND_NAMES[ep]][ep] = blend_equal([out[nm][ep] for nm in mem])
    return out


oof = {m: {ep: np.full(len(train_df), np.nan) for ep in EPS} for m in MODEL_ORDER}
loco_rows = []

for fi, f in enumerate(tqdm(LOCO_FOLDS, desc="Leave-one-cluster-out", unit="fold")):
    te_i = np.flatnonzero(f["mask"])
    tr_i = np.flatnonzero(~f["mask"])
    if not np.isfinite(Y_train_full[tr_i]).any():
        tqdm.write(f"  {f['fold']}: no labels in the training half; skipped")
        continue
    probas = _fit_committee(tr_i, te_i, CONFIG.random_state + 7000 + fi, f["fold"])
    for nm in MODEL_ORDER:
        for t, ep in enumerate(EPS):
            p = probas.get(nm, {}).get(ep)
            if p is None:
                continue
            oof[nm][ep][te_i] = p
            m_lab = np.isfinite(Y_train_full[te_i, t])
            # F7 reported a pKi fold with n=5 whose metrics meant nothing; floor raised.
            if m_lab.sum() < CONFIG.loco_min_fold_n or \
               len(np.unique(Y_train_full[te_i][m_lab, t])) < 2:
                continue
            loco_rows.append({
                "fold": f["fold"], "endpoint": ep, "model": nm,
                "dominant_source": f["dominant_source"],
                "mean_tanimoto_to_train": f["mean_tanimoto_to_train"],
                **classification_metrics(Y_train_full[te_i][m_lab, t].astype(int),
                                         p[m_lab]),
            })

loco_eval_df = pd.DataFrame(loco_rows)
print(f"\nCollected {len(loco_eval_df)} (fold, endpoint, model) results "
      f"(folds needed >= {CONFIG.loco_min_fold_n} labelled test compounds to be reported; "
      f"all folds still contribute to the pooled estimate).")

# --- random-split reference -------------------------------------------------------------
from sklearn.model_selection import StratifiedShuffleSplit

rand_rows = []
_strat_ref = np.where(np.isfinite(Y_train_full[:, 0]), Y_train_full[:, 0], 2).astype(int)
_sss = StratifiedShuffleSplit(n_splits=CONFIG.loco_random_reference_repeats,
                              test_size=CONFIG.test_size, random_state=CONFIG.random_state)
for r, (tr_i, te_i) in enumerate(tqdm(list(_sss.split(X_train_full, _strat_ref)),
                                       desc="Random-split reference", unit="split")):
    probas = _fit_committee(tr_i, te_i, CONFIG.random_state + 9000 + r, f"random {r}")
    for nm in MODEL_ORDER:
        for t, ep in enumerate(EPS):
            p = probas.get(nm, {}).get(ep)
            if p is None:
                continue
            m_lab = np.isfinite(Y_train_full[te_i, t])
            if m_lab.sum() < CONFIG.loco_min_fold_n or \
               len(np.unique(Y_train_full[te_i][m_lab, t])) < 2:
                continue
            rand_rows.append({
                "repeat": r, "endpoint": ep, "model": nm,
                **classification_metrics(Y_train_full[te_i][m_lab, t].astype(int),
                                         p[m_lab]),
            })

random_eval_df = pd.DataFrame(rand_rows)
print(f"Collected {len(random_eval_df)} (repeat, endpoint, model) random-split results.")

## 10b. Part E.2b — Results, per family and pooled

Read **pooled** first: every compound scored once, on a prediction made while its whole
chemical family was absent from training. **Per-fold** says which families are reachable.
**Macro** only shows the spread.

New in F8, and the most informative part: **the floor comparisons.** kNN (Tanimoto) is pure
similarity lookup and LogReg (L2) is a linear model on fingerprints. Any model that cannot
clear those by a clear margin is not learning structure–activity relationships so much as
recognising neighbours — which, for a pipeline whose purpose is to rank unfamiliar
chemistry, is the thing most worth knowing. The cell also checks whether the blend beats its
best single member, since F7 found that on pKi it did not.

In [ ]:
# --- Summaries -----------------------------------------------------------------------------
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 70)

_metrics = [c for c in ["roc_auc", "pr_auc", "balanced_acc", "mcc", "precision", "recall",
                        "brier", "ECE"] if c in loco_eval_df.columns]
_efs = [c for c in loco_eval_df.columns if c.startswith("EF@")]
_models = [m for m in MODEL_ORDER if m in set(loco_eval_df["model"])]

print("=" * 120)
print("PER-FOLD RESULTS")
print("=" * 120)
display(loco_eval_df.set_index(["endpoint", "fold", "model"])[
    ["n", "n_active", "base_rate", "mean_tanimoto_to_train"] + _metrics + _efs].round(3))

print("\n" + "=" * 120)
print("POOLED — out-of-fold predictions concatenated and scored once. QUOTE THESE.")
print("=" * 120)
pooled_rows = []
for ep_i, ep in enumerate(EPS):
    y_col = Y_train_full[:, ep_i]
    for nm in _models:
        p = oof[nm][ep]
        ok = np.isfinite(p) & np.isfinite(y_col)
        if ok.sum() < 20 or len(np.unique(y_col[ok])) < 2:
            continue
        met = classification_metrics(y_col[ok].astype(int), p[ok])
        met.update({"endpoint": ep, "model": nm, "coverage": int(ok.sum())})
        met.update({f"precision@{k}": precision_at_n(y_col[ok].astype(int), p[ok], k)
                    for k in CONFIG.precision_at_n})
        pooled_rows.append(met)
pooled_df = pd.DataFrame(pooled_rows).set_index(["endpoint", "model"]).round(3)
display(pooled_df)

print("\n" + "=" * 120)
print("MACRO — mean +/- s.d. across folds (spread, not an estimate)")
print("=" * 120)
macro_df = (loco_eval_df.groupby(["endpoint", "model"])[_metrics + _efs]
            .agg(["mean", "std"]).round(3))
display(macro_df)

print("\n" + "=" * 120)
print("RANDOM-SPLIT REFERENCE — the gap to pooled is the leakage")
print("=" * 120)
rand_summary = (random_eval_df.groupby(["endpoint", "model"])[_metrics + _efs]
                .agg(["mean", "std"]).round(3))
display(rand_summary)

print("\nHeadline — ROC-AUC, random split vs leave-one-cluster-out (pooled):")
for ep in EPS:
    print(f"\n  {ep}:")
    try:
        order = pooled_df.loc[ep, "roc_auc"].sort_values(ascending=False).index
    except KeyError:
        continue
    for nm in order:
        pl = pooled_df.loc[(ep, nm), "roc_auc"]
        try:
            rd = rand_summary.loc[(ep, nm), ("roc_auc", "mean")]
            print(f"    {nm:<20s} random {rd:.3f}  ->  pooled {pl:.3f}   [{pl - rd:+.3f}]")
        except KeyError:
            print(f"    {nm:<20s} random   n/a  ->  pooled {pl:.3f}")

# =====================================================================================
# The two floors: does anything beat similarity lookup, or a linear model?
# =====================================================================================
print("\n" + "=" * 120)
print("FLOOR COMPARISONS — what are the extra models actually buying?")
print("=" * 120)
print("kNN (Tanimoto) is nearest-neighbour lookup: 'this looks like a known active'. It is")
print("the floor any QSAR model has to clear to be worth building. LogReg (L2) is the")
print("linear floor: if the ensembles cannot beat it, the complexity is not paying.\n")

for ep in EPS:
    try:
        col = pooled_df.loc[ep]
    except KeyError:
        continue
    print(f"  {ep} (pooled, whole families held out):")
    # F9: measured on ROC-AUC *and* on top-of-ranking precision. F8 compared AUC only and
    # reported a tie against both floors on pIC50 — but precision@50 there ran 0.88 for the
    # best model against 0.64 for kNN. AUC averages over the whole ranking, including the
    # middle nobody acts on; a screen only ever touches the top. Both are shown because
    # they can disagree, and when they do, the precision column is the operative one.
    _pn = f"precision@{CONFIG.precision_at_n[0]}"
    for floor in ("kNN (Tanimoto)", "LogReg (L2)"):
        if floor not in col.index:
            print(f"    {floor}: not available")
            continue
        base_a = col.loc[floor, "roc_auc"]
        base_p = col.loc[floor].get(_pn, np.nan)
        print(f"    {floor:<16s} ROC-AUC {base_a:.3f}"
              + (f" | {_pn} {base_p:.3f}" if np.isfinite(base_p) else ""))
        beat_a = sorted([(nm, col.loc[nm, "roc_auc"] - base_a) for nm in col.index
                         if nm != floor and col.loc[nm, "roc_auc"] - base_a > 0.02],
                        key=lambda r: -r[1])
        beat_p = []
        if np.isfinite(base_p):
            beat_p = sorted([(nm, col.loc[nm].get(_pn, np.nan) - base_p)
                             for nm in col.index
                             if nm != floor
                             and np.isfinite(col.loc[nm].get(_pn, np.nan))
                             and col.loc[nm].get(_pn, np.nan) - base_p > 0.05],
                            key=lambda r: -r[1])
        print("      on AUC (>0.02)      : "
              + (", ".join(f"{nm} ({d:+.3f})" for nm, d in beat_a[:5])
                 if beat_a else "NOTHING clears it"))
        print(f"      on {_pn} (>0.05): "
              + (", ".join(f"{nm} ({d:+.3f})" for nm, d in beat_p[:5])
                 if beat_p else "NOTHING clears it"))
        if not beat_a and not beat_p:
            if floor.startswith("kNN"):
                print("      !! Matched by nearest-neighbour lookup on BOTH measures. Nothing")
                print("         is being learned beyond 'this resembles a known active'.")
            else:
                print("      !! Matched by a logistic regression on BOTH measures. The trees")
                print("         and kernels are not paying for their complexity.")
        elif not beat_a and beat_p:
            print("      -> tie on whole-ranking discrimination, but the models ARE ahead at")
            print("         the top of the ranking, which is the part a screen uses.")
        elif beat_a and not beat_p:
            print("      -> ahead on overall discrimination but NOT at the top of the")
            print("         ranking. For a shortlist that is the wrong way round.")
    print()

# blend vs its best member
print("  Does the committee beat its best single member?")
for ep in EPS:
    try:
        col = pooled_df.loc[ep, "roc_auc"]
    except KeyError:
        continue
    blends = [m for m in col.index if m.startswith("Blend")]
    singles = [m for m in col.index if not m.startswith("Blend")]
    if not blends or not singles:
        continue
    b_name = max(blends, key=lambda m: col[m])
    s_name = max(singles, key=lambda m: col[m])
    d = col[b_name] - col[s_name]
    verdict = ("the blend earns its place" if d > 0.01
               else f"'{s_name}' alone is as good or better — the blend is not earning it"
               if d < -0.01 else "blend and best single are within 0.01")
    print(f"    {ep:<6s}: {b_name} {col[b_name]:.3f} vs best single {s_name} "
          f"{col[s_name]:.3f} ({d:+.3f}) — {verdict}")

print("\n  Per-endpoint best pooled model (this is what to deploy):")
for ep in EPS:
    try:
        col = pooled_df.loc[ep, "roc_auc"].sort_values(ascending=False)
        print(f"    {ep:<6s}: {col.index[0]} ({col.iloc[0]:.3f})")
    except KeyError:
        continue

## 10c. Part E.2c — Recalibrating the probabilities

F6's probabilities were roughly twice too high: mean predicted 0.50 against an observed
rate of 0.265, with expected calibration error of 0.24–0.31 across models. The cause is
`class_weight="balanced"`, which by construction pushes outputs toward 0.5 in order to
weight the minority class up. The *ranking* was fine — that is what ROC-AUC measures — but
the numbers were scores, not probabilities, and quoting one as "a 53% chance of being
active" would have been wrong.

The fix is a monotone post-hoc map from score to probability, fitted **cross-fitted across
the folds**: for each fold, the calibrator is trained on the out-of-fold predictions of
every *other* fold and then applied to this one. That matters — fitting a calibrator on the
same predictions you then evaluate makes the result look better than it is, which is the
same mistake F5 made with its blend weights.

**Platt scaling is the default**, not isotonic. The miscalibration being corrected is a
systematic ~2× inflation caused by the balanced class weighting, which is close to a pure
logit shift — precisely what a two-parameter sigmoid models. Isotonic regression is more
flexible, but it fits a step function, and with only a handful of folds it overfits and
collapses distinct scores into ties. In principle a monotone map cannot change a ranking or
an AUC; in practice isotonic's ties can, and a smoke test on noise features moved AUC by
0.10 that way. The cell reports the largest AUC change so this is visible rather than
silent — anything beyond rounding means ties were collapsed. Set
`CONFIG.recalibration = "isotonic"` if the before/after table shows a miscalibration shape
a sigmoid cannot follow.

A final calibrator fitted on all out-of-fold predictions is then applied to the library
scores in Part E.4, so the P(active) column there means what it says.

In [ ]:
# --- Cross-fitted probability recalibration -----------------------------------------------
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression as _LR


class _Calibrator:
    """Monotone score -> probability map."""

    def __init__(self, method="sigmoid"):
        self.method = method
        self._m = None

    def fit(self, p, y):
        p, y = np.asarray(p, dtype=float), np.asarray(y, dtype=int)
        ok = np.isfinite(p) & np.isfinite(y)
        p, y = p[ok], y[ok]
        if len(p) < 20 or len(np.unique(y)) < 2 or self.method == "none":
            self._m = None
            return self
        if self.method == "isotonic":
            self._m = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(p, y)
        else:
            self._m = _LR(max_iter=1000).fit(p.reshape(-1, 1), y)
        return self

    def transform(self, p):
        p = np.asarray(p, dtype=float)
        if self._m is None:
            return p
        out = np.full_like(p, np.nan)
        ok = np.isfinite(p)
        if self.method == "isotonic":
            out[ok] = self._m.predict(p[ok])
        else:
            out[ok] = self._m.predict_proba(p[ok].reshape(-1, 1))[:, 1]
        return np.clip(out, 0.0, 1.0)

    @property
    def slope(self) -> float:
        """Sign of the score -> probability relationship. Negative means the calibrator
        inverted the ranking: on the rows it saw, a higher score went with a LOWER chance
        of being active. That is a finding about the model, not about the calibration."""
        if self._m is None:
            return float("nan")
        if self.method == "isotonic":
            y = np.asarray(self._m.y_thresholds_).ravel()
            return float(y[-1] - y[0]) if len(y) > 1 else 0.0
        return float(np.ravel(self._m.coef_)[0])


METHOD = CONFIG.recalibration
print(f"Recalibration: '{METHOD}', cross-fitted across {len(LOCO_FOLDS)} folds.")
print(f"Skipped where the raw ECE is already below {CONFIG.recalibrate_min_ece:.2f} — "
      f"F7 recalibrated\npKi anyway (raw ECE 0.072-0.084) and made the blend worse "
      f"(0.084 -> 0.167).\n")

oof_cal = {m: {ep: np.full(len(train_df), np.nan) for ep in EPS} for m in MODEL_ORDER}
final_calibrators: Dict[tuple, _Calibrator] = {}
fold_slopes: List[dict] = []
skipped_pairs: List[tuple] = []

for ep_i, ep in enumerate(EPS):
    y_col = Y_train_full[:, ep_i]
    for nm in MODEL_ORDER:
        p = oof[nm][ep]
        ok_all = np.isfinite(p) & np.isfinite(y_col)
        if ok_all.sum() < 20 or len(np.unique(y_col[ok_all])) < 2:
            continue
        raw_ece = expected_calibration_error(y_col[ok_all].astype(int),
                                             np.clip(p[ok_all], 0, 1))
        if raw_ece < CONFIG.recalibrate_min_ece:
            oof_cal[nm][ep] = p.copy()
            final_calibrators[(nm, ep)] = _Calibrator("none").fit(p[ok_all],
                                                                  y_col[ok_all].astype(int))
            skipped_pairs.append((nm, ep, round(float(raw_ece), 4)))
            continue
        for fi in range(len(LOCO_FOLDS)):
            in_f = fold_of == fi
            out_f = (fold_of != fi) & (fold_of >= 0)
            tr_ok = out_f & np.isfinite(p) & np.isfinite(y_col)
            if tr_ok.sum() < 20:
                oof_cal[nm][ep][in_f] = p[in_f]
                continue
            cal = _Calibrator(METHOD).fit(p[tr_ok], y_col[tr_ok].astype(int))
            oof_cal[nm][ep][in_f] = cal.transform(p[in_f])
            fold_slopes.append({"model": nm, "endpoint": ep, "fold": fi,
                                "slope": cal.slope, "n_fit": int(tr_ok.sum())})
        final_calibrators[(nm, ep)] = _Calibrator(METHOD).fit(
            p[ok_all], y_col[ok_all].astype(int))

if skipped_pairs:
    print("Already well calibrated, left alone:")
    for nm, ep, e in sorted(skipped_pairs, key=lambda r: r[2]):
        print(f"  {nm:<18s} / {ep:<6s}  raw ECE {e:.4f}")
    print()

# --- before / after ---------------------------------------------------------------------
cal_rows = []
for ep_i, ep in enumerate(EPS):
    y_col = Y_train_full[:, ep_i]
    for nm in MODEL_ORDER:
        raw, cal = oof[nm][ep], oof_cal[nm][ep]
        ok = np.isfinite(raw) & np.isfinite(cal) & np.isfinite(y_col)
        if ok.sum() < 20 or len(np.unique(y_col[ok])) < 2:
            continue
        yb = y_col[ok].astype(int)
        cal_rows.append({
            "endpoint": ep, "model": nm, "n": int(ok.sum()),
            "calibrated": (nm, ep) not in [(a, b) for a, b, _ in skipped_pairs],
            "observed_rate": round(float(yb.mean()), 3),
            "mean_pred_raw": round(float(raw[ok].mean()), 3),
            "mean_pred_cal": round(float(cal[ok].mean()), 3),
            "ECE_raw": round(expected_calibration_error(yb, np.clip(raw[ok], 0, 1)), 4),
            "ECE_cal": round(expected_calibration_error(yb, np.clip(cal[ok], 0, 1)), 4),
            "brier_raw": round(float(brier_score_loss(yb, np.clip(raw[ok], 0, 1))), 4),
            "brier_cal": round(float(brier_score_loss(yb, np.clip(cal[ok], 0, 1))), 4),
            "auc_raw": round(_auc_safe(yb, raw[ok]), 4),
            "auc_cal": round(_auc_safe(yb, cal[ok]), 4),
        })

recalibration_df = pd.DataFrame(cal_rows).set_index(["endpoint", "model"])
display(recalibration_df)

_touched = recalibration_df[recalibration_df["calibrated"]]
if len(_touched):
    _imp = _touched["ECE_raw"] - _touched["ECE_cal"]
    print(f"Of the {len(_touched)} pairs that were recalibrated, ECE improved for "
          f"{int((_imp > 0).sum())}; mean reduction {_imp.mean():+.4f}")

_shift = (recalibration_df["auc_cal"] - recalibration_df["auc_raw"]).abs().max()
print(f"\nLargest absolute AUC change from calibration: {_shift:.4f}")
if _shift > 0.01:
    print("   This is EXPECTED and is not an inversion. The calibrated out-of-fold column")
    print("   is assembled from one map PER FOLD, so while each map preserves order within")
    print("   its own fold, the relative order of compounds in DIFFERENT folds can change")
    print("   when they are pooled. (F7's diagnostic blamed a negative slope here; that")
    print("   was wrong — the slopes were all positive.) Consequence:")
    print("     * quote the RAW pooled AUC for discrimination (Part E.2b)")
    print("     * quote the CALIBRATED numbers only for probability quality")
    print("   The library is scored with a single deployment calibrator per model, so its")
    print("   ranking is untouched.")

# --- genuine inversions ------------------------------------------------------------------
slopes_df = pd.DataFrame(fold_slopes)
_inv_fold = slopes_df[slopes_df["slope"] < 0] if len(slopes_df) else pd.DataFrame()
_inv_final = [(n, e, c.slope) for (n, e), c in final_calibrators.items()
              if np.isfinite(c.slope) and c.slope < 0]
if len(_inv_fold) or _inv_final:
    print("\n!! RANKING INVERSION — a calibrator learned that a higher score goes with a")
    print("   LOWER chance of being active, i.e. the model was anti-correlated with the")
    print("   label on the rows used to fit it.")
    if len(_inv_fold):
        summ = (_inv_fold.groupby(["endpoint", "model"])
                .agg(inverted_folds=("fold", "count"), worst_slope=("slope", "min")).round(3))
        _tot = slopes_df.groupby(["endpoint", "model"])["fold"].count()
        summ["total_folds"] = [int(_tot.get((e, m), 0)) for e, m in summ.index]
        display(summ)
    if _inv_final:
        print("   deployment calibrators (applied to the library):")
        for n, e, sl in sorted(_inv_final, key=lambda r: r[2]):
            print(f"     {n:<18s} / {e:<6s}  slope {sl:+.3f}  -- DO NOT SHIP this ranking")
    print("   Expect this only where the pooled AUC sits at or below ~0.5.")
else:
    print("   No calibrator inverted its ranking (all slopes positive).")

# --- reliability diagram -----------------------------------------------------------------
fig, axes = plt.subplots(1, len(EPS) + 1, figsize=(6.2 * (len(EPS) + 1), 5.0))
axes = np.atleast_1d(axes)
_cols = plt.get_cmap("tab10")
_show = [m for m in MODEL_ORDER if m.startswith("Blend")][:1] + \
        [m for m in ("LightGBM", "SVM (Tanimoto)", "kNN (Tanimoto)") if m in MODEL_ORDER]

for ep_i, ep in enumerate(EPS):
    ax = axes[ep_i]
    ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="perfect")
    y_col = Y_train_full[:, ep_i]
    for style, store, lbl in [(":", oof, "raw"), ("-", oof_cal, "calibrated")]:
        for mi, nm in enumerate(_show):
            p = store.get(nm, {}).get(ep)
            if p is None:
                continue
            ok = np.isfinite(p) & np.isfinite(y_col)
            if ok.sum() < 40:
                continue
            yb = (y_col[ok] == POSITIVE_CLASS).astype(int)
            pv = np.clip(p[ok], 0, 1)
            try:
                bins = pd.qcut(pv, q=min(8, max(2, ok.sum() // 25)), duplicates="drop")
            except Exception:
                continue
            xs, ys = [], []
            for b in bins.categories:
                sel = bins == b
                if sel.sum() < 5:
                    continue
                xs.append(pv[sel].mean()); ys.append(yb[sel].mean())
            if xs:
                ax.plot(xs, ys, style, marker="o", markersize=4, color=_cols(mi),
                        linewidth=1.5, alpha=0.9 if lbl == "calibrated" else 0.45,
                        label=f"{nm} ({lbl})")
    ax.set_xlabel("mean predicted P(active)")
    ax.set_ylabel("observed fraction active")
    ax.set_title(f"{ep}: reliability, out-of-fold\n(dotted = raw, solid = calibrated)")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=0.25); ax.legend(fontsize=7)

ax = axes[-1]
ep0 = EPS[0]
_bn = BLEND_NAMES[ep0]
p = oof_cal.get(_bn, {}).get(ep0)
if p is None:
    p = oof_cal[MODEL_ORDER[0]][ep0]
y_col = Y_train_full[:, 0]
ok = np.isfinite(p) & np.isfinite(y_col)
for c, nm in enumerate(CLASS_NAMES):
    sel = ok & (y_col == c)
    if sel.sum():
        ax.hist(np.clip(p[sel], 0, 1), bins=25, alpha=0.6, label=f"{nm} (n={int(sel.sum())})")
ax.set_xlabel(f"calibrated P(active) — blend, {ep0}")
ax.set_ylabel("compounds")
ax.set_title("Separation between the true classes")
ax.legend(fontsize=8); ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 11. Part E.3 — Screen the library

Each committee member scores every library compound on pKi; the blend is their
equal-weight average; and the **deployment calibrators from Part E.2c are applied**, so the
`p_active_*` columns are probabilities rather than raw scores.

There is one endpoint now, so there is no combined score and no cross-endpoint
agreement to lean on. The single calibrated pKi blend is the ranking, which puts the
weight on the controls in Part E.5: the known compounds must rank near the top, and
the enumerated products must beat property-matched decoys of the same size.

The MC-dropout and bootstrap spreads are reported as `*_spread`. They measure model
disagreement, not predictive error — the calibrated probability is the uncertainty that
matters, and Part E.2c is its test.

In [ ]:
# --- Score the library --------------------------------------------------------------------
CHUNK = CONFIG.screen_chunk_size
print(f"Scoring {len(screen_df):,} library compounds. Fingerprint-kernel models "
      f"(SVM, kNN) are\nevaluated in blocks of {CHUNK:,} so their Gram against the "
      f"training set is never\nmaterialised in full.\n")

P_nn_mean = P_nn_std = None
if NN_ENABLED:
    P_nn_mean, P_nn_std = predict_proba_mt_mc(nn_model_deploy, X_screen,
                                              CONFIG.nn_mc_dropout_passes)

screen_results = screen_df.copy().reset_index(drop=True)
_combined = []

for t, ep in enumerate(tqdm(EPS, desc="Library predictions", unit="endpoint")):
    raw: Dict[str, np.ndarray] = {}
    spreads = []

    if NN_ENABLED:
        raw["NN"] = P_nn_mean[:, t, POSITIVE_CLASS]
        spreads.append(P_nn_std[:, t, POSITIVE_CLASS] ** 2)

    for nm in tqdm(list(models_deploy[ep]), desc=f"  {ep}", unit="model", leave=False):
        boots = models_boot[ep].get(nm, [])
        try:
            if boots:
                stack = np.stack([
                    predict_proba_any(nm, b, X_screen, screen_fps,
                                      chunk=CHUNK)[:, POSITIVE_CLASS]
                    for b in tqdm(boots, desc=f"    {nm} bootstrap", unit="model",
                                   leave=False)], axis=0)
                raw[nm] = stack.mean(axis=0)
                spreads.append(stack.std(axis=0) ** 2)
            else:
                raw[nm] = predict_proba_any(nm, models_deploy[ep][nm], X_screen,
                                            screen_fps, chunk=CHUNK)[:, POSITIVE_CLASS]
        except Exception as e:
            tqdm.write(f"  {ep}/{nm}: scoring failed ({type(e).__name__}); skipped")

    mem = [nm for nm in committee_for(ep) if nm in raw]
    raw_blend = blend_equal([raw[nm] for nm in mem]) if len(mem) >= 2 else \
        next(iter(raw.values()))

    for nm, p in raw.items():
        c = final_calibrators.get((nm, ep))
        col = "p_%s_%s" % (ep, re.sub(r"[^A-Za-z0-9]", "", nm))
        screen_results[col] = c.transform(p) if c is not None else p

    cal = final_calibrators.get((BLEND_NAMES[ep], ep))
    cal_blend = cal.transform(raw_blend) if cal is not None else raw_blend
    screen_results[f"p_active_{ep}"] = cal_blend
    screen_results[f"p_active_{ep}_raw"] = raw_blend
    screen_results[f"p_active_{ep}_spread"] = (
        np.sqrt(np.sum(spreads, axis=0) / len(spreads)) if spreads
        else np.full(len(screen_results), np.nan))
    _combined.append(cal_blend)
    print(f"  {ep}: blend of {len(mem)} members ({', '.join(mem)})")

screen_results["p_active_combined"] = np.mean(np.stack(_combined, axis=0), axis=0)
screen_results["endpoints_agree"] = (np.min(np.stack(_combined, axis=0), axis=0) >= 0.5) \
    if len(_combined) > 1 else np.nan
screen_results = pd.concat([screen_results, ad_scores.reset_index(drop=True)], axis=1)

print()
for ep in EPS:
    c, r = screen_results[f"p_active_{ep}"], screen_results[f"p_active_{ep}_raw"]
    print(f"  {ep}: calibrated median {c.median():.3f}, 99.9th pct {c.quantile(0.999):.3f}, "
          f"max {c.max():.3f}   (raw max {r.max():.3f})")
cc = screen_results["p_active_combined"]
print(f"  combined: median {cc.median():.3f}, max {cc.max():.3f}\n")
for thr in (0.5, 0.7, 0.9):
    n = int((cc >= thr).sum())
    n_in = int(((cc >= thr) & ~screen_results["consensus_OOD"]).sum())
    print(f"  combined P(active) >= {thr:.1f}: {n:>7,} compounds ({n_in:,} in-domain)")
screen_results.head()

## 12. Part E.4 — Confidence against domain distance

Where do the confident predictions sit relative to the applicability domain? A compound
with high P(active) and low Tanimoto distance is a candidate. One with high P(active) and
high distance is the model extrapolating with conviction — and Part E.2's per-fold results
say how much that conviction is worth on chemistry it has no analogue for.

In F6 the model was *more* confident on unfamiliar compounds than familiar ones (mean 0.194
vs 0.139), which is the clearest possible sign that confidence was not tracking
reliability. The same check runs here on the recalibrated probabilities.

In [ ]:
# --- P(active) against domain distance ----------------------------------------------------
_SCORE = "p_active_combined"
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

ax = axes[0]
_s = screen_results.sample(min(40000, len(screen_results)), random_state=CONFIG.random_state)
sc = ax.scatter(_s["tanimoto_dissimilarity"], _s[_SCORE], c=_s["n_flags_OOD"],
                cmap="viridis", alpha=0.35, s=8, linewidths=0)
ax.axvline(ad.thresholds["tanimoto"], color="#d1495b", linestyle="--", linewidth=1.8,
           label=f"Tanimoto threshold ({ad.thresholds['tanimoto']:.3f})")
ax.set_xlabel("Tanimoto dissimilarity to nearest training compound")
ax.set_ylabel("calibrated combined P(active)")
ax.set_title("Confidence vs domain distance\n(right of the line = structurally unfamiliar)")
plt.colorbar(sc, ax=ax, label="# AD flags")
ax.legend(fontsize=8, loc="upper left")
ax.grid(alpha=0.2)

ax = axes[1]
_bins = pd.qcut(screen_results["tanimoto_dissimilarity"], q=10, duplicates="drop")
_g = screen_results.groupby(_bins, observed=True)[_SCORE]
_mid = [iv.mid for iv in _g.mean().index]
ax.plot(_mid, _g.mean().values, "o-", linewidth=1.8, label="mean P(active)")
ax.fill_between(_mid, _g.quantile(0.1).values, _g.quantile(0.9).values, alpha=0.2,
                label="10th-90th pct")
ax.axvline(ad.thresholds["tanimoto"], color="#d1495b", linestyle="--", linewidth=1.8)
ax.set_xlabel("Tanimoto dissimilarity (decile midpoint)")
ax.set_ylabel("calibrated combined P(active)")
ax.set_title("Does confidence fall further from home?")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

_far = screen_results[screen_results["tanimoto_OOD"]]
_near = screen_results[~screen_results["tanimoto_OOD"]]
print(f"Structurally familiar (tanimoto_OOD False): {len(_near):,} compounds, "
      f"mean P(active) {_near[_SCORE].mean():.3f}")
print(f"Structurally unfamiliar (tanimoto_OOD True): {len(_far):,} compounds, "
      f"mean P(active) {_far[_SCORE].mean():.3f}")
if len(_far) and len(_near):
    if _far[_SCORE].mean() >= _near[_SCORE].mean():
        print("\n!! Still no less confident on unfamiliar chemistry than on familiar "
              "chemistry.")
        print("   Confidence is not tracking reliability, so the applicability-domain "
              "flags —")
        print("   not the probability — are what must gate the shortlist.")
    else:
        print("\n   Confidence does fall on unfamiliar chemistry, which is the behaviour "
              "wanted.")
        print("   It still does not replace the AD flags, but it no longer contradicts "
              "them.")

## 13. Part E.5 — Shortlist, by provenance

Four things to read here, in this order. The first two decide whether the rest means
anything.

**1. Do the known compounds rank at the top?** Enumerated products whose structure matches
a training compound are flagged `is_known`. They are a positive control: the model has seen
them and labelled them active, so if they do not land near the top of the ranking something
is broken upstream and nothing further down is worth reading. Their median percentile rank
is printed first.

**2. Is the ranking separating chemistry, or separating size?** The score distribution is
broken out by `source`. The enumerated products are Glu-ureas built from the training
pharmacophore, so they *should* score highly — that is close to circular. The informative
comparison is against `library_matched`, the property-matched decoys, because those control
for the molecular-weight difference that would otherwise do the work.

**3. Does any catalogue compound outrank the enumerated products?** That would be a genuine
scaffold hop — a chemotype outside the Glu-urea class that the model nonetheless likes — and
it is the most interesting thing this pipeline can produce. `library_near` is where to look.

**4. Which caps does the model prefer?** Within the enumerated set, the cores are nearly
constant and the caps vary. If the score spread across caps is narrow, the model is
recognising the core and telling you nothing about the variable part; a wide spread with a
chemically coherent pattern is a real structure–activity signal. The per-cap table reports
that spread.

Every enumerated row carries its core, reaction and cap, so a hit is a one-step synthesis
proposal rather than only a structure.

In [ ]:
# --- Shortlist, broken down by provenance ---------------------------------------------------
EP = PRIMARY
SCORE = f"p_active_{EP}"
if SCORE not in screen_results.columns:
    SCORE = "p_active_combined"

_id_cols = [c for c in ("compound_id", "smiles") if c in screen_results.columns]
_prov = [c for c in ("source", "core_id", "reaction", "cap_id", "is_known")
         if c in screen_results.columns]
cols = _id_cols + [SCORE, f"{SCORE}_spread"] + _prov \
       + ["n_flags_OOD", "consensus_OOD", "tanimoto_dissimilarity"]
cols = [c for c in cols if c in screen_results.columns]

in_domain = screen_results[~screen_results["consensus_OOD"]]
print(f"{len(in_domain):,} / {len(screen_results):,} compounds in the applicability domain "
      f"({100 * len(in_domain) / max(len(screen_results), 1):.2f}%)")
print(f"  Tanimoto veto: {'ON' if CONFIG.ad_tanimoto_veto else 'OFF'}")

# =====================================================================================
# 1. POSITIVE CONTROL — do known compounds rank at the top?
# =====================================================================================
print("\n" + "=" * 96)
print("1. POSITIVE CONTROL")
print("=" * 96)
if "is_known" in screen_results.columns and bool(screen_results["is_known"].any()):
    r = screen_results[SCORE].rank(pct=True, ascending=True)
    kn = r[screen_results["is_known"].fillna(False).astype(bool)]
    print(f"  {len(kn)} enumerated products match a known training compound.")
    print(f"  Their percentile rank by {SCORE}: median {100 * kn.median():.1f}, "
          f"min {100 * kn.min():.1f}, max {100 * kn.max():.1f}")
    if kn.median() > 0.9:
        print("  -> Known compounds sit in the top decile. The ranking behaves as it must,")
        print("     which is the minimum bar for reading anything else here.")
    elif kn.median() > 0.7:
        print("  -> Known compounds are in the upper third but not the top decile. Weak.")
        print("     Treat the rest of the ranking with corresponding caution.")
    else:
        print("  !! Known ACTIVES the model was trained on do NOT rank highly. Something is")
        print("     wrong upstream — featurization, label assignment or the deployed model.")
        print("     Nothing further down this ranking should be read until that is resolved.")
else:
    print("  No enumerated product matches a known training compound, so there is no")
    print("  internal control on this ranking. Add cores/caps that reconstruct a few known")
    print("  ligands — without that the shortlist is unfalsifiable.")

# =====================================================================================
# 2. SCORE BY SOURCE — chemistry or size?
# =====================================================================================
print("\n" + "=" * 96)
print("2. SCORE DISTRIBUTION BY PROVENANCE")
print("=" * 96)
if "source" in screen_results.columns and screen_results["source"].notna().any():
    g = screen_results.groupby("source")
    by_src = pd.DataFrame({
        "n": g.size(),
        "median_score": g[SCORE].median().round(4),
        "p90_score": g[SCORE].quantile(0.9).round(4),
        "max_score": g[SCORE].max().round(4),
        "median_MW": g["MW"].median().round(0) if "MW" in screen_results else np.nan,
        "in_domain_%": (100 * g["consensus_OOD"].apply(lambda s: (~s).mean())).round(2),
    }).sort_values("median_score", ascending=False)
    display(by_src)

    if {"enumerated", "library_matched"} <= set(by_src.index):
        de = by_src.loc["enumerated", "median_score"]
        dm = by_src.loc["library_matched", "median_score"]
        mwe = by_src.loc["enumerated", "median_MW"]
        mwm = by_src.loc["library_matched", "median_MW"]
        print(f"\n  enumerated vs property-matched decoys: median {de:.3f} vs {dm:.3f} "
              f"({de - dm:+.3f})")
        print(f"  median MW {mwe:.0f} vs {mwm:.0f} — the matching controls for size, so this")
        print("  gap is the part attributable to chemistry rather than to bulk properties.")
        if de - dm < 0.05:
            print("  !! Barely separated from its own size-matched decoys. The model is not")
            print("     distinguishing the pharmacophore from arbitrary molecules of the")
            print("     same size, which undercuts the whole enumeration.")
    if {"enumerated", "library_random"} <= set(by_src.index):
        print(f"\n  Note: enumerated vs library_random is the FLATTERING comparison — those")
        print("  populations differ in size as well as chemistry.")
        if "library_matched" in by_src.index:
            print("  Use the matched row above, which controls for size.")
        else:
            print("  !! There is no `library_matched` row, so that control is MISSING: the")
            print("     catalogue had no compounds in the enumerated library's size range")
            print("     (Part A.2c reported the shortfall). Any separation below is")
            print("     confounded with molecular size and should not be read as chemistry.")
else:
    print("  No `source` column: the library was not mixed. Set "
          "CONFIG.screening_source = 'enumeration+library' to get this comparison.")

# =====================================================================================
# 3. SCAFFOLD HOPS — any catalogue compound beating the enumerated products?
# =====================================================================================
print("\n" + "=" * 96)
print("3. SCAFFOLD-HOP CANDIDATES")
print("=" * 96)
if "source" in screen_results.columns and screen_results["source"].notna().any():
    cat = screen_results[screen_results["source"].astype(str).str.startswith("library")]
    enum = screen_results[screen_results["source"] == "enumerated"]
    if len(cat) and len(enum):
        thr = enum[SCORE].quantile(0.9)
        hop = cat[(cat[SCORE] >= thr) & (~cat["consensus_OOD"])]
        print(f"  Enumerated 90th-percentile score: {thr:.4f}")
        print(f"  Catalogue compounds matching it AND in domain: {len(hop):,}")
        if len(hop):
            print("  These are chemotypes outside the Glu-urea class that the model rates as")
            print("  highly as the pharmacophore it was trained on — the most interesting")
            print("  output available here, and the only part that is not circular:")
            display(hop.sort_values(SCORE, ascending=False)[cols].head(15).round(4))
        else:
            print("  None. No catalogue compound reaches the enumerated library's top decile")
            print("  while staying in domain. Consistent with F8's finding that the two")
            print("  chemical populations barely overlap — and it means this screen offers")
            print("  no scaffold hop, only variations on the known core.")
else:
    print("  Library not mixed; no scaffold-hop comparison available.")

# =====================================================================================
# 4. WHICH CAPS? — is there signal in the variable part?
# =====================================================================================
print("\n" + "=" * 96)
print("4. CAP PREFERENCE — signal in the variable part, or just the core?")
print("=" * 96)
if "cap_id" in screen_results.columns and screen_results["cap_id"].notna().any():
    e = screen_results[screen_results["cap_id"].notna()]
    cap_tab = (e.groupby(["cap_role", "cap_id"])[SCORE]
               .agg(["size", "median", "max"]).round(4)
               .sort_values("median", ascending=False))
    display(cap_tab.head(20))
    spread = float(e.groupby("cap_id")[SCORE].median().std())
    within = float(e.groupby("cap_id")[SCORE].std().mean())
    print(f"  s.d. of per-cap median score : {spread:.4f}")
    print(f"  mean within-cap s.d.          : {within:.4f}")
    if spread < 0.02:
        print("  !! The caps are barely distinguished. The model is recognising the shared")
        print("     core and saying almost nothing about the variable part, so the ranking")
        print("     within the enumerated library carries little information.")
    elif spread < within:
        print("  -> Between-cap variation is smaller than the variation within a single cap")
        print("     (different cores and regiochemistries). Read cap preferences cautiously.")
    else:
        print("  -> Between-cap variation exceeds within-cap variation: there is a real")
        print("     preference among the caps. Worth reading the top of that table as a")
        print("     structure-activity hint, and worth checking it is chemically coherent")
        print("     rather than driven by one outlying product.")
else:
    print("  No enumerated products with cap provenance.")

# =====================================================================================
# The shortlist itself
# =====================================================================================
print("\n" + "=" * 96)
print("SHORTLIST (in domain, ranked by score)")
print("=" * 96)
top_hits = in_domain.sort_values(SCORE, ascending=False)[cols].head(30)
display(top_hits.round(4))

key = None
try:
    key = pooled_df.loc[EP, "roc_auc"].idxmax()
except Exception:
    pass
if key is not None:
    r = pooled_df.loc[(EP, key)]
    print(f"\nEXPECTED YIELD — '{key}', pooled leave-one-cluster-out, families held out:")
    print(f"  ROC-AUC {r['roc_auc']:.3f} | PR-AUC {r['pr_auc']:.3f} "
          f"(base rate {r['base_rate']:.3f})")
    for n in CONFIG.precision_at_n:
        k = f"precision@{n}"
        if k in r and np.isfinite(r[k]):
            print(f"  precision@{n:<4d}: {r[k]:.3f}")
    if "kNN (Tanimoto)" in pooled_df.loc[EP].index:
        kn = pooled_df.loc[(EP, "kNN (Tanimoto)"), "roc_auc"]
        print(f"  vs nearest-neighbour floor: {kn:.3f} ({r['roc_auc'] - kn:+.3f})")
    print("\n  That precision figure was measured on real compounds with real labels whose")
    print("  whole chemical family was held out. It does NOT transfer automatically to an")
    print("  enumerated library built from the training pharmacophore — there the model is")
    print("  partly recognising its own training distribution. Read it as an upper bound.")

## 14. Part E.6 — Which atoms drive P(active)?

Per-atom attribution by **substructure ablation**: each "on" fingerprint bit is set to 0,
the calibrated blended P(active) recomputed, and the change spread over the atoms in that
bit's environment.

This corrects F5, which nudged each on-bit from 1 to 1.05 and divided by 0.05. That does
nothing to a tree ensemble — a 0.05 change in a binary feature essentially never crosses a
split threshold — so LightGBM contributed exactly zero and the map was an NN gradient
labelled as a blend. Setting the bit to 0 asks the question that means something, and every
committee member responds to it.

In [ ]:
# --- Which atoms drive P(active)? -----------------------------------------------------------
from rdkit.Chem import Draw
from collections import defaultdict


def get_bit_atoms(mol, bit_info: dict, bit_id: int) -> set:
    atoms = set()
    for atom_idx, radius in bit_info[bit_id]:
        if radius == 0:
            atoms.add(atom_idx)
        else:
            env = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_idx)
            amap: dict = {}
            Chem.PathToSubmol(mol, env, atomMap=amap)
            atoms.update(amap.keys())
    return atoms


def predict_blend_single(x_row: np.ndarray, fp, ep_idx: int, ep: str) -> float:
    """Calibrated blended P(active) for one compound on one endpoint.

    Uses the same committee membership as the screen. Fingerprint-kernel members are
    served their single fingerprint directly."""
    xb = x_row.reshape(1, -1).astype(np.float32)
    members = []
    for nm in committee_for(ep):
        try:
            if nm == "NN":
                members.append(predict_proba_mt(nn_model_deploy, xb)[0, ep_idx,
                                                                     POSITIVE_CLASS])
            else:
                members.append(predict_proba_any(nm, models_deploy[ep][nm], xb,
                                                 [fp])[0, POSITIVE_CLASS])
        except Exception:
            continue
    if not members:
        return float("nan")
    raw = float(np.mean(members))
    cal = final_calibrators.get((BLEND_NAMES.get(ep), ep))
    return float(cal.transform(np.array([raw]))[0]) if cal is not None else raw


def compute_atom_sensitivities(mol, x_row, fp, ep_idx, ep) -> Dict[int, float]:
    """Per-atom contribution by ABLATING each 'on' fingerprint bit (1 -> 0).

    F5 nudged each on-bit from 1 to 1.05, which does nothing to a tree ensemble — a 0.05
    change in a binary feature essentially never crosses a split threshold, so LightGBM
    contributed exactly zero and the map was an NN gradient wearing a blend's label.
    Setting the bit to 0 asks the meaningful question and every member responds.

    Caveat with the Tanimoto-kernel members: zeroing a bit changes the design-matrix row
    but not the RDKit fingerprint object, so SVM and kNN see the unmodified molecule and
    contribute a constant. Their share of the blend therefore flattens the map rather than
    distorting it."""
    _, bit_info = featurizer.morgan_fp_with_bit_info(mol)
    base = predict_blend_single(x_row, fp, ep_idx, ep)
    scores: Dict[int, float] = defaultdict(float)
    on_bits = [b for b in bit_info if x_row[b] > 0.5]
    for b in tqdm(on_bits, desc="  ablating bits", unit="bit", leave=False):
        xp = x_row.copy()
        xp[b] = 0.0
        delta = base - predict_blend_single(xp, fp, ep_idx, ep)
        atoms = get_bit_atoms(mol, bit_info, b)
        if atoms:
            for a in atoms:
                scores[a] += delta / len(atoms)
    return scores


_ep_idx, _ep = 0, EPS[0]
if CONFIG.feature_set == "descriptors_only":
    print("feature_set='descriptors_only' -> no fingerprint bits to attribute; skipped.")
elif len(top_hits) == 0:
    print("No in-domain compounds to attribute.")
else:
    n_show = min(4, len(top_hits))
    fig, axes = plt.subplots(1, n_show, figsize=(4.2 * n_show, 4.4))
    axes = np.atleast_1d(axes)
    cmap = plt.get_cmap("coolwarm")
    for ax, ridx in zip(axes, tqdm(list(top_hits.index[:n_show]),
                                    desc="Atom-contribution maps", unit="mol")):
        mol = screen_feat["mols"][ridx]
        scores = compute_atom_sensitivities(mol, X_screen[ridx], screen_fps[ridx],
                                            _ep_idx, _ep)
        if not scores:
            ax.set_title("no 'on' bits"); ax.axis("off"); continue
        mx = max(abs(v) for v in scores.values()) or 1.0
        colors = {a: cmap(0.5 + 0.5 * (v / mx))[:3] for a, v in scores.items()}
        ax.imshow(Draw.MolToImage(mol, size=(320, 320), highlightAtoms=list(scores),
                                  highlightAtomColors=colors))
        ax.axis("off")
        cid = (screen_results.loc[ridx, "compound_id"]
               if "compound_id" in screen_results.columns else ridx)
        ax.set_title(f"{cid}\nP(active) {screen_results.loc[ridx, 'p_active_combined']:.3f}",
                     fontsize=9)
    plt.suptitle(f"Atom contributions to calibrated P(active) on {_ep} by substructure "
                 "ablation (red = raises, blue = lowers)", y=1.03)
    plt.tight_layout()
    plt.show()

## 15. Part F — Save outputs

Everything needed to reproduce, audit or redeploy: the shortlist and full library
predictions, the pKi dataset with its source tags and label provenance,
all model families per endpoint with their bootstrap ensembles, the **probability
calibrators**, the shared feature scaler, the descriptor selector, the applicability-domain
object, and every evaluation table.

The **out-of-fold predictions are saved raw and calibrated**. Every pooled number in Part
E.2 derives from them, so the enrichment, precision@N and calibration figures can be
recomputed — or challenged — without re-running the folds.

F7 writes to its own directory. F6 left its artifacts beside F5's in a single folder, which
made it impossible to tell which run a file came from.

In [ ]:
# --- Save everything ------------------------------------------------------------------------
import joblib

out_dir = Path(CONFIG.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {out_dir}")

screen_results.to_csv(out_dir / "screening_predictions.csv", index=False)
top_hits.to_csv(out_dir / "shortlist.csv", index=False)
train_df.to_csv(out_dir / "psma_pki_dataset.csv", index=False)

loco_eval_df.to_csv(out_dir / "loco_per_fold.csv", index=False)
loco_folds_df.to_csv(out_dir / "loco_fold_definitions.csv")
pooled_df.to_csv(out_dir / "loco_pooled.csv")
macro_df.to_csv(out_dir / "loco_macro.csv")
random_eval_df.to_csv(out_dir / "random_split_reference.csv", index=False)
ref_eval_df.to_csv(out_dir / "reference_split_scores.csv")
recalibration_df.to_csv(out_dir / "recalibration_before_after.csv")
if len(tuning_df):
    tuning_df.to_csv(out_dir / "tuning_summary.csv", index=False)
if "ablation_df" in globals() and len(ablation_df):
    ablation_df.to_csv(out_dir / "transfer_learning_ablation.csv")
if len(derived_report):
    derived_report.to_csv(out_dir / "derived_endpoint_report.csv", index=False)

# out-of-fold predictions, raw and calibrated, for every model and endpoint
_oof = pd.DataFrame({"smiles": train_df["smiles"].values,
                     "source_tag": train_df["source_tag"].values,
                     "butina_cluster": butina_groups, "fold": fold_of})
for t, ep in enumerate(EPS):
    _oof[f"y_{ep}"] = Y_train_full[:, t]
for nm in MODEL_ORDER:
    for ep in EPS:
        if np.isfinite(oof[nm][ep]).any():
            _oof[f"raw::{nm}::{ep}"] = oof[nm][ep]
            _oof[f"cal::{nm}::{ep}"] = oof_cal[nm][ep]
_oof.to_csv(out_dir / "out_of_fold_predictions.csv", index=False)

if NN_ENABLED:
    torch.save({
        "trunk_state_dict": {k: v.cpu()
                             for k, v in nn_model_deploy.trunk.state_dict().items()},
        "task_heads_state_dict": {k: v.cpu()
                                  for k, v in nn_model_deploy.task_heads.state_dict().items()},
        "hidden_dims": nn_hidden_dims_final,
        "dropout": nn_best_params["dropout"],
        "endpoints": EPS,
        "active_thresholds": {ep: CONFIG.active_thresholds[ep] for ep in EPS},
        "class_names": CLASS_NAMES,
        "used_pretraining": bool(CONFIG.use_pretraining),
    }, out_dir / "nn_multitask_classifier.pt")

_safe = lambda s: re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")
for ep in EPS:
    for nm, mdl in models_deploy[ep].items():
        joblib.dump(mdl, out_dir / f"model_{_safe(nm)}_{ep}.joblib")
        if models_boot[ep].get(nm):
            joblib.dump(models_boot[ep][nm],
                        out_dir / f"model_{_safe(nm)}_{ep}_bootstrap.joblib")

# Plain data FIRST. A joblib dump of the calibrators has to pickle a class defined in
# this notebook, so it is the one save here that can fail, and it must not be able to
# take the rest of the outputs with it.
_cal_export = {}
for (nm, ep), cal in final_calibrators.items():
    entry = {"method": cal.method, "fitted": cal._m is not None}
    m = cal._m
    if m is not None:
        if cal.method == "isotonic":
            entry["x"] = [float(v) for v in np.asarray(m.X_thresholds_).ravel()]
            entry["y"] = [float(v) for v in np.asarray(m.y_thresholds_).ravel()]
            entry["apply"] = "numpy.interp(score, x, y), clipped to [0, 1]"
        else:
            entry["coef"] = float(np.ravel(m.coef_)[0])
            entry["intercept"] = float(np.ravel(m.intercept_)[0])
            entry["apply"] = "1 / (1 + exp(-(coef * score + intercept)))"
    _cal_export[f"{nm}::{ep}"] = entry
with open(out_dir / "probability_calibrators.json", "w") as fh:
    json.dump(_cal_export, fh, indent=2)

# Now the convenience pickle, which is optional: probability_calibrators.json above is
# the authoritative copy and reloads anywhere.
try:
    joblib.dump(final_calibrators, out_dir / "probability_calibrators.joblib")
except Exception as _e:
    print(f"  (could not pickle the calibrators: {type(_e).__name__}: {_e})")
    print("   probability_calibrators.json carries the same parameters and its `apply`")
    print("   field gives the formula, so nothing is lost.")

for _obj, _fn in [(feature_scaler, "feature_scaler.joblib"),
                  (descriptor_selector, "descriptor_selector.joblib"),
                  (ad, "applicability_domain.joblib")]:
    try:
        joblib.dump(_obj, out_dir / _fn)
    except Exception as _e:
        print(f"  (could not pickle {_fn}: {type(_e).__name__}: {_e})")

with open(out_dir / "config.json", "w") as fh:
    json.dump({k: (list(v) if isinstance(v, (tuple, list)) else v)
               for k, v in CONFIG.__dict__.items()}, fh, indent=2, default=str)
with open(out_dir / "best_params.json", "w") as fh:
    json.dump({"per_endpoint": {ep: best_params[ep] for ep in EPS},
               "nn": nn_best_params if NN_ENABLED else None,
               "committee": {ep: committee_for(ep) for ep in EPS},
               "feature_set": CONFIG.feature_set,
               "use_pretraining": CONFIG.use_pretraining,
               "use_neural_network": NN_ENABLED,
               "recalibration": CONFIG.recalibration}, fh, indent=2, default=str)

print(f"\nSaved {len(list(out_dir.iterdir()))} files:")
for p in sorted(out_dir.iterdir()):
    print(f" - {p.name}")

## 16. Notes for reuse

### Reading the results

- **Quote the pooled leave-one-cluster-out numbers**, not the random-split reference (kept
  only to make the leakage gap visible) and not the macro average (kept only to show the
  spread across families).
- **Read the floor comparison in Part E.2b before anything else.** If the models do not clear
  Tanimoto kNN, the pipeline is a similarity search with calibrated probabilities — a useful
  thing, but a different claim. If they do not clear `LogReg (L2)`, the ensembles are not
  paying for their complexity and a single logistic regression is the honest deployment.
- **For the enumerated library, Part E.4's four checks come before the shortlist**, and the
  positive control comes before the other three.
- **`precision@N` beats `roc_auc` for a screen.** They can disagree; F8 showed a case where
  AUC said "tied with lookup" and precision@50 said "0.88 against 0.64".

### Knobs

| setting | default | effect |
|---|---|---|
| `screening_source` | `"enumeration+library"` | `"enumeration"` for the virtual library alone, `"file"` for the old catalogue-only behaviour |
| `mix_n_near` / `mix_n_matched` / `mix_n_random` | 5000 / 10000 / 5000 | the three catalogue strata. Raising `mix_n_near` widens the scaffold-hop search |
| `mix_pool_sample` | 60000 | how much of the file to consider before computing distances. Raise for a more thorough `library_near`, at a runtime cost |
| `enum_property_quantile` | 0.05 | the product window, as percentiles of the **known actives'** properties. Do not replace this with a rule-of-five filter: PSMA-617 is MW ≈ 1042 with eight acids and Ro5 would delete the entire active class |
| `active_thresholds["pKi"]` | 8.0 | 10 nM. Change before looking at results, then re-run Part A.1b |
| `drop_derived_endpoint_values` / `derived_endpoint` | True / `"pKi"` | removes Cheng–Prusoff values. For a new target, check which endpoint is the derived one |
| `ad_tanimoto_veto` | True | makes structural dissimilarity decisive. Off reproduces F5's much larger, much less trustworthy in-domain set |
| `committee["pKi"]` | 5 models | the blend members. F8 found the blend did not beat its best member, so compare against the single-model rows before trusting it |

### Extending the enumeration

Add a reaction by adding one `REACTION_LIBRARY` entry **with a test pair** — untested
templates are refused. Add building blocks to `CORES` (needing one reactive handle and the
pharmacophore) or `CAPS` (tagged with the role a reaction consumes). The combinatorics are
core × reaction × cap, so adding caps is linear and adding cores multiplies.

For two-step routes — building the urea core *in situ* rather than starting from it — feed
the output of `enumerate_library` back in as `CORES`. Note that the enumeration models the
net transformation, not a protected synthetic route: real synthesis of these needs the Glu
and Lys carboxylates protected and a final deprotection.

If you want fragment-replacement rather than reaction enumeration, CReM (`pip install crem`)
does chemically reasonable mutations from a precomputed fragment database and would slot in
at the same point.

### Still outstanding

- **Nothing has been validated in a lab.** Every number here is the model grading its own
  homework under progressively stricter exam conditions.
- **282 pKi labels is a small dataset**, one filing supplies a sixth of the carried set, and
  the largest Butina cluster is an eighth of it. PSMA actives from ChEMBL absent from
  BindingDB, and property-matched decoys as presumed inactives, would both add more than any
  further modelling change.
- **The enumerated library is a proposal, not a compound collection.** Products are the net
  transformation of a reaction template; synthetic accessibility, protecting-group strategy
  and actual availability of the building blocks are all unchecked. Treat the shortlist as a
  list of molecules to *consider making*.
- **The AutoQSAR gap penalty prefers under-fit models.** It is kept for continuity with
  Dixon et al., and the tuning cell now prints what the choice cost in held-out AUC. Consider
  selecting on held-out AUC directly.